In [ ]:
!nvidia-smi

Sat Sep  5 06:26:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

for folder in [
    "/content/ResumeOptimizer-LoRA",
    "/content/qwen-resume-lora-v2",
    "/content/qwen-resume-lora"
]:
    print(folder, "->", os.path.exists(folder))

/content/ResumeOptimizer-LoRA -> False
/content/qwen-resume-lora-v2 -> False
/content/qwen-resume-lora -> False


In [ ]:
!pip install -U transformers datasets peft trl bitsandbytes accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 34.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 20.4 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
!pip install -U "transformers>=4.51.0" accelerate

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
print(model.device)

cuda:0


In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is the difference between Python and C++?"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=300
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, the user is asking about the difference between Python and C++. Let me start by recalling what I know about these two programming languages.

First, Python is a high-level, interpreted language known for its simplicity and readability. C++ is a mid-level language that's compiled, which gives it more control over system resources. I should mention that Python is easier to learn and use, while C++ requires more knowledge of memory management and pointers.

Next, performance. C++ is generally faster because it's compiled directly to machine code, whereas Python is interpreted, which can make it slower. But Python has some optimizations like JIT compilation in CPython, which might help, but overall, C++ is still faster for most tasks.

Syntax is another point. Python uses indentation for blocks, which makes the code more readable but can be a source of errors if not done correctly. C++ uses braces and semicolons, which are more traditional and less error-prone in that sense.


In [ ]:
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
messages = [
    {
        "role": "user",
        "content": """
You are a resume analysis assistant.

JOB DESCRIPTION:
Software Development Engineer

Requirements:
- Python
- Java
- Data Structures and Algorithms
- SQL
- AWS

CANDIDATE RESUME:
Skills: Python, HTML, CSS, JavaScript, SQL

Project:
Built a web application using Python and SQL.

Analyze the resume against the job description.

Give me:
1. Matching skills
2. Missing skills
3. Resume improvement suggestions

Do not invent skills or experience that are not present in the resume.
"""
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=500
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, let's start by looking at the job description for the Software Development Engineer position. The requirements listed are Python, Java, Data Structures and Algorithms, SQL, and AWS. The candidate's resume mentions skills in Python, HTML, CSS, JavaScript, and SQL. 

First, I need to compare the candidate's skills with the job requirements. The candidate has Python and SQL, which are both required. But the job also asks for Java, which the candidate doesn't mention. So that's a missing skill.

Next, the job requires knowledge of Data Structures and Algorithms. The candidate's resume doesn't mention anything about that, so that's another missing area. Also, the job specifies AWS, which the candidate hasn't listed as a skill. 

Looking at the projects, the candidate built a web application using Python and SQL. The job description mentions Python and SQL, so that's a match. However, the project doesn't mention any specific technologies related to AWS, so maybe that's a point 

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [ ]:
print(type(tokenizer))

<class 'transformers.models.qwen2.tokenization_qwen2.Qwen2Tokenizer'>


In [ ]:
training_data = [
    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert resume optimization assistant. Never invent skills, experience, education, or achievements."
            },
            {
                "role": "user",
                "content": """JOB DESCRIPTION:
Python Developer

Requirements:
Python
Django
SQL
REST APIs

RESUME:
Skills: Python, SQL, HTML, CSS

Project:
Built a web application using Python and SQL.

Analyze the resume against the job description.
Identify matching skills, missing skills, and safe resume improvements."""
            },
            {
                "role": "assistant",
                "content": """MATCHING SKILLS:
- Python
- SQL

MISSING SKILLS:
- Django
- REST APIs

RESUME IMPROVEMENTS:
- Highlight the existing Python and SQL experience.
- Improve the project description using only information already supported by the resume.
- Do not claim Django or REST API experience unless the candidate actually has it."""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert resume optimization assistant. Never invent skills or experience."
            },
            {
                "role": "user",
                "content": """JOB DESCRIPTION:
Software Development Engineer

Requirements:
C++
Java
Data Structures
Algorithms
AWS

RESUME:
Skills: Python, C++, SQL

Project:
Developed a command-line application using C++.

Analyze the resume against the job description."""
            },
            {
                "role": "assistant",
                "content": """MATCHING SKILLS:
- C++

MISSING SKILLS:
- Java
- Data Structures
- Algorithms
- AWS

RESUME IMPROVEMENTS:
- Highlight the existing C++ project experience.
- If the candidate has genuinely used data structures or algorithms in the project, those can be described more clearly.
- Do not add Java or AWS experience without evidence."""
            }
        ]
    },

    {
        "messages": [
            {
                "role": "system",
                "content": "You are an expert resume optimization assistant. Never invent candidate qualifications."
            },
            {
                "role": "user",
                "content": """JOB DESCRIPTION:
Frontend Software Engineer

Requirements:
JavaScript
React
HTML
CSS
Git

RESUME:
Skills: JavaScript, HTML, CSS

Project:
Created a responsive website using HTML, CSS and JavaScript.

Analyze the resume against the job description."""
            },
            {
                "role": "assistant",
                "content": """MATCHING SKILLS:
- JavaScript
- HTML
- CSS

MISSING SKILLS:
- React
- Git

RESUME IMPROVEMENTS:
- Emphasize the responsive website project.
- Make the JavaScript, HTML and CSS contribution clear.
- Add React or Git only if the candidate actually has experience with them.
- Do not invent experience."""
            }
        ]
    }
]

In [ ]:
import json

with open("resume_training.jsonl", "w", encoding="utf-8") as f:
    for example in training_data:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Dataset created successfully!")

Dataset created successfully!


In [ ]:
with open("resume_training.jsonl", "r", encoding="utf-8") as f:
    print(f.read())

{"messages": [{"role": "system", "content": "You are an expert resume optimization assistant. Never invent skills, experience, education, or achievements."}, {"role": "user", "content": "JOB DESCRIPTION:\nPython Developer\n\nRequirements:\nPython\nDjango\nSQL\nREST APIs\n\nRESUME:\nSkills: Python, SQL, HTML, CSS\n\nProject:\nBuilt a web application using Python and SQL.\n\nAnalyze the resume against the job description.\nIdentify matching skills, missing skills, and safe resume improvements."}, {"role": "assistant", "content": "MATCHING SKILLS:\n- Python\n- SQL\n\nMISSING SKILLS:\n- Django\n- REST APIs\n\nRESUME IMPROVEMENTS:\n- Highlight the existing Python and SQL experience.\n- Improve the project description using only information already supported by the resume.\n- Do not claim Django or REST API experience unless the candidate actually has it."}]}
{"messages": [{"role": "system", "content": "You are an expert resume optimization assistant. Never invent skills or experience."}, {"

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="resume_training.jsonl"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 3
    })
})


In [ ]:
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

print("Model loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
print(dataset["train"][0])

{'messages': [{'role': 'system', 'content': 'You are an expert resume optimization assistant. Never invent skills, experience, education, or achievements.'}, {'role': 'user', 'content': 'JOB DESCRIPTION:\nPython Developer\n\nRequirements:\nPython\nDjango\nSQL\nREST APIs\n\nRESUME:\nSkills: Python, SQL, HTML, CSS\n\nProject:\nBuilt a web application using Python and SQL.\n\nAnalyze the resume against the job description.\nIdentify matching skills, missing skills, and safe resume improvements.'}, {'role': 'assistant', 'content': 'MATCHING SKILLS:\n- Python\n- SQL\n\nMISSING SKILLS:\n- Django\n- REST APIs\n\nRESUME IMPROVEMENTS:\n- Highlight the existing Python and SQL experience.\n- Improve the project description using only information already supported by the resume.\n- Do not claim Django or REST API experience unless the candidate actually has it.'}]}


In [ ]:
split_dataset = dataset["train"].train_test_split(
    test_size=1,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

Training examples: 2
Testing examples: 1


In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_chat)
test_dataset = test_dataset.map(format_chat)

print(train_dataset[0]["text"])

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/1 [00:00<?, ? examples/s]

<|im_start|>system
You are an expert resume optimization assistant. Never invent skills or experience.<|im_end|>
<|im_start|>user
JOB DESCRIPTION:
Software Development Engineer

Requirements:
C++
Java
Data Structures
Algorithms
AWS

RESUME:
Skills: Python, C++, SQL

Project:
Developed a command-line application using C++.

Analyze the resume against the job description.<|im_end|>
<|im_start|>assistant
<think>

</think>

MATCHING SKILLS:
- C++

MISSING SKILLS:
- Java
- Data Structures
- Algorithms
- AWS

RESUME IMPROVEMENTS:
- Highlight the existing C++ project experience.
- If the candidate has genuinely used data structures or algorithms in the project, those can be described more clearly.
- Do not add Java or AWS experience without evidence.<|im_end|>



In [ ]:
import torch
from transformers import BitsAndBytesConfig

print("CUDA available:", torch.cuda.is_available())

CUDA available: True


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit QLoRA configuration created!")

4-bit QLoRA configuration created!


In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [ ]:
model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded!")

Tokenizer loaded!


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit configuration ready!")

4-bit configuration ready!


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen loaded in 4-bit mode!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen loaded in 4-bit mode!


In [ ]:
from transformers import AutoModelForCausalLM

model_name = "Qwen/Qwen3-4B"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Qwen loaded in 4-bit mode!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Qwen loaded in 4-bit mode!


In [ ]:
print(torch.cuda.memory_allocated() / 1024**3, "GB used")

10.262778282165527 GB used


In [ ]:
!nvidia-smi

Fri Sep  4 08:31:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   75C    P0             33W /   70W |   14213MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
import os

print(os.path.exists("resume_training.jsonl"))

True


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="resume_training.jsonl"
)

print(dataset)

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 3
    })
})


In [ ]:
split_dataset = dataset["train"].train_test_split(
    test_size=1,
    seed=42
)

train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Testing examples:", len(test_dataset))

Training examples: 2
Testing examples: 1


In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded successfully!")

Tokenizer loaded successfully!


In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_chat)
test_dataset = test_dataset.map(format_chat)

print("Dataset formatted successfully!")

Dataset formatted successfully!


In [ ]:
print(train_dataset[0]["text"])

<|im_start|>system
You are an expert resume optimization assistant. Never invent skills or experience.<|im_end|>
<|im_start|>user
JOB DESCRIPTION:
Software Development Engineer

Requirements:
C++
Java
Data Structures
Algorithms
AWS

RESUME:
Skills: Python, C++, SQL

Project:
Developed a command-line application using C++.

Analyze the resume against the job description.<|im_end|>
<|im_start|>assistant
<think>

</think>

MATCHING SKILLS:
- C++

MISSING SKILLS:
- Java
- Data Structures
- Algorithms
- AWS

RESUME IMPROVEMENTS:
- Highlight the existing C++ project experience.
- If the candidate has genuinely used data structures or algorithms in the project, those can be described more clearly.
- Do not add Java or AWS experience without evidence.<|im_end|>



In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA training!")

Model prepared for QLoRA training!


In [ ]:
model.print_trainable_parameters()

trainable params: 0 || all params: 4,055,498,240 || trainable%: 0.0000


In [ ]:
!pip install -U "trl>=0.20.0"

In [ ]:
import trl

print("TRL version:", trl.__version__)

from trl import SFTTrainer, SFTConfig

print("SFTTrainer and SFTConfig loaded successfully!")

TRL version: 1.12.0
SFTTrainer and SFTConfig loaded successfully!


In [ ]:
from trl import SFTTrainer, SFTConfig

In [ ]:
training_args = SFTConfig(
    output_dir="./qwen-resume-lora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=1,
    save_strategy="no",

    fp16=True,

    max_length=1024,

    report_to="none"
)

print("Training configuration created!")

Training configuration created!


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

print("Trainer created successfully!")

Tokenizing train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/2 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1 [00:00<?, ? examples/s]

Trainer created successfully!


In [ ]:
import torch

print("GPU:", torch.cuda.get_device_name(0))
print("FP16 supported:", torch.cuda.is_available())

GPU: Tesla T4
FP16 supported: True


In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-resume-lora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    learning_rate=2e-4,

    logging_steps=1,
    save_strategy="no",

    fp16=False,
    bf16=False,

    max_length=1024,

    report_to="none"
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer
)

print("Trainer recreated successfully!")

Trainer recreated successfully!


In [ ]:
trainer.save_model("./qwen-resume-lora")

tokenizer.save_pretrained("./qwen-resume-lora")

print("LoRA adapter saved!")

LoRA adapter saved!


In [ ]:
import os

print(os.listdir("./qwen-resume-lora"))

['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'training_args.bin', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
trainer.save_model("./qwen-resume-lora")
tokenizer.save_pretrained("./qwen-resume-lora")

print("Fine-tuned LoRA adapter saved successfully!")

Fine-tuned LoRA adapter saved successfully!


In [ ]:
test_prompt = """
You are a resume optimization assistant.

JOB DESCRIPTION:
Software Engineer

Requirements:
Python
SQL
REST APIs
Git
AWS

CANDIDATE RESUME:
Skills: Python, SQL, HTML, CSS

Project:
Built a web application using Python and SQL.

Identify:
1. Matching skills
2. Missing skills
3. Safe resume improvements

Never invent candidate experience.
"""

messages = [
    {
        "role": "system",
        "content": "You are an expert resume optimization assistant."
    },
    {
        "role": "user",
        "content": test_prompt
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt"
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=400
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, let's see. The user wants me to optimize a resume for a Software Engineer position. The job description requires Python, SQL, REST APIs, Git, and AWS. The candidate's resume has skills in Python, SQL, HTML, CSS. Their project is a web app using Python and SQL.

First, I need to check for matching skills. The candidate has Python and SQL, which are both required. But the job also mentions REST APIs, Git, and AWS. The candidate doesn't have those listed. So, matching skills are Python and SQL.

Next, missing skills are REST APIs, Git, and AWS. Also, the project mentions a web application using Python and SQL. The job description includes REST APIs, so maybe the candidate should elaborate on that. But the resume doesn't mention REST APIs, so that's a gap.

For safe resume improvements, I should suggest adding REST APIs and Git to the skills. Also, the project could be expanded to mention if they used any frameworks (like Flask or Django) or if they used any version control. 

In [ ]:
import json

examples = [
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """Looking for a Software Development Engineer to design, develop,
test and maintain software. Requirements include C++, Java or Python,
data structures, algorithms, object-oriented design and problem solving.""",
        "resume": """SUMMARY:
Computer Science graduate looking for a software job.

SKILLS:
Python, C++, SQL, HTML, CSS

PROJECT:
Built a student management system using Python and SQL.""",
        "optimized_resume": """SUMMARY:
Computer Science graduate with hands-on experience building Python and SQL applications. Strong foundation in C++, data structures, algorithms and problem solving, seeking a Software Development Engineer role.

SKILLS:
C++, Python, SQL, Data Structures, Algorithms, HTML, CSS

PROJECT:
Student Management System
- Built a student management application using Python and SQL to manage and retrieve student records.
- Designed the application to handle common student data operations through a structured database."""
    },

    {
        "company": "Microsoft",
        "role": "Backend Software Engineer",
        "jd": """Develop backend services and APIs. Experience with Python,
REST APIs, SQL, databases and problem solving is preferred.""",
        "resume": """SUMMARY:
I am a hardworking graduate who knows Python.

SKILLS:
Python, SQL, JavaScript, HTML, CSS

PROJECT:
Created a website using Python and SQL.""",
        "optimized_resume": """SUMMARY:
Computer Science graduate with experience building web applications using Python and SQL. Interested in backend software development and building reliable server-side applications.

SKILLS:
Python, SQL, JavaScript, HTML, CSS

PROJECT:
Web Application
- Developed a web application using Python and SQL.
- Implemented application functionality for working with data stored in a SQL database."""
    },

    {
        "company": "Google",
        "role": "Frontend Software Engineer",
        "jd": """Build user-facing web applications using JavaScript, HTML,
CSS and modern frontend technologies.""",
        "resume": """SUMMARY:
Computer Science student interested in web development.

SKILLS:
HTML, CSS, JavaScript, Python, SQL

PROJECT:
Built a responsive portfolio website using HTML, CSS and JavaScript.""",
        "optimized_resume": """SUMMARY:
Computer Science student with hands-on experience building responsive web applications using HTML, CSS and JavaScript. Interested in frontend development and creating clear, user-friendly web experiences.

SKILLS:
JavaScript, HTML, CSS, Python, SQL

PROJECT:
Responsive Portfolio Website
- Built a responsive portfolio website using HTML, CSS and JavaScript.
- Structured the interface for consistent presentation across different screen sizes."""
    },

    {
        "company": "Accenture",
        "role": "Java Software Engineer",
        "jd": """Develop and maintain Java applications. Knowledge of OOP,
SQL, data structures and debugging is required.""",
        "resume": """SUMMARY:
Graduate with knowledge of Java and programming.

SKILLS:
Java, Python, SQL, C++

PROJECT:
Created a library management system using Java.""",
        "optimized_resume": """SUMMARY:
Computer Science graduate with experience developing Java applications and a strong foundation in object-oriented programming, SQL and problem solving.

SKILLS:
Java, OOP, SQL, C++, Python, Data Structures

PROJECT:
Library Management System
- Developed a library management application using Java.
- Applied object-oriented programming concepts to organize the application's functionality."""
    },

    {
        "company": "Amazon",
        "role": "Data Analyst",
        "jd": """Analyze datasets, create reports and work with SQL and Python
to identify useful insights.""",
        "resume": """SUMMARY:
Computer Science graduate interested in data.

SKILLS:
Python, SQL, Excel, Java

PROJECT:
Analyzed sales data using Python and created charts.""",
        "optimized_resume": """SUMMARY:
Computer Science graduate with hands-on experience using Python and SQL to work with data. Interested in data analysis roles involving data exploration, reporting and problem solving.

SKILLS:
Python, SQL, Excel, Data Analysis, Java

PROJECT:
Sales Data Analysis
- Analyzed sales data using Python to identify patterns in the dataset.
- Created charts to present findings from the analysis."""
    },

    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "jd": """Work with machine learning models and Python. Knowledge of
machine learning, data processing and model development is preferred.""",
        "resume": """SUMMARY:
Student interested in artificial intelligence.

SKILLS:
Python, Machine Learning, SQL, C++

PROJECT:
Built a machine learning model to predict house prices using Python.""",
        "optimized_resume": """SUMMARY:
Computer Science student with hands-on experience building machine learning projects using Python. Familiar with data preparation and model development, with an interest in machine learning engineering.

SKILLS:
Python, Machine Learning, SQL, C++

PROJECT:
House Price Prediction
- Built a machine learning model in Python to predict house prices.
- Prepared the dataset and trained the model to make price predictions."""
    },

    {
        "company": "Razorpay",
        "role": "Full Stack Software Engineer",
        "jd": """Develop web applications across frontend and backend components.
Experience with JavaScript, APIs, databases and web development is preferred.""",
        "resume": """SUMMARY:
Web developer with experience creating websites.

SKILLS:
JavaScript, React, Node.js, MongoDB, HTML, CSS

PROJECT:
Created an expense tracking web application.""",
        "optimized_resume": """SUMMARY:
Web developer with experience building applications using JavaScript, React and Node.js. Familiar with frontend development, backend services and database-driven applications.

SKILLS:
JavaScript, React, Node.js, MongoDB, HTML, CSS

PROJECT:
Expense Tracking Application
- Built an expense tracking web application using React and JavaScript.
- Developed backend functionality using Node.js and connected the application to MongoDB."""
    },

    {
        "company": "Intel",
        "role": "Software Engineer",
        "jd": """Develop software using C++. Strong programming fundamentals,
data structures, algorithms and debugging skills are preferred.""",
        "resume": """SUMMARY:
Engineering graduate with C++ knowledge.

SKILLS:
C++, Python, DSA, Git

PROJECT:
Implemented sorting and searching algorithms in C++.""",
        "optimized_resume": """SUMMARY:
Engineering graduate with a strong foundation in C++, data structures and algorithms. Experienced in implementing and testing common algorithms and interested in software engineering roles.

SKILLS:
C++, Data Structures, Algorithms, Python, Git

PROJECT:
Algorithms Implementation
- Implemented sorting and searching algorithms using C++.
- Tested different algorithm implementations to understand their behavior and performance."""
    },

    {
        "company": "IBM",
        "role": "DevOps Engineer",
        "jd": """Support software delivery and automation processes. Familiarity
with Linux, Git, CI/CD and cloud technologies is preferred.""",
        "resume": """SUMMARY:
Computer Science graduate with Linux and Git knowledge.

SKILLS:
Linux, Git, Python, Docker

PROJECT:
Created a Docker-based development environment.""",
        "optimized_resume": """SUMMARY:
Computer Science graduate with hands-on experience using Linux, Git and Docker. Interested in DevOps roles involving automation, development workflows and software delivery.

SKILLS:
Linux, Git, Docker, Python

PROJECT:
Docker Development Environment
- Created a Docker-based development environment for running an application consistently across environments.
- Used Git to manage project changes and maintain the codebase."""
    },

    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """Design, implement and test software. Strong programming,
data structures, algorithms and problem-solving skills are expected.""",
        "resume": """SUMMARY:
B.Tech Computer Science student seeking a software job.

SKILLS:
Python, Java, C++, SQL

PROJECTS:
Built a quiz application using Python.
Created a calculator using Java.""",
        "optimized_resume": """SUMMARY:
B.Tech Computer Science student with hands-on experience building applications using Python and Java. Strong foundation in programming, problem solving and core computer science concepts, seeking a Software Development Engineer role.

SKILLS:
Python, Java, C++, SQL, Data Structures, Algorithms

PROJECTS:

Quiz Application
- Built a quiz application using Python with functionality for presenting questions and processing answers.

Calculator Application
- Developed a calculator application using Java to perform common arithmetic operations."""
    }
]

# Convert each example into the format used for fine-tuning
training_examples = []

for item in examples:
    user_input = f"""COMPANY:
{item['company']}

TARGET ROLE:
{item['role']}

JOB DESCRIPTION:
{item['jd']}

EXISTING RESUME:
{item['resume']}

TASK:
Optimize the existing resume for the target company, role and job description.

Rules:
- Preserve the candidate's real information.
- Improve relevance and wording.
- Prioritize skills relevant to the target role.
- Humanize the writing.
- Keep the writing natural and professional.
- Do not invent skills, experience, projects, education, certifications, achievements or metrics.
"""

    assistant_output = item["optimized_resume"]

    training_examples.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert resume optimization assistant. "
                    "Optimize an existing resume for a specific company, "
                    "role and job description. Keep the candidate's "
                    "information truthful. Write naturally and professionally. "
                    "Never invent qualifications or achievements."
                )
            },
            {
                "role": "user",
                "content": user_input
            },
            {
                "role": "assistant",
                "content": assistant_output
            }
        ]
    })

# Save JSONL
with open("resume_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in training_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Created resume_dataset.jsonl")
print("Number of examples:", len(training_examples))

Created resume_dataset.jsonl
Number of examples: 10


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="resume_dataset.jsonl"
)

print(dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 10
    })
})


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="resume_dataset.jsonl"
)

# 8 training + 2 test
split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split["train"]
test_dataset = split["test"]

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

Training examples: 8
Test examples: 2


In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="resume_dataset.jsonl"
)

# 8 training + 2 test
split = dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

train_dataset = split["train"]
test_dataset = split["test"]

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

Training examples: 8
Test examples: 2


In [ ]:
print(train_dataset[0]["messages"][1]["content"])

COMPANY:
Amazon

TARGET ROLE:
Software Development Engineer

JOB DESCRIPTION:
Looking for a Software Development Engineer to design, develop,
test and maintain software. Requirements include C++, Java or Python,
data structures, algorithms, object-oriented design and problem solving.

EXISTING RESUME:
SUMMARY:
Computer Science graduate looking for a software job.

SKILLS:
Python, C++, SQL, HTML, CSS

PROJECT:
Built a student management system using Python and SQL.

TASK:
Optimize the existing resume for the target company, role and job description.

Rules:
- Preserve the candidate's real information.
- Improve relevance and wording.
- Prioritize skills relevant to the target role.
- Humanize the writing.
- Keep the writing natural and professional.
- Do not invent skills, experience, projects, education, certifications, achievements or metrics.



In [ ]:
print(train_dataset[0]["messages"][2]["content"])

SUMMARY:
Computer Science graduate with hands-on experience building Python and SQL applications. Strong foundation in C++, data structures, algorithms and problem solving, seeking a Software Development Engineer role.

SKILLS:
C++, Python, SQL, Data Structures, Algorithms, HTML, CSS

PROJECT:
Student Management System
- Built a student management application using Python and SQL to manage and retrieve student records.
- Designed the application to handle common student data operations through a structured database.


In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_chat)
test_dataset = test_dataset.map(format_chat)

print("Training dataset formatted!")
print("Test dataset formatted!")

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Training dataset formatted!
Test dataset formatted!


In [ ]:
print(train_dataset[0]["text"])

<|im_start|>system
You are an expert resume optimization assistant. Optimize an existing resume for a specific company, role and job description. Keep the candidate's information truthful. Write naturally and professionally. Never invent qualifications or achievements.<|im_end|>
<|im_start|>user
COMPANY:
Amazon

TARGET ROLE:
Software Development Engineer

JOB DESCRIPTION:
Looking for a Software Development Engineer to design, develop,
test and maintain software. Requirements include C++, Java or Python,
data structures, algorithms, object-oriented design and problem solving.

EXISTING RESUME:
SUMMARY:
Computer Science graduate looking for a software job.

SKILLS:
Python, C++, SQL, HTML, CSS

PROJECT:
Built a student management system using Python and SQL.

TASK:
Optimize the existing resume for the target company, role and job description.

Rules:
- Preserve the candidate's real information.
- Improve relevance and wording.
- Prioritize skills relevant to the target role.
- Humanize th

In [ ]:
import json

examples = [
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """
Strong programming skills in C++, Java or Python.
Knowledge of data structures, algorithms, object-oriented
programming, SQL and distributed systems is preferred.
""",
        "resume": """
SUMMARY:
Computer Science graduate looking for software engineering opportunities.

SKILLS:
Python, C++, SQL, Git

PROJECT:
Built a student management system using Python and SQL.
Implemented searching and sorting functionality.
""",
        "output": """
SUMMARY:
Computer Science graduate with hands-on experience building applications
using Python and SQL. Strong foundation in C++ and algorithmic problem
solving, seeking a Software Development Engineer role.

SKILLS:
Python, C++, SQL, Data Structures, Algorithms, Git

PROJECT:
Student Management System
- Built a student management application using Python and SQL.
- Implemented searching and sorting functionality for managing student records.
"""
    },

    {
        "company": "Microsoft",
        "role": "Software Engineer",
        "jd": """
Experience with Java, Spring Boot, REST APIs, SQL, Git and backend
application development. Knowledge of microservices is a plus.
""",
        "resume": """
SUMMARY:
Java developer interested in backend development.

SKILLS:
Java, Spring Boot, SQL, Git

PROJECT:
Developed a backend application using Java and Spring Boot.
Created REST APIs for handling application data.
""",
        "output": """
SUMMARY:
Java developer with hands-on experience building backend applications
using Java and Spring Boot. Experienced in developing REST APIs and
working with SQL databases.

SKILLS:
Java, Spring Boot, REST APIs, SQL, Git

PROJECT:
Backend Application
- Developed a backend application using Java and Spring Boot.
- Created REST APIs for handling application data.
"""
    },

    {
        "company": "Google",
        "role": "Software Engineer",
        "jd": """
Strong programming fundamentals with Python or C++.
Experience with algorithms, data structures, Linux and Git.
""",
        "resume": """
SUMMARY:
Computer Science student with Python programming experience.

SKILLS:
Python, C++, Git, Linux

PROJECT:
Implemented common sorting and searching algorithms in Python.
""",
        "output": """
SUMMARY:
Computer Science student with experience using Python and C++ and
a strong foundation in data structures and algorithms. Familiar with
Linux and Git development environments.

SKILLS:
Python, C++, Data Structures, Algorithms, Linux, Git

PROJECT:
Algorithms Implementation
- Implemented sorting and searching algorithms using Python.
- Practiced common data structure and algorithm implementations.
"""
    },

    {
        "company": "Razorpay",
        "role": "Software Engineer",
        "jd": """
Build scalable web applications using JavaScript, React, Node.js,
REST APIs and databases. Experience with MongoDB is preferred.
""",
        "resume": """
SUMMARY:
Web developer with experience building web applications.

SKILLS:
JavaScript, React, Node.js, MongoDB, HTML, CSS

PROJECT:
Built an expense tracking application using React and Node.js.
Connected the application to MongoDB.
""",
        "output": """
SUMMARY:
Web developer with experience building applications using JavaScript,
React and Node.js. Hands-on experience with REST-based application
development and MongoDB.

SKILLS:
JavaScript, React, Node.js, REST APIs, MongoDB, HTML, CSS

PROJECT:
Expense Tracking Application
- Built an expense tracking application using React and Node.js.
- Connected the application to MongoDB for storing application data.
"""
    },

    {
        "company": "IBM",
        "role": "Software Engineer",
        "jd": """
Experience with Python, SQL, APIs, Git and cloud technologies.
Knowledge of Docker and AWS is preferred.
""",
        "resume": """
SUMMARY:
Python developer with experience building backend applications.

SKILLS:
Python, SQL, Git

PROJECT:
Built a Python backend application and created REST APIs for
working with application data.
""",
        "output": """
SUMMARY:
Python developer with experience building backend applications
using Python and SQL. Experienced in developing REST APIs and
working with Git.

SKILLS:
Python, SQL, REST APIs, Git

PROJECT:
Python Backend Application
- Built a backend application using Python and SQL.
- Created REST APIs for working with application data.
"""
    },

    {
        "company": "NVIDIA",
        "role": "Software Engineer",
        "jd": """
Strong C++ programming skills with knowledge of algorithms,
data structures, Linux and debugging.
""",
        "resume": """
SUMMARY:
Engineering student with C++ programming experience.

SKILLS:
C++, Python, DSA, Linux

PROJECT:
Built a C++ application that processes and searches structured data.
""",
        "output": """
SUMMARY:
Engineering student with hands-on C++ experience and a strong
foundation in data structures and algorithms. Familiar with Linux
development environments.

SKILLS:
C++, Data Structures, Algorithms, Python, Linux

PROJECT:
C++ Data Processing Application
- Built a C++ application for processing and searching structured data.
- Applied data structures and algorithms to organize and retrieve data.
"""
    },

    {
        "company": "Accenture",
        "role": "Software Engineer",
        "jd": """
Develop applications using Java, SQL and object-oriented programming.
Experience with debugging, Git and REST APIs is preferred.
""",
        "resume": """
SUMMARY:
Graduate with Java programming knowledge.

SKILLS:
Java, SQL, Python, Git

PROJECT:
Created a library management system using Java and SQL.
""",
        "output": """
SUMMARY:
Computer Science graduate with experience developing Java applications
using object-oriented programming and SQL. Familiar with Git and
interested in software engineering roles.

SKILLS:
Java, Object-Oriented Programming, SQL, Python, Git

PROJECT:
Library Management System
- Developed a library management application using Java and SQL.
- Applied object-oriented programming concepts to structure the application.
"""
    },

    {
        "company": "Intel",
        "role": "Software Engineer",
        "jd": """
Develop software in C++. Strong knowledge of algorithms, data
structures, operating systems and computer architecture is preferred.
""",
        "resume": """
SUMMARY:
Computer Science graduate with C++ experience.

SKILLS:
C++, Python, SQL

PROJECT:
Implemented sorting and searching algorithms in C++.
""",
        "output": """
SUMMARY:
Computer Science graduate with hands-on C++ experience and a strong
foundation in data structures and algorithms.

SKILLS:
C++, Data Structures, Algorithms, Python, SQL

PROJECT:
Algorithms Implementation
- Implemented sorting and searching algorithms using C++.
- Worked with different algorithm implementations to understand
their behavior and performance.
"""
    },

    {
        "company": "Deloitte",
        "role": "Software Engineer",
        "jd": """
Experience with Python, Django, REST APIs and PostgreSQL.
Knowledge of Docker and AWS is a plus.
""",
        "resume": """
SUMMARY:
Python developer with web development experience.

SKILLS:
Python, Django, PostgreSQL, HTML, CSS

PROJECT:
Built a Django web application using PostgreSQL.
""",
        "output": """
SUMMARY:
Python developer with experience building web applications using
Django and PostgreSQL. Interested in backend software development
and API-driven applications.

SKILLS:
Python, Django, PostgreSQL, HTML, CSS

PROJECT:
Django Web Application
- Built a web application using Django and PostgreSQL.
- Developed application functionality for working with stored data.
"""
    },

    {
        "company": "Oracle",
        "role": "Backend Software Engineer",
        "jd": """
Build backend services using Java, Spring Boot, REST APIs, SQL
and relational databases.
""",
        "resume": """
SUMMARY:
Java developer with backend development experience.

SKILLS:
Java, Spring Boot, SQL, PostgreSQL

PROJECT:
Built a Java application using Spring Boot and PostgreSQL.
Implemented APIs for managing records.
""",
        "output": """
SUMMARY:
Java developer with backend development experience using Spring Boot,
SQL and PostgreSQL. Hands-on experience building APIs for database-driven
applications.

SKILLS:
Java, Spring Boot, REST APIs, SQL, PostgreSQL

PROJECT:
Backend Application
- Built a Java application using Spring Boot and PostgreSQL.
- Implemented APIs for managing application records.
"""
    }
]

# Add the common instruction to every example
training_examples = []

for item in examples:

    user_content = f"""
COMPANY:
{item["company"]}

TARGET ROLE:
{item["role"]}

JOB DESCRIPTION:
{item["jd"]}

EXISTING RESUME:
{item["resume"]}

TASK:
Optimize the existing resume specifically for this company, role and
job description.

RULES:
1. Preserve the candidate's real information.
2. Prioritize skills relevant to the target role.
3. You may add a skill to the optimized resume ONLY when that skill
   is clearly supported by the existing resume or its projects.
4. Do not add a skill merely because it appears in the JD.
5. Improve wording without changing the candidate's actual experience.
6. Humanize the writing.
7. Avoid generic AI phrases and unnecessary buzzwords.
8. Do not invent achievements, metrics, employers, projects,
   certifications, education or experience.
9. Keep the result concise and professional.
"""

    training_examples.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert resume optimization assistant. "
                    "Transform an existing resume for a specific company, "
                    "role and job description. You may surface or add skills "
                    "that are genuinely supported by the existing resume, "
                    "including skills demonstrated in projects. Never invent "
                    "qualifications. Write naturally and professionally."
                )
            },
            {
                "role": "user",
                "content": user_content
            },
            {
                "role": "assistant",
                "content": item["output"]
            }
        ]
    })

with open("resume_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in training_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Dataset created successfully!")
print("Examples:", len(training_examples))

Dataset created successfully!
Examples: 10


In [ ]:
import json

examples = [

    # 1. Software Engineer - C++ / DSA
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """
        Looking for a Software Development Engineer with strong
        programming fundamentals. Experience with C++, Python,
        data structures, algorithms, object-oriented programming,
        debugging and problem solving.
        """,
        "resume": """
        SUMMARY:
        Computer Science graduate looking for software opportunities.

        TECHNICAL SKILLS:
        C++, Python, SQL, Git

        PROJECT:
        Built a C++ student management application.
        Implemented searching and sorting functionality.
        Worked with a team of three students to complete the project.
        """,
        "output": """
        SUMMARY:
        Computer Science graduate with hands-on experience building
        applications using C++ and Python. Strong foundation in
        data structures, algorithms and problem solving, with
        experience working collaboratively on software projects.

        TECHNICAL SKILLS:
        C++, Python, SQL, Git, Data Structures, Algorithms,
        Object-Oriented Programming

        SOFT SKILLS:
        Problem Solving, Team Collaboration

        PROJECT:
        Student Management Application
        - Built a student management application using C++.
        - Implemented searching and sorting functionality.
        - Collaborated with a three-member team during development.
        """
    },

    # 2. Backend - Java
    {
        "company": "Microsoft",
        "role": "Backend Software Engineer",
        "jd": """
        Develop backend services using Java and Spring Boot.
        Experience with REST APIs, SQL, databases, Git and
        debugging is preferred.
        """,
        "resume": """
        SUMMARY:
        Java developer interested in backend development.

        TECHNICAL SKILLS:
        Java, Spring Boot, SQL, Git

        PROJECT:
        Developed a backend application using Java and Spring Boot.
        Created REST APIs for managing application data.
        Fixed bugs during testing.
        """,
        "output": """
        SUMMARY:
        Java developer with hands-on experience building backend
        applications using Java and Spring Boot. Experienced in
        developing REST APIs and working with SQL databases.

        TECHNICAL SKILLS:
        Java, Spring Boot, REST APIs, SQL, Git

        SOFT SKILLS:
        Problem Solving, Debugging

        PROJECT:
        Backend Application
        - Developed a backend application using Java and Spring Boot.
        - Created REST APIs for managing application data.
        - Debugged and fixed issues identified during testing.
        """
    },

    # 3. Frontend
    {
        "company": "Google",
        "role": "Frontend Software Engineer",
        "jd": """
        Build user-facing web applications using JavaScript,
        React, HTML and CSS. Attention to detail and collaboration
        are important for the role.
        """,
        "resume": """
        SUMMARY:
        Computer Science student interested in web development.

        TECHNICAL SKILLS:
        JavaScript, React, HTML, CSS, Git

        PROJECT:
        Built a responsive portfolio website using React.
        Worked with a designer to improve the page layout.
        """,
        "output": """
        SUMMARY:
        Computer Science student with hands-on experience building
        responsive web applications using React, JavaScript, HTML
        and CSS. Interested in creating clear and user-friendly
        web experiences.

        TECHNICAL SKILLS:
        React, JavaScript, HTML, CSS, Git

        SOFT SKILLS:
        Collaboration, Attention to Detail

        PROJECT:
        Responsive Portfolio Website
        - Built a responsive portfolio website using React,
          JavaScript, HTML and CSS.
        - Worked with a designer to improve the page layout
          and user experience.
        """
    },

    # 4. Full Stack
    {
        "company": "Razorpay",
        "role": "Full Stack Software Engineer",
        "jd": """
        Build web applications using JavaScript, React, Node.js,
        REST APIs and MongoDB. Ability to work across frontend
        and backend systems is preferred.
        """,
        "resume": """
        SUMMARY:
        Web developer with experience creating websites.

        TECHNICAL SKILLS:
        JavaScript, React, Node.js, MongoDB, HTML, CSS

        PROJECT:
        Built an expense tracking web application using React
        and Node.js. Connected the application to MongoDB.
        """,
        "output": """
        SUMMARY:
        Web developer with experience building full stack
        applications using React, JavaScript and Node.js.
        Familiar with working across frontend, backend and
        database components.

        TECHNICAL SKILLS:
        JavaScript, React, Node.js, REST APIs, MongoDB, HTML, CSS

        SOFT SKILLS:
        Adaptability, Problem Solving

        PROJECT:
        Expense Tracking Application
        - Built an expense tracking application using React
          and JavaScript.
        - Developed backend functionality using Node.js.
        - Connected the application to MongoDB for storing data.
        """
    },

    # 5. Data Analyst
    {
        "company": "Amazon",
        "role": "Data Analyst",
        "jd": """
        Analyze datasets using SQL and Python, create reports,
        identify trends and communicate findings clearly.
        Experience with Excel is preferred.
        """,
        "resume": """
        SUMMARY:
        Computer Science graduate interested in data.

        TECHNICAL SKILLS:
        Python, SQL, Excel, Java

        PROJECT:
        Analyzed sales data using Python and SQL.
        Created charts and presented the findings to classmates.
        """,
        "output": """
        SUMMARY:
        Computer Science graduate with hands-on experience using
        Python and SQL for data analysis. Experienced in exploring
        datasets, creating visualizations and presenting findings.

        TECHNICAL SKILLS:
        Python, SQL, Excel, Data Analysis, Data Visualization

        SOFT SKILLS:
        Analytical Thinking, Communication

        PROJECT:
        Sales Data Analysis
        - Analyzed sales data using Python and SQL.
        - Created charts to present patterns identified in the data.
        - Presented the analysis and findings to classmates.
        """
    },

    # 6. ML Engineer
    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "jd": """
        Work with Python and machine learning models.
        Experience with data preparation, model development
        and experimentation is preferred.
        """,
        "resume": """
        SUMMARY:
        Student interested in artificial intelligence.

        TECHNICAL SKILLS:
        Python, Machine Learning, SQL

        PROJECT:
        Built a house price prediction model using Python.
        Prepared the dataset and experimented with different
        models.
        """,
        "output": """
        SUMMARY:
        Computer Science student with hands-on experience building
        machine learning projects using Python. Experienced in
        preparing datasets and experimenting with different models.

        TECHNICAL SKILLS:
        Python, Machine Learning, SQL, Data Preprocessing,
        Model Development

        SOFT SKILLS:
        Analytical Thinking, Experimentation, Problem Solving

        PROJECT:
        House Price Prediction
        - Built a machine learning model in Python to predict
          house prices.
        - Prepared the dataset for model development.
        - Experimented with different models to compare their results.
        """
    },

    # 7. DevOps
    {
        "company": "IBM",
        "role": "DevOps Engineer",
        "jd": """
        Support development and deployment workflows.
        Experience with Linux, Git, Docker and CI/CD is preferred.
        """,
        "resume": """
        SUMMARY:
        Computer Science graduate interested in DevOps.

        TECHNICAL SKILLS:
        Linux, Git, Docker, Python

        PROJECT:
        Created a Docker-based development environment.
        Documented the setup steps so other team members could
        use the environment.
        """,
        "output": """
        SUMMARY:
        Computer Science graduate with hands-on experience using
        Linux, Git and Docker. Interested in DevOps and software
        development workflows.

        TECHNICAL SKILLS:
        Linux, Git, Docker, Python

        SOFT SKILLS:
        Documentation, Collaboration, Organization

        PROJECT:
        Docker Development Environment
        - Created a Docker-based development environment.
        - Documented the setup process for other team members.
        """
    },

    # 8. C++ Engineer
    {
        "company": "Intel",
        "role": "C++ Software Engineer",
        "jd": """
        Develop software using C++. Strong programming fundamentals,
        data structures, algorithms and debugging skills are preferred.
        """,
        "resume": """
        SUMMARY:
        Engineering graduate with C++ experience.

        TECHNICAL SKILLS:
        C++, Python, Git

        PROJECT:
        Implemented sorting and searching algorithms in C++.
        Tested the implementations and investigated incorrect results.
        """,
        "output": """
        SUMMARY:
        Engineering graduate with hands-on C++ experience and a
        strong foundation in data structures and algorithms.
        Experienced in testing and debugging algorithm implementations.

        TECHNICAL SKILLS:
        C++, Python, Git, Data Structures, Algorithms

        SOFT SKILLS:
        Analytical Thinking, Debugging, Attention to Detail

        PROJECT:
        Algorithms Implementation
        - Implemented sorting and searching algorithms using C++.
        - Tested the implementations and investigated incorrect results.
        """
    },

    # 9. Python Backend
    {
        "company": "Deloitte",
        "role": "Python Software Engineer",
        "jd": """
        Develop backend applications using Python and Django.
        Experience with REST APIs, PostgreSQL and Git is preferred.
        """,
        "resume": """
        SUMMARY:
        Python developer with web development experience.

        TECHNICAL SKILLS:
        Python, Django, PostgreSQL, Git

        PROJECT:
        Built a Django web application using PostgreSQL.
        Worked with another student to develop the application.
        """,
        "output": """
        SUMMARY:
        Python developer with experience building web applications
        using Django and PostgreSQL. Familiar with backend development
        and collaborative software projects.

        TECHNICAL SKILLS:
        Python, Django, PostgreSQL, Git

        SOFT SKILLS:
        Collaboration, Communication

        PROJECT:
        Django Web Application
        - Built a web application using Django and PostgreSQL.
        - Worked with another student during application development.
        """
    },

    # 10. Fresher Software Engineer
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """
        Design, implement and test software. Strong programming,
        data structures, algorithms and problem-solving skills
        are expected.
        """,
        "resume": """
        SUMMARY:
        B.Tech Computer Science student seeking a software job.

        TECHNICAL SKILLS:
        Python, Java, C++, SQL

        PROJECTS:
        Built a quiz application using Python.
        Created a calculator using Java.
        Solved programming problems using C++.
        """,
        "output": """
        SUMMARY:
        B.Tech Computer Science student with hands-on experience
        building applications using Python and Java. Strong
        foundation in programming, data structures and algorithms,
        with an interest in software engineering.

        TECHNICAL SKILLS:
        Python, Java, C++, SQL, Data Structures, Algorithms

        SOFT SKILLS:
        Problem Solving, Analytical Thinking

        PROJECTS:

        Quiz Application
        - Built a quiz application using Python.

        Calculator Application
        - Developed a calculator application using Java.

        Programming Practice
        - Solved programming problems using C++ to strengthen
          data structures and algorithmic problem-solving skills.
        """
    }
]

# Convert examples into chat fine-tuning format
training_examples = []

for item in examples:

    user_prompt = f"""
COMPANY:
{item["company"]}

TARGET ROLE:
{item["role"]}

JOB DESCRIPTION:
{item["jd"]}

EXISTING RESUME:
{item["resume"]}

TASK:
Optimize the existing resume for the company, target role and
job description.

IMPORTANT RULES:

1. Preserve the candidate's actual information.
2. Identify relevant technical skills from the existing resume.
3. You may surface a technical skill that is demonstrated inside
   an existing project even if it was not listed in the original
   Skills section.
4. You may add role-relevant skills only when the resume or project
   content provides evidence for them.
5. Never add a skill only because it appears in the JD.
6. Do not invent work experience, projects, certifications,
   education, achievements or numerical results.
7. Improve the professional summary based on the target role.
8. Prioritize relevant technical skills.
9. Include soft skills only when they are supported by the
   candidate's resume or project descriptions.
10. Do not use the same soft-skill list for every candidate.
11. Use different soft skills when the evidence supports them.
12. Humanize the writing.
13. Avoid generic AI phrases such as "results-driven",
    "passionate professional", "proven track record",
    "leveraged cutting-edge technologies", etc.
14. Keep the writing concise, natural and professional.
15. Preserve the candidate's actual level of experience.
"""

    training_examples.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert resume optimization assistant. "
                    "You optimize an existing resume for a specific "
                    "company, role and job description. You may surface "
                    "skills demonstrated by the candidate's existing "
                    "experience or projects. You must never invent "
                    "qualifications. Write naturally, concisely and "
                    "professionally."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": item["output"]
            }
        ]
    })

# Save as JSONL
with open("resume_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in training_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Dataset created successfully!")
print("Number of examples:", len(training_examples))
print("File: resume_dataset.jsonl")

Dataset created successfully!
Number of examples: 10
File: resume_dataset.jsonl


In [ ]:
import random
import json

roles = {
    "Software Engineer": [
        ["Python", "SQL", "Git", "Data Structures", "Algorithms"],
        ["C++", "Python", "Git", "Data Structures", "Algorithms"],
        ["Java", "SQL", "Spring Boot", "Git"],
        ["JavaScript", "React", "Node.js", "MongoDB"],
        ["Python", "Django", "PostgreSQL", "REST APIs"],
        ["C++", "Linux", "Git", "Algorithms"],
        ["Java", "Spring Boot", "REST APIs", "PostgreSQL"],
        ["Python", "FastAPI", "SQL", "Git"],
        ["JavaScript", "TypeScript", "React", "Node.js"],
        ["Go", "REST APIs", "Docker", "Git"]
    ],

    "Backend Engineer": [
        ["Python", "Django", "PostgreSQL", "REST APIs"],
        ["Java", "Spring Boot", "SQL", "REST APIs"],
        ["Node.js", "Express", "MongoDB", "REST APIs"],
        ["Python", "FastAPI", "PostgreSQL", "Docker"],
        ["Java", "Spring Boot", "MySQL", "Git"]
    ],

    "Frontend Engineer": [
        ["HTML", "CSS", "JavaScript", "React"],
        ["JavaScript", "TypeScript", "React", "Git"],
        ["React", "JavaScript", "CSS", "REST APIs"],
        ["Angular", "TypeScript", "HTML", "CSS"],
        ["Vue.js", "JavaScript", "HTML", "CSS"]
    ],

    "Data Analyst": [
        ["Python", "SQL", "Excel", "Data Analysis"],
        ["SQL", "Excel", "Power BI", "Data Visualization"],
        ["Python", "Pandas", "SQL", "Data Visualization"],
        ["SQL", "Tableau", "Excel", "Data Analysis"]
    ],

    "ML Engineer": [
        ["Python", "Machine Learning", "SQL", "Pandas"],
        ["Python", "PyTorch", "Machine Learning", "NumPy"],
        ["Python", "Scikit-learn", "Pandas", "SQL"],
        ["Python", "TensorFlow", "Machine Learning", "NumPy"]
    ],

    "DevOps Engineer": [
        ["Linux", "Git", "Docker", "Python"],
        ["Docker", "Kubernetes", "Linux", "Git"],
        ["AWS", "Docker", "Linux", "CI/CD"],
        ["Azure", "Docker", "Git", "CI/CD"]
    ]
}

soft_skills = [
    "Problem Solving",
    "Analytical Thinking",
    "Team Collaboration",
    "Communication",
    "Attention to Detail",
    "Adaptability",
    "Debugging",
    "Organization",
    "Documentation",
    "Critical Thinking"
]

companies = [
    "Amazon",
    "Microsoft",
    "Google",
    "NVIDIA",
    "IBM",
    "Oracle",
    "Intel",
    "Accenture",
    "Deloitte",
    "Razorpay"
]

def create_example(role, skills):

    company = random.choice(companies)

    # Pick 2-4 skills as candidate skills.
    candidate_skills = random.sample(
        skills,
        random.randint(2, min(4, len(skills)))
    )

    # Pick soft skills independently.
    candidate_soft_skills = random.sample(
        soft_skills,
        random.randint(2, 3)
    )

    # Create a simple candidate resume.
    resume = f"""
SUMMARY:
Computer Science graduate interested in {role.lower()}.

TECHNICAL SKILLS:
{", ".join(candidate_skills)}

PROJECT:
Built a project using {candidate_skills[0]} and
worked with {candidate_skills[1]} during development.

EXPERIENCE:
Worked with a team to develop and test software.
"""

    # JD contains the full skill set.
    jd = f"""
We are looking for a {role} with experience in:

{", ".join(skills)}

The candidate should demonstrate problem solving,
collaboration and the ability to work on software projects.
"""

    # Only skills actually present in the resume can be added.
    output = f"""
SUMMARY:
Computer Science graduate interested in {role.lower()} with
hands-on experience using {candidate_skills[0]} and
{candidate_skills[1]}.

TECHNICAL SKILLS:
{", ".join(candidate_skills)}

SOFT SKILLS:
{", ".join(candidate_soft_skills)}

PROJECT:
- Built a project using {candidate_skills[0]} and
  {candidate_skills[1]}.
- Worked with a team during development and testing.
"""

    return {
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert resume optimization assistant. "
                    "Optimize the candidate's existing resume for the "
                    "company, role and job description. Surface skills "
                    "only when supported by the existing resume. "
                    "Use role-relevant soft skills only when supported. "
                    "Never invent qualifications, experience, projects, "
                    "achievements or metrics. Write naturally and "
                    "professionally without generic AI language."
                )
            },
            {
                "role": "user",
                "content": f"""
COMPANY:
{company}

TARGET ROLE:
{role}

JOB DESCRIPTION:
{jd}

EXISTING RESUME:
{resume}

TASK:
Create an enhanced version of the existing resume for this
specific role and job description.

Preserve the candidate's actual experience.
Prioritize relevant skills.
Improve the summary.
Improve project/experience wording.
Use natural human-sounding language.
Do not invent qualifications.
"""
            },
            {
                "role": "assistant",
                "content": output
            }
        ]
    }


# Generate examples
all_examples = []

for role, skill_sets in roles.items():

    for skills in skill_sets:

        for _ in range(5):
            all_examples.append(
                create_example(role, skills)
            )


# Shuffle
random.shuffle(all_examples)

# Save
with open("resume_large_dataset.jsonl", "w", encoding="utf-8") as f:
    for example in all_examples:
        f.write(
            json.dumps(example, ensure_ascii=False) + "\n"
        )

print("Dataset created!")
print("Total examples:", len(all_examples))
print("File: resume_large_dataset.jsonl")

Dataset created!
Total examples: 160
File: resume_large_dataset.jsonl


In [ ]:
from datasets import load_dataset

large_dataset = load_dataset(
    "json",
    data_files="resume_large_dataset.jsonl"
)

print(large_dataset)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 160
    })
})


In [ ]:
print(large_dataset["train"][0]["messages"][1]["content"])
print("--------------------------------------------------")
print(large_dataset["train"][0]["messages"][2]["content"])


COMPANY:
Deloitte

TARGET ROLE:
Frontend Engineer

JOB DESCRIPTION:

We are looking for a Frontend Engineer with experience in:

Vue.js, JavaScript, HTML, CSS

The candidate should demonstrate problem solving,
collaboration and the ability to work on software projects.


EXISTING RESUME:

SUMMARY:
Computer Science graduate interested in frontend engineer.

TECHNICAL SKILLS:
CSS, JavaScript, Vue.js, HTML

PROJECT:
Built a project using CSS and
worked with JavaScript during development.

EXPERIENCE:
Worked with a team to develop and test software.


TASK:
Create an enhanced version of the existing resume for this
specific role and job description.

Preserve the candidate's actual experience.
Prioritize relevant skills.
Improve the summary.
Improve project/experience wording.
Use natural human-sounding language.
Do not invent qualifications.

--------------------------------------------------

SUMMARY:
Computer Science graduate interested in frontend engineer with
hands-on experience usi

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="resume_large_dataset.jsonl"
)["train"]

# First: 90% train, 10% temporary
split_1 = dataset.train_test_split(
    test_size=0.10,
    seed=42
)

train_dataset = split_1["train"]
temp_dataset = split_1["test"]

# Split the temporary 10% into validation and test
split_2 = temp_dataset.train_test_split(
    test_size=0.50,
    seed=42
)

validation_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))
print("Test examples:", len(test_dataset))

Training examples: 144
Validation examples: 8
Test examples: 8


In [ ]:
print("TRAIN:")
print(train_dataset[0]["messages"][1]["content"][:500])

print("\n---------------------------\n")

print("VALIDATION:")
print(validation_dataset[0]["messages"][1]["content"][:500])

print("\n---------------------------\n")

print("TEST:")
print(test_dataset[0]["messages"][1]["content"][:500])

TRAIN:

COMPANY:
NVIDIA

TARGET ROLE:
Frontend Engineer

JOB DESCRIPTION:

We are looking for a Frontend Engineer with experience in:

React, JavaScript, CSS, REST APIs

The candidate should demonstrate problem solving,
collaboration and the ability to work on software projects.


EXISTING RESUME:

SUMMARY:
Computer Science graduate interested in frontend engineer.

TECHNICAL SKILLS:
React, CSS, REST APIs

PROJECT:
Built a project using React and
worked with CSS during development.

EXPERIENCE:
Worked 

---------------------------

VALIDATION:

COMPANY:
IBM

TARGET ROLE:
ML Engineer

JOB DESCRIPTION:

We are looking for a ML Engineer with experience in:

Python, Machine Learning, SQL, Pandas

The candidate should demonstrate problem solving,
collaboration and the ability to work on software projects.


EXISTING RESUME:

SUMMARY:
Computer Science graduate interested in ml engineer.

TECHNICAL SKILLS:
Pandas, Machine Learning

PROJECT:
Built a project using Pandas and
worked with Machine

In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(
    format_chat,
    remove_columns=["messages"]
)

validation_dataset = validation_dataset.map(
    format_chat,
    remove_columns=["messages"]
)

test_dataset = test_dataset.map(
    format_chat,
    remove_columns=["messages"]
)

print("All datasets formatted for Qwen!")

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

All datasets formatted for Qwen!


In [ ]:
print(train_dataset[0]["text"][:2000])

<|im_start|>system
You are an expert resume optimization assistant. Optimize the candidate's existing resume for the company, role and job description. Surface skills only when supported by the existing resume. Use role-relevant soft skills only when supported. Never invent qualifications, experience, projects, achievements or metrics. Write naturally and professionally without generic AI language.<|im_end|>
<|im_start|>user

COMPANY:
NVIDIA

TARGET ROLE:
Frontend Engineer

JOB DESCRIPTION:

We are looking for a Frontend Engineer with experience in:

React, JavaScript, CSS, REST APIs

The candidate should demonstrate problem solving,
collaboration and the ability to work on software projects.


EXISTING RESUME:

SUMMARY:
Computer Science graduate interested in frontend engineer.

TECHNICAL SKILLS:
React, CSS, REST APIs

PROJECT:
Built a project using React and
worked with CSS during development.

EXPERIENCE:
Worked with a team to develop and test software.


TASK:
Create an enhanced ve

In [ ]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-resume-lora",

    # Training
    num_train_epochs=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    # Learning
    learning_rate=2e-4,

    # Evaluation
    eval_strategy="steps",
    eval_steps=25,

    # Logging
    logging_steps=5,

    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    # T4-safe settings
    fp16=False,
    bf16=False,

    # Sequence length
    max_length=1024,

    # Disable external logging
    report_to="none",

    # Don't remove columns automatically
    dataset_kwargs={
        "skip_prepare_dataset": True
    }
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=validation_dataset,

    processing_class=tokenizer
)

print("QLoRA trainer created successfully!")

QLoRA trainer created successfully!


In [ ]:
model.print_trainable_parameters()

trainable params: 0 || all params: 4,055,498,240 || trainable%: 0.0000


In [ ]:
print(train_dataset.column_names)

['text']


In [ ]:
def tokenize_dataset(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,
        padding=False
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_dataset,
    remove_columns=train_dataset.column_names
)

validation_tokenized = validation_dataset.map(
    tokenize_dataset,
    remove_columns=validation_dataset.column_names
)

test_tokenized = test_dataset.map(
    tokenize_dataset,
    remove_columns=test_dataset.column_names
)

print(train_tokenized.column_names)

Map:   0%|          | 0/144 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']


In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./qwen-resume-lora",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    report_to="none",

    remove_unused_columns=False
)

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized
)

print("Trainer created successfully!")

Trainer created successfully!


In [ ]:
print(train_tokenized[0].keys())
print("Training:", len(train_tokenized))
print("Validation:", len(validation_tokenized))

dict_keys(['input_ids', 'attention_mask', 'labels'])
Training: 144
Validation: 8


In [ ]:
trainer.save_model("./qwen-resume-lora")
tokenizer.save_pretrained("./qwen-resume-lora")

print("LoRA model saved successfully!")

LoRA model saved successfully!


In [ ]:
import os

print(os.listdir("./qwen-resume-lora"))

['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'training_args.bin', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    quantization_config=bnb_config,
    device_map="auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    "./qwen-resume-lora"
)

fine_tuned_model.eval()

print("Fine-tuned model loaded!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Fine-tuned model loaded!


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": (
            "You are an expert resume optimization assistant. "
            "Optimize the existing resume for the target company, "
            "role and job description. Keep all information truthful. "
            "You may surface skills demonstrated by the resume or "
            "projects. Do not invent qualifications. Write naturally "
            "and professionally."
        )
    },
    {
        "role": "user",
        "content": """
COMPANY:
Amazon

TARGET ROLE:
Software Development Engineer

JOB DESCRIPTION:
We are looking for a Software Development Engineer with experience
in Java, Python or C++. Strong knowledge of data structures,
algorithms, object-oriented programming, SQL and problem solving
is preferred. Experience with cloud technologies is a plus.

EXISTING RESUME:

SUMMARY:
B.Tech Computer Science graduate looking for a software engineering role.

SKILLS:
Python, Java, SQL, HTML, CSS, Git

PROJECTS:

Online Book Store
- Built an online book store using Python.
- Used SQL to store and retrieve book information.
- Worked with two classmates during development.

Student Quiz Application
- Created a quiz application using Java.
- Implemented question and answer functionality.

TASK:
Create an enhanced version of the existing resume.

Improve:
1. Professional Summary
2. Technical Skills
3. Soft Skills when supported by the resume
4. Project descriptions

Make it relevant to the target role while keeping it natural
and human-sounding.

Do not invent AWS, cloud experience, metrics, certifications,
employment or other qualifications.
"""
    }
]

In [ ]:
inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to(fine_tuned_model.device)

with torch.no_grad():
    outputs = fine_tuned_model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)

<think>
Okay, let's tackle this resume optimization for Amazon's Software Development Engineer role. The user wants to enhance the existing resume with a professional summary, technical skills, soft skills, and project descriptions. First, I need to make sure all the changes are based on the existing resume without inventing any qualifications.

Starting with the Summary. The original is pretty brief. I should expand it to highlight relevant experience and skills aligned with Amazon's needs. Mentioning B.Tech in Computer Science is good, but adding a line about passion for software engineering and a focus on problem-solving and cloud technologies would make it more tailored.

Next, Technical Skills. The existing resume lists Python, Java, SQL, HTML, CSS, Git. The job description emphasizes Java, Python, C++, data structures, algorithms, OOP, SQL, cloud tech. The user's resume has Java, Python, SQL, but not C++. Should I include C++? The original resume doesn�t have it, so maybe not. Bu

In [ ]:
test_cases = [
    {
        "company": "Microsoft",
        "role": "Software Engineer",
        "jd": """
        Looking for a Software Engineer with experience in Java,
        Python, SQL, REST APIs, Git and problem solving.
        """,
        "resume": """
        SUMMARY:
        Computer Science graduate looking for a software job.

        SKILLS:
        Java, Python, SQL, Git

        PROJECT:
        Built a Java application for managing student records.
        Used SQL to store student information.
        Worked with two classmates on the project.
        """
    },

    {
        "company": "Google",
        "role": "Frontend Software Engineer",
        "jd": """
        Build web applications using JavaScript, React, TypeScript,
        HTML and CSS. Attention to detail and collaboration are valued.
        """,
        "resume": """
        SUMMARY:
        Computer Science student interested in web development.

        SKILLS:
        JavaScript, React, HTML, CSS

        PROJECT:
        Built a portfolio website using React and JavaScript.
        Improved the layout after receiving feedback from classmates.
        """
    },

    {
        "company": "NVIDIA",
        "role": "Software Engineer",
        "jd": """
        Develop software using C++. Experience with algorithms,
        data structures, Linux and debugging is preferred.
        """,
        "resume": """
        SUMMARY:
        Engineering graduate with programming experience.

        SKILLS:
        C++, Python, Git, Linux

        PROJECT:
        Implemented sorting and searching algorithms in C++.
        Tested the programs and fixed incorrect outputs.
        """
    },

    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """
        Strong programming fundamentals in Python, Java or C++.
        Knowledge of data structures, algorithms, SQL, AWS and Docker.
        """,
        "resume": """
        SUMMARY:
        B.Tech Computer Science graduate.

        SKILLS:
        Python, SQL, Git

        PROJECT:
        Built a Python application that stores information
        in a SQL database.
        """
    },

    {
        "company": "Razorpay",
        "role": "Backend Software Engineer",
        "jd": """
        Develop backend services using Node.js, REST APIs,
        PostgreSQL, Docker and cloud technologies.
        """,
        "resume": """
        SUMMARY:
        Web developer with backend experience.

        SKILLS:
        Node.js, JavaScript, PostgreSQL, Git

        PROJECT:
        Built a Node.js application connected to PostgreSQL.
        Created APIs for retrieving and updating application data.
        """
    }
]

In [ ]:
def generate_resume(case):

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert resume optimization assistant. "
                "Optimize the existing resume for the target company, "
                "role and job description. Improve wording and relevance "
                "while preserving truthful information. Skills may be "
                "surfaced when demonstrated by the resume or projects. "
                "Do not invent qualifications, experience, metrics or "
                "technologies. Use natural human-sounding language."
            )
        },
        {
            "role": "user",
            "content": f"""
COMPANY:
{case["company"]}

TARGET ROLE:
{case["role"]}

JOB DESCRIPTION:
{case["jd"]}

EXISTING RESUME:
{case["resume"]}

Create an enhanced version of the existing resume.

Improve:
- Summary
- Technical Skills
- Soft Skills when supported
- Project/experience wording

Keep the resume truthful and natural.
"""
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(fine_tuned_model.device)

    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False
        )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

In [ ]:
for i, case in enumerate(test_cases, 1):

    print("=" * 70)
    print(f"TEST CASE {i}")
    print("=" * 70)

    result = generate_resume(case)

    print(result)
    print()

TEST CASE 1
<think>
Okay, let's see. The user wants to optimize their resume for a Software Engineer position at Microsoft, focusing on Java, Python, SQL, REST APIs, Git, and problem-solving. The existing resume has a summary, skills, and a project. I need to enhance each section while keeping it truthful and natural.

First, the summary. The original says "Computer Science graduate looking for a software job." That's a bit generic. Maybe make it more specific and align it with Microsoft's values. Mention problem-solving and collaboration, which are important for Microsoft. Also, include the technologies they have.

Next, technical skills. The original lists Java, Python, SQL, Git. The job description mentions REST APIs, which the user didn't have. Wait, the existing resume doesn't mention REST APIs. But the job description does. Hmm, the user's resume doesn't have that. But the job description says "experience in Java, Python, SQL, REST APIs, Git and problem solving." So maybe the use

In [ ]:
extra_examples = [

    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """
        Experience with Python, SQL, REST APIs, AWS and Docker.
        Strong problem-solving and communication skills are preferred.
        """,
        "resume": """
        SUMMARY:
        Computer Science graduate with Python experience.

        SKILLS:
        Python, SQL, Git

        PROJECT:
        Built a Python backend application.
        Created REST APIs to allow users to retrieve and update data.
        Worked with two teammates during development.
        """,
        "output": """
        SUMMARY:
        Computer Science graduate with hands-on experience building
        Python backend applications. Experienced in working with SQL
        and REST APIs and collaborating on software projects.

        TECHNICAL SKILLS:
        Python, SQL, REST APIs, Git

        SOFT SKILLS:
        Team Collaboration, Problem Solving

        PROJECT:
        Python Backend Application
        - Built a backend application using Python and SQL.
        - Created REST APIs for retrieving and updating application data.
        - Collaborated with two teammates during development.
        """
    },

    {
        "company": "Microsoft",
        "role": "Software Engineer",
        "jd": """
        Strong experience with Java, Spring Boot, REST APIs, Azure,
        Docker and microservices.
        """,
        "resume": """
        SUMMARY:
        Java developer with backend development experience.

        SKILLS:
        Java, Spring Boot, SQL, Git

        PROJECT:
        Developed a Spring Boot application.
        Created REST APIs for managing customer records.
        Fixed bugs during testing.
        """,
        "output": """
        SUMMARY:
        Java developer with hands-on experience building backend
        applications using Java and Spring Boot. Experienced in
        developing REST APIs and debugging application issues.

        TECHNICAL SKILLS:
        Java, Spring Boot, REST APIs, SQL, Git

        SOFT SKILLS:
        Problem Solving, Attention to Detail

        PROJECT:
        Backend Application
        - Developed a backend application using Java and Spring Boot.
        - Created REST APIs for managing customer records.
        - Debugged and fixed issues identified during testing.
        """
    },

    {
        "company": "Google",
        "role": "Software Engineer",
        "jd": """
        Experience with C++, algorithms, data structures, Linux,
        distributed systems and system design.
        """,
        "resume": """
        SUMMARY:
        Computer Science student interested in software engineering.

        SKILLS:
        C++, Python, Git

        PROJECT:
        Implemented sorting and searching algorithms in C++.
        Compared the execution time of different implementations.
        """,
        "output": """
        SUMMARY:
        Computer Science student with hands-on C++ experience and
        a strong foundation in algorithms and data structures.
        Experienced in comparing algorithm implementations.

        TECHNICAL SKILLS:
        C++, Python, Algorithms, Data Structures, Git

        SOFT SKILLS:
        Analytical Thinking, Problem Solving

        PROJECT:
        Algorithm Analysis
        - Implemented sorting and searching algorithms using C++.
        - Compared the execution time of different implementations.
        """
    },

    {
        "company": "Razorpay",
        "role": "Backend Software Engineer",
        "jd": """
        Build reliable backend services using Node.js, Express,
        PostgreSQL, REST APIs, AWS and Kubernetes.
        """,
        "resume": """
        SUMMARY:
        Web developer with backend experience.

        SKILLS:
        JavaScript, Node.js, PostgreSQL, Git

        PROJECT:
        Built a Node.js application using Express.
        Created REST APIs for adding and retrieving records.
        """,
        "output": """
        SUMMARY:
        Web developer with backend experience using JavaScript,
        Node.js and PostgreSQL. Hands-on experience developing
        REST APIs and server-side applications.

        TECHNICAL SKILLS:
        JavaScript, Node.js, Express, PostgreSQL, REST APIs, Git

        SOFT SKILLS:
        Problem Solving, Adaptability

        PROJECT:
        Node.js Backend Application
        - Built a backend application using Node.js and Express.
        - Created REST APIs for adding and retrieving records.
        """
    },

    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "jd": """
        Develop machine learning solutions using Python, PyTorch,
        NumPy, model evaluation and cloud platforms.
        """,
        "resume": """
        SUMMARY:
        Computer Science student interested in machine learning.

        SKILLS:
        Python, NumPy, Machine Learning

        PROJECT:
        Built a machine learning model using Python and NumPy.
        Prepared the dataset and evaluated model predictions.
        """,
        "output": """
        SUMMARY:
        Computer Science student with hands-on experience building
        machine learning projects using Python and NumPy. Experienced
        in preparing datasets and evaluating model predictions.

        TECHNICAL SKILLS:
        Python, NumPy, Machine Learning, Data Preparation,
        Model Evaluation

        SOFT SKILLS:
        Analytical Thinking, Attention to Detail

        PROJECT:
        Machine Learning Prediction Model
        - Built a machine learning model using Python and NumPy.
        - Prepared the dataset for model training.
        - Evaluated model predictions to assess the results.
        """
    }
]

In [ ]:
import json

for item in extra_examples:

    user_prompt = f"""
COMPANY:
{item["company"]}

TARGET ROLE:
{item["role"]}

JOB DESCRIPTION:
{item["jd"]}

EXISTING RESUME:
{item["resume"]}

TASK:
Optimize the existing resume for the target company, role and JD.

RULES:
- Preserve the candidate's actual information.
- Prioritize skills relevant to the target role.
- A skill may be added to the Skills section when it is explicitly
  present or clearly demonstrated in the existing resume/projects.
- Never add a JD-only skill that the resume does not support.
- Soft skills must be supported by evidence in the resume.
- Do not use the same soft skills for every candidate.
- Improve the professional summary.
- Improve existing project and experience bullets.
- Keep the writing natural and human-sounding.
- Avoid generic AI phrases and excessive buzzwords.
- Never invent experience, achievements, metrics, certifications,
  education or technologies.
"""

    training_examples.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert resume optimization assistant. "
                    "Optimize an existing resume for a specific company, "
                    "role and job description. Surface skills that are "
                    "explicitly stated or clearly demonstrated by the "
                    "candidate. Never invent qualifications. Use natural, "
                    "concise and professional language."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": item["output"]
            }
        ]
    })

with open("resume_dataset_v2.jsonl", "w", encoding="utf-8") as f:
    for example in training_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Created:", len(training_examples), "examples")
print("File: resume_dataset_v2.jsonl")

Created: 15 examples
File: resume_dataset_v2.jsonl


In [ ]:
import json

for item in extra_examples:

    user_prompt = f"""
COMPANY:
{item["company"]}

TARGET ROLE:
{item["role"]}

JOB DESCRIPTION:
{item["jd"]}

EXISTING RESUME:
{item["resume"]}

TASK:
Optimize the existing resume for the target company, role and JD.

RULES:
- Preserve the candidate's actual information.
- Prioritize skills relevant to the target role.
- A skill may be added to the Skills section when it is explicitly
  present or clearly demonstrated in the existing resume/projects.
- Never add a JD-only skill that the resume does not support.
- Soft skills must be supported by evidence in the resume.
- Do not use the same soft skills for every candidate.
- Improve the professional summary.
- Improve existing project and experience bullets.
- Keep the writing natural and human-sounding.
- Avoid generic AI phrases and excessive buzzwords.
- Never invent experience, achievements, metrics, certifications,
  education or technologies.
"""

    training_examples.append({
        "messages": [
            {
                "role": "system",
                "content": (
                    "You are an expert resume optimization assistant. "
                    "Optimize an existing resume for a specific company, "
                    "role and job description. Surface skills that are "
                    "explicitly stated or clearly demonstrated by the "
                    "candidate. Never invent qualifications. Use natural, "
                    "concise and professional language."
                )
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": item["output"]
            }
        ]
    })

with open("resume_dataset_v2.jsonl", "w", encoding="utf-8") as f:
    for example in training_examples:
        f.write(json.dumps(example, ensure_ascii=False) + "\n")

print("Created:", len(training_examples), "examples")
print("File: resume_dataset_v2.jsonl")

Created: 20 examples
File: resume_dataset_v2.jsonl


In [ ]:
from datasets import load_dataset

dataset_v2 = load_dataset(
    "json",
    data_files="resume_dataset_v2.jsonl"
)

print(dataset_v2)
print("Total examples:", len(dataset_v2["train"]))

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 20
    })
})
Total examples: 20


In [ ]:
for i in range(3):
    print("=" * 80)
    print(f"EXAMPLE {i + 1}")
    print("=" * 80)

    print("\nINPUT:")
    print(dataset_v2["train"][i]["messages"][1]["content"])

    print("\nEXPECTED OUTPUT:")
    print(dataset_v2["train"][i]["messages"][2]["content"])

EXAMPLE 1

INPUT:

COMPANY:
Amazon

TARGET ROLE:
Software Development Engineer

JOB DESCRIPTION:

        Looking for a Software Development Engineer with strong
        programming fundamentals. Experience with C++, Python,
        data structures, algorithms, object-oriented programming,
        debugging and problem solving.
        

EXISTING RESUME:

        SUMMARY:
        Computer Science graduate looking for software opportunities.

        TECHNICAL SKILLS:
        C++, Python, SQL, Git

        PROJECT:
        Built a C++ student management application.
        Implemented searching and sorting functionality.
        Worked with a team of three students to complete the project.
        

TASK:
Optimize the existing resume for the company, target role and
job description.

IMPORTANT RULES:

1. Preserve the candidate's actual information.
2. Identify relevant technical skills from the existing resume.
3. You may surface a technical skill that is demonstrated inside
   an exis

In [ ]:
full_dataset = dataset_v2["train"]

# 80% training / 20% temporary
split_1 = full_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

train_dataset = split_1["train"]
temp_dataset = split_1["test"]

# Split remaining 20% into validation and test
split_2 = temp_dataset.train_test_split(
    test_size=0.50,
    seed=42
)

validation_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Training: 16
Validation: 2
Test: 2


In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False
        )
    }

train_dataset = train_dataset.map(format_chat)
validation_dataset = validation_dataset.map(format_chat)
test_dataset = test_dataset.map(format_chat)

print(train_dataset.column_names)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

['messages', 'text']


In [ ]:
def tokenize_dataset(example):

    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,
        padding=False
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens

In [ ]:
train_tokenized = train_dataset.map(
    tokenize_dataset,
    remove_columns=train_dataset.column_names
)

validation_tokenized = validation_dataset.map(
    tokenize_dataset,
    remove_columns=validation_dataset.column_names
)

test_tokenized = test_dataset.map(
    tokenize_dataset,
    remove_columns=test_dataset.column_names
)

print(train_tokenized.column_names)

Map:   0%|          | 0/16 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-resume-lora-v2",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=5,

    eval_strategy="steps",
    eval_steps=25,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    remove_unused_columns=False,

    report_to="none"
)

print("Training configuration ready!")

Training configuration ready!


In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized
)

print("Trainer created successfully!")

Trainer created successfully!


In [ ]:
model.print_trainable_parameters()

trainable params: 0 || all params: 4,055,498,240 || trainable%: 0.0000


In [ ]:
train_result = trainer.train()



AttributeError: 'CausalLMOutputWithPast' object has no attribute 'last_hidden_state'

In [ ]:
trainer.save_model("./qwen-resume-lora-v2")
tokenizer.save_pretrained("./qwen-resume-lora-v2")

print("Model saved!")

Model saved!


In [ ]:
trainer.save_model("./qwen-resume-lora-v2")
tokenizer.save_pretrained("./qwen-resume-lora-v2")

print("QLoRA v2 saved successfully!")

QLoRA v2 saved successfully!


In [ ]:
import os

files = os.listdir("./qwen-resume-lora-v2")

for f in files:
    print(f)

README.md
adapter_config.json
adapter_model.safetensors
training_args.bin
chat_template.jinja
tokenizer.json
tokenizer_config.json


In [ ]:
test_messages = [
    {
        "role": "system",
        "content": """
You are an expert resume optimization assistant.

Optimize the candidate's existing resume for the company,
target role and job description.

Rules:
- Preserve the candidate's actual information.
- Improve the summary.
- Prioritize relevant technical skills.
- Surface skills demonstrated by projects when appropriate.
- Add soft skills only when supported by the resume.
- Do not invent skills, experience, projects, certifications,
  achievements or metrics.
- Make the writing natural, concise and professional.
- Avoid generic AI-generated phrases.
"""
    },
    {
        "role": "user",
        "content": """
COMPANY:
Amazon

TARGET ROLE:
Software Development Engineer

JOB DESCRIPTION:
We are looking for engineers with experience in Python, Java
or C++. Strong knowledge of data structures, algorithms,
SQL, REST APIs and AWS is preferred.

EXISTING RESUME:

SUMMARY:
B.Tech Computer Science graduate interested in software development.

SKILLS:
Python, Java, SQL, Git

PROJECTS:

Task Management Application
- Built a task management application using Python.
- Created REST APIs for adding and updating tasks.
- Used SQL to store application data.
- Worked with two classmates during development.

Quiz Application
- Created a quiz application using Java.
- Implemented question and answer functionality.

TASK:
Create an enhanced version of this existing resume.
"""
    }
]

In [ ]:
inputs = tokenizer.apply_chat_template(
    test_messages,
    add_generation_prompt=True,
    tokenize=True,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=False
    )

result = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(result)

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
[transformers] Caching is incompatible with gradient checkpointing in Qwen3DecoderLayer. Setting `past_key_values=None`.
/usr/local/lib/python3.13/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


<think> Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers Answers 

In [ ]:
import torch
from transformers import AutoModelForCausalLM
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-4B"
ADAPTER_PATH = "./qwen-resume-lora-v2"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

fine_tuned_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

fine_tuned_model.eval()

print("Fine-tuned model loaded!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Fine-tuned model loaded!


/usr/local/lib/python3.13/dist-packages/peft/peft_model.py:665: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.o_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.gate_proj.lora_B.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_A.default.weight', 'base_model.model.model.layers.0.mlp.up_proj.lora_B.default.we

In [ ]:
test_messages = [
    {
        "role": "system",
        "content": """
You are an expert resume optimization assistant.

Optimize the candidate's existing resume for the target company,
role and job description.

Rules:
- Preserve the candidate's actual information.
- Improve the professional summary.
- Prioritize relevant technical skills.
- Surface skills demonstrated by projects.
- Add soft skills only when supported by the resume.
- Do not invent skills, experience, projects, certifications,
  achievements or metrics.
- Write naturally and professionally.
- Avoid generic AI-generated wording.
"""
    },
    {
        "role": "user",
        "content": """
COMPANY:
Amazon

TARGET ROLE:
Software Development Engineer

JOB DESCRIPTION:
We are looking for engineers with experience in Python, Java
or C++. Strong knowledge of data structures, algorithms,
SQL, REST APIs and AWS is preferred.

EXISTING RESUME:

SUMMARY:
B.Tech Computer Science graduate interested in software development.

SKILLS:
Python, Java, SQL, Git

PROJECTS:

Task Management Application
- Built a task management application using Python.
- Created REST APIs for adding and updating tasks.
- Used SQL to store application data.
- Worked with two classmates during development.

Quiz Application
- Created a quiz application using Java.
- Implemented question and answer functionality.

TASK:
Create an enhanced version of this existing resume.
"""
    }
]

In [ ]:
def generate_with_model(model, messages):

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [ ]:
fine_tuned_result = generate_with_model(
    fine_tuned_model,
    test_messages
)

print("===== FINE-TUNED MODEL =====")
print(fine_tuned_result)

===== FINE-TUNED MODEL =====
<think>
Okay, let's see. The user wants to optimize their resume for Amazon's Software Development Engineer role. The existing resume has a summary that's pretty basic, and the projects need more detail. The job description mentions Python, Java, C++, data structures, algorithms, SQL, REST APIs, and AWS. The candidate has Python, Java, SQL, Git as skills. Projects include a task management app in Python with REST APIs and SQL, and a quiz app in Java.

First, the summary needs to be more professional and highlight relevant experience. The original says "B.Tech Computer Science graduate interested in software development." That's too generic. Maybe start with a stronger opening, like "Experienced software developer with a focus on..." and mention specific skills and experience.

Next, the skills section. The existing ones are Python, Java, SQL, Git. The job description also mentions C++, data structures, algorithms, REST APIs, AWS. The candidate might not hav

In [ ]:
base_only_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)

base_only_model.eval()

base_result = generate_with_model(
    base_only_model,
    test_messages
)

print("===== BASE QWEN =====")
print(base_result)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

===== BASE QWEN =====
<think>
Okay, let's see. The user wants to optimize their resume for Amazon's Software Development Engineer role. The existing resume has a summary that's pretty basic, and the projects need more detail. The job description mentions Python, Java, C++, data structures, algorithms, SQL, REST APIs, and AWS. The candidate has Python, Java, SQL, Git as skills. Projects include a task management app in Python with REST APIs and SQL, and a quiz app in Java.

First, the summary needs to be more professional and highlight relevant experience. The original says "B.Tech Computer Science graduate interested in software development." That's too generic. Maybe start with a stronger opening, like "Experienced software developer with a focus on..." and mention specific skills and experience.

Next, the skills section. The existing ones are Python, Java, SQL, Git. The job description also mentions C++, data structures, algorithms, REST APIs, AWS. The candidate might not have C++ e

In [ ]:
import re

def evaluate_resume(output, original_resume, jd):

    output_lower = output.lower()
    resume_lower = original_resume.lower()
    jd_lower = jd.lower()

    # Common AI-style phrases we want to discourage
    ai_phrases = [
        "results-driven",
        "highly motivated",
        "passionate professional",
        "proven track record",
        "cutting-edge",
        "leveraged cutting-edge",
        "dynamic professional",
        "detail-oriented professional",
        "seasoned professional",
        "exceptional results"
    ]

    # Common technical skills to check
    technical_skills = [
        "python", "java", "c++", "sql", "aws", "azure",
        "docker", "kubernetes", "react", "node.js",
        "spring boot", "django", "git", "linux",
        "rest apis", "mongodb", "postgresql",
        "data structures", "algorithms"
    ]

    # Find skills mentioned in JD
    jd_skills = [
        skill for skill in technical_skills
        if skill in jd_lower
    ]

    # Skills present in original resume
    original_skills = [
        skill for skill in technical_skills
        if skill in resume_lower
    ]

    # Skills appearing in output
    output_skills = [
        skill for skill in technical_skills
        if skill in output_lower
    ]

    # Skills that appeared in output but weren't supported
    unsupported_skills = [
        skill for skill in output_skills
        if skill not in original_skills
    ]

    # AI phrase detection
    detected_ai_phrases = [
        phrase for phrase in ai_phrases
        if phrase in output_lower
    ]

    # JD relevance
    relevant_skills = [
        skill for skill in output_skills
        if skill in jd_skills
    ]

    relevance_score = 0

    if jd_skills:
        relevance_score = round(
            len(relevant_skills) / len(jd_skills) * 100,
            2
        )

    # Humanization score
    humanization_score = 100

    humanization_score -= len(detected_ai_phrases) * 15

    if humanization_score < 0:
        humanization_score = 0

    return {
        "JD skills": jd_skills,
        "Original skills": original_skills,
        "Output skills": output_skills,
        "Unsupported skills": unsupported_skills,
        "AI phrases": detected_ai_phrases,
        "JD relevance %": relevance_score,
        "Humanization score": humanization_score
    }

In [ ]:
evaluation = evaluate_resume(
    fine_tuned_result,
    test_messages[1]["content"],
    test_messages[1]["content"]
)

for key, value in evaluation.items():
    print(f"{key}: {value}")

JD skills: ['python', 'java', 'c++', 'sql', 'aws', 'git', 'rest apis', 'data structures', 'algorithms']
Original skills: ['python', 'java', 'c++', 'sql', 'aws', 'git', 'rest apis', 'data structures', 'algorithms']
Output skills: ['python', 'java', 'c++', 'sql', 'aws', 'git', 'rest apis', 'data structures', 'algorithms']
Unsupported skills: []
AI phrases: []
JD relevance %: 100.0
Humanization score: 100


In [ ]:
original_resume = """
SUMMARY:
B.Tech Computer Science graduate interested in software development.

SKILLS:
Python, Java, SQL, Git

PROJECTS:

Task Management Application
- Built a task management application using Python.
- Created REST APIs for adding and updating tasks.
- Used SQL to store application data.
- Worked with two classmates during development.

Quiz Application
- Created a quiz application using Java.
- Implemented question and answer functionality.
"""

jd = """
We are looking for engineers with experience in Python, Java
or C++. Strong knowledge of data structures, algorithms,
SQL, REST APIs and AWS is preferred.
"""

evaluation = evaluate_resume(
    fine_tuned_result,
    original_resume,
    jd
)

for key, value in evaluation.items():
    print(f"{key}: {value}")

JD skills: ['python', 'java', 'c++', 'sql', 'aws', 'rest apis', 'data structures', 'algorithms']
Original skills: ['python', 'java', 'sql', 'git', 'rest apis']
Output skills: ['python', 'java', 'c++', 'sql', 'aws', 'git', 'rest apis', 'data structures', 'algorithms']
Unsupported skills: ['c++', 'aws', 'data structures', 'algorithms']
AI phrases: []
JD relevance %: 100.0
Humanization score: 100


In [ ]:
evaluation_cases = [

    # 1
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": "Python, Java, C++, SQL, data structures, algorithms, REST APIs",
        "resume": """
        SUMMARY:
        Computer Science graduate.

        SKILLS:
        Python, SQL, Git

        PROJECT:
        Built a Python application using SQL.
        Created REST APIs for managing application records.
        """
    },

    # 2
    {
        "company": "Microsoft",
        "role": "Backend Software Engineer",
        "jd": "Java, Spring Boot, REST APIs, PostgreSQL, Git, Docker",
        "resume": """
        SUMMARY:
        Java developer interested in backend development.

        SKILLS:
        Java, Spring Boot, PostgreSQL, Git

        PROJECT:
        Built a Spring Boot application.
        Created REST APIs for managing customer information.
        """
    },

    # 3
    {
        "company": "Google",
        "role": "Frontend Software Engineer",
        "jd": "JavaScript, React, TypeScript, HTML, CSS, accessibility",
        "resume": """
        SUMMARY:
        Computer Science student interested in frontend development.

        SKILLS:
        JavaScript, React, HTML, CSS

        PROJECT:
        Built a responsive portfolio website using React.
        Improved the page layout based on feedback from classmates.
        """
    },

    # 4
    {
        "company": "NVIDIA",
        "role": "C++ Software Engineer",
        "jd": "C++, algorithms, data structures, Linux, debugging",
        "resume": """
        SUMMARY:
        Engineering graduate with programming experience.

        SKILLS:
        C++, Python, Git, Linux

        PROJECT:
        Implemented sorting and searching algorithms using C++.
        Tested the programs and fixed incorrect outputs.
        """
    },

    # 5
    {
        "company": "Razorpay",
        "role": "Backend Software Engineer",
        "jd": "Node.js, Express, REST APIs, PostgreSQL, Docker, AWS",
        "resume": """
        SUMMARY:
        Web developer with backend experience.

        SKILLS:
        JavaScript, Node.js, PostgreSQL, Git

        PROJECT:
        Built a Node.js application using Express.
        Created REST APIs for retrieving and updating records.
        """
    },

    # 6
    {
        "company": "IBM",
        "role": "DevOps Engineer",
        "jd": "Linux, Git, Docker, Kubernetes, CI/CD, AWS",
        "resume": """
        SUMMARY:
        Computer Science graduate interested in DevOps.

        SKILLS:
        Linux, Git, Docker, Python

        PROJECT:
        Created a Docker-based development environment.
        Documented the setup process for other students.
        """
    },

    # 7
    {
        "company": "Deloitte",
        "role": "Data Analyst",
        "jd": "Python, SQL, Excel, Power BI, data visualization",
        "resume": """
        SUMMARY:
        Computer Science graduate interested in data.

        SKILLS:
        Python, SQL, Excel

        PROJECT:
        Analyzed sales data using Python and SQL.
        Created charts to present the results.
        Presented the findings to classmates.
        """
    },

    # 8
    {
        "company": "Oracle",
        "role": "Java Software Engineer",
        "jd": "Java, Spring Boot, SQL, REST APIs, microservices",
        "resume": """
        SUMMARY:
        Java developer.

        SKILLS:
        Java, Spring Boot, SQL

        PROJECT:
        Built a Java application using Spring Boot and SQL.
        Developed APIs for managing records.
        """
    },

    # 9
    {
        "company": "Accenture",
        "role": "Python Developer",
        "jd": "Python, Django, REST APIs, PostgreSQL, Git",
        "resume": """
        SUMMARY:
        Python developer with web development experience.

        SKILLS:
        Python, Django, PostgreSQL, Git

        PROJECT:
        Built a Django web application.
        Used PostgreSQL to store application data.
        """
    },

    # 10
    {
        "company": "Intel",
        "role": "Software Engineer",
        "jd": "C++, operating systems, algorithms, Linux, debugging",
        "resume": """
        SUMMARY:
        Computer Science graduate with C++ experience.

        SKILLS:
        C++, Python, Linux

        PROJECT:
        Built a C++ application for processing structured data.
        Debugged the application during testing.
        """
    },

    # 11
    {
        "company": "Amazon",
        "role": "Software Engineer",
        "jd": "Java, Python, SQL, AWS, Docker, algorithms",
        "resume": """
        SUMMARY:
        B.Tech graduate interested in software development.

        SKILLS:
        Java, Python, SQL, Git

        PROJECT:
        Built a Java application for managing student records.
        Used SQL to store and retrieve information.
        """
    },

    # 12
    {
        "company": "Microsoft",
        "role": "Full Stack Engineer",
        "jd": "React, JavaScript, Node.js, REST APIs, MongoDB",
        "resume": """
        SUMMARY:
        Web developer.

        SKILLS:
        JavaScript, React, Node.js, MongoDB

        PROJECT:
        Built an expense tracking application using React and Node.js.
        Connected the application to MongoDB.
        """
    },

    # 13
    {
        "company": "Google",
        "role": "Software Engineer",
        "jd": "Python, algorithms, data structures, testing, Git",
        "resume": """
        SUMMARY:
        Computer Science student.

        SKILLS:
        Python, Git

        PROJECT:
        Solved programming problems using Python.
        Wrote tests for several solutions.
        """
    },

    # 14
    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "jd": "Python, PyTorch, machine learning, NumPy, model evaluation",
        "resume": """
        SUMMARY:
        Student interested in machine learning.

        SKILLS:
        Python, NumPy, Machine Learning

        PROJECT:
        Built a machine learning model using Python and NumPy.
        Evaluated model predictions using test data.
        """
    },

    # 15
    {
        "company": "Razorpay",
        "role": "Software Engineer",
        "jd": "JavaScript, TypeScript, React, APIs, Git, testing",
        "resume": """
        SUMMARY:
        Web development student.

        SKILLS:
        JavaScript, React, Git

        PROJECT:
        Built a React web application using JavaScript.
        Tested the application and fixed UI issues.
        """
    },

    # 16
    {
        "company": "IBM",
        "role": "Python Software Engineer",
        "jd": "Python, FastAPI, PostgreSQL, REST APIs, Docker",
        "resume": """
        SUMMARY:
        Python developer.

        SKILLS:
        Python, FastAPI, PostgreSQL

        PROJECT:
        Built a backend service using FastAPI.
        Created REST APIs and stored data in PostgreSQL.
        """
    },

    # 17
    {
        "company": "Deloitte",
        "role": "Software Engineer",
        "jd": "Java, SQL, Git, OOP, debugging",
        "resume": """
        SUMMARY:
        Computer Science graduate.

        SKILLS:
        Java, SQL, Git

        PROJECT:
        Created a library management application using Java.
        Used SQL for storing library records.
        Fixed bugs during testing.
        """
    },

    # 18
    {
        "company": "Oracle",
        "role": "Backend Engineer",
        "jd": "Python, Flask, REST APIs, SQL, cloud deployment",
        "resume": """
        SUMMARY:
        Python developer with backend experience.

        SKILLS:
        Python, Flask, SQL

        PROJECT:
        Developed a Flask application.
        Created REST APIs for managing user records.
        """
    },

    # 19
    {
        "company": "Accenture",
        "role": "Software Engineer",
        "jd": "C++, Java, Python, data structures, algorithms, teamwork",
        "resume": """
        SUMMARY:
        Engineering graduate.

        SKILLS:
        C++, Python, Java

        PROJECT:
        Built a C++ application with two classmates.
        Implemented searching and sorting functionality.
        """
    },

    # 20
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": "C++, Python, Java, distributed systems, SQL, problem solving",
        "resume": """
        SUMMARY:
        Computer Science graduate seeking a software engineering role.

        SKILLS:
        C++, Python, SQL, Git

        PROJECT:
        Built a Python application for managing inventory records.
        Used SQL for storing inventory information.
        Worked with a team during development.
        """
    }
]

print("Evaluation cases:", len(evaluation_cases))

Evaluation cases: 20


In [ ]:
evaluation_results = []

for i, case in enumerate(evaluation_cases, 1):

    messages = [
        {
            "role": "system",
            "content": """
            You are an expert resume optimization assistant.

            Optimize the existing resume for the target company,
            role and job description.

            Preserve truthful information.
            Surface skills demonstrated by the resume or projects.
            Do not invent qualifications.
            Use role-relevant soft skills only when supported.
            Write naturally and professionally.
            Avoid generic AI-generated wording.
            """
        },
        {
            "role": "user",
            "content": f"""
            COMPANY:
            {case["company"]}

            TARGET ROLE:
            {case["role"]}

            JOB DESCRIPTION:
            {case["jd"]}

            EXISTING RESUME:
            {case["resume"]}

            Create the optimized resume.
            """
        }
    ]

    result = generate_with_model(
        fine_tuned_model,
        messages
    )

    evaluation_results.append({
        "case": i,
        "company": case["company"],
        "role": case["role"],
        "output": result
    })

    print(f"Completed test {i}/20")

Completed test 1/20
Completed test 2/20
Completed test 3/20
Completed test 4/20
Completed test 5/20
Completed test 6/20
Completed test 7/20
Completed test 8/20
Completed test 9/20
Completed test 10/20
Completed test 11/20
Completed test 12/20
Completed test 13/20
Completed test 14/20
Completed test 15/20
Completed test 16/20
Completed test 17/20
Completed test 18/20
Completed test 19/20


In [ ]:
for item in evaluation_results:

    print("=" * 80)
    print(f"TEST {item['case']} — {item['company']} — {item['role']}")
    print("=" * 80)

    print(item["output"])
    print()

TEST 1 — Amazon — Software Development Engineer
<think>
Okay, let's see. The user wants to optimize their resume for a Software Development Engineer position at Amazon, focusing on Python, Java, C++, SQL, data structures, algorithms, and REST APIs. The existing resume has a summary, skills, and a project section.

First, I need to make sure the resume highlights the relevant skills and experiences. The summary is pretty brief, just stating they're a CS graduate. Maybe expand that to show more about their background and passion for software development. But the user mentioned not to invent qualifications, so I should stick to what's there.

Looking at the skills section: Python, SQL, Git. The job description includes Java, C++, and REST APIs. The existing resume doesn't mention Java or C++, so maybe those are areas to improve. But the user's project uses Python and SQL, and REST APIs. The project description is a bit vague. Need to elaborate on that.

The project is "Built a Python appl

In [ ]:
import re

def evaluate_output(output, resume, jd):
    output_lower = output.lower()
    resume_lower = resume.lower()
    jd_lower = jd.lower()

    # Skills we can check
    skills = [
        "python", "java", "c++", "javascript", "typescript",
        "sql", "git", "linux", "aws", "azure", "docker",
        "kubernetes", "react", "node.js", "express",
        "spring boot", "django", "flask", "fastapi",
        "mongodb", "postgresql", "rest apis",
        "data structures", "algorithms", "machine learning",
        "pytorch", "numpy", "html", "css"
    ]

    # AI-style phrases
    ai_phrases = [
        "results-driven",
        "highly motivated",
        "passionate professional",
        "proven track record",
        "cutting-edge",
        "dynamic professional",
        "leveraged cutting-edge",
        "exceptional results",
        "seasoned professional"
    ]

    resume_skills = {
        s for s in skills
        if s in resume_lower
    }

    jd_skills = {
        s for s in skills
        if s in jd_lower
    }

    output_skills = {
        s for s in skills
        if s in output_lower
    }

    # Skills appearing in output but not in original resume
    # These require manual verification because some can be
    # legitimately demonstrated by projects.
    potentially_unsupported = output_skills - resume_skills

    relevant_output_skills = output_skills & jd_skills

    ai_detected = [
        phrase for phrase in ai_phrases
        if phrase in output_lower
    ]

    # Relevance score
    if jd_skills:
        relevance = (
            len(relevant_output_skills)
            / len(jd_skills)
        ) * 100
    else:
        relevance = 0

    # Humanization score
    humanization = max(
        0,
        100 - (len(ai_detected) * 15)
    )

    return {
        "JD skills": sorted(jd_skills),
        "Original skills": sorted(resume_skills),
        "Output skills": sorted(output_skills),
        "Potentially unsupported skills": sorted(
            potentially_unsupported
        ),
        "AI phrases": ai_detected,
        "JD relevance": round(relevance, 1),
        "Humanization": humanization
    }


scores = []

for item, case in zip(
    evaluation_results,
    evaluation_cases
):

    score = evaluate_output(
        item["output"],
        case["resume"],
        case["jd"]
    )

    scores.append({
        "test": item["case"],
        "company": case["company"],
        "role": case["role"],
        **score
    })

print("Evaluation completed!")

Evaluation completed!


In [ ]:
for score in scores:

    print("=" * 80)
    print(
        f"TEST {score['test']} | "
        f"{score['company']} | "
        f"{score['role']}"
    )

    print("JD relevance:",
          score["JD relevance"])

    print("Humanization:",
          score["Humanization"])

    print("Potentially unsupported skills:",
          score["Potentially unsupported skills"])

    print("AI phrases:",
          score["AI phrases"])

TEST 1 | Amazon | Software Development Engineer
JD relevance: 100.0
Humanization: 100
Potentially unsupported skills: ['algorithms', 'c++', 'data structures', 'django', 'flask', 'java']
AI phrases: []


In [ ]:
avg_relevance = sum(
    x["JD relevance"]
    for x in scores
) / len(scores)

avg_humanization = sum(
    x["Humanization"]
    for x in scores
) / len(scores)

ai_phrase_count = sum(
    len(x["AI phrases"])
    for x in scores
)

print("================================")
print("OVERALL EVALUATION")
print("================================")
print(
    "Average JD relevance:",
    round(avg_relevance, 2)
)
print(
    "Average humanization:",
    round(avg_humanization, 2)
)
print(
    "Total AI-style phrases:",
    ai_phrase_count
)

OVERALL EVALUATION
Average JD relevance: 100.0
Average humanization: 100.0
Total AI-style phrases: 0


In [ ]:
import json

training_examples = []

def add_example(
    company,
    role,
    jd,
    resume,
    optimized_summary,
    optimized_skills,
    soft_skills,
    optimized_experience
):
    training_examples.append({
        "messages": [
            {
                "role": "system",
                "content": """
You are an expert resume optimization assistant.

Your task is to optimize an existing resume for a specific company,
target role and job description.

Rules:

1. Preserve the candidate's actual information.
2. Use the JD to determine which existing information should be
   emphasized.
3. A technical skill may be surfaced if it is explicitly listed
   or clearly demonstrated by the candidate's projects or experience.
4. Do not add a JD skill merely because it appears in the JD.
5. Soft skills must be supported by the candidate's resume or
   demonstrated activities.
6. Do not use the same soft-skill list for every candidate.
7. Improve weak or generic wording.
8. Keep the candidate's actual experience level.
9. Never invent employers, projects, certifications, technologies,
   achievements or metrics.
10. Do not fabricate numbers.
11. Use concise, natural, human-sounding professional language.
12. Avoid generic AI phrases and excessive buzzwords.
13. Do not keyword-stuff the resume.
"""
            },
            {
                "role": "user",
                "content": f"""
COMPANY:
{company}

TARGET ROLE:
{role}

JOB DESCRIPTION:
{jd}

EXISTING RESUME:
{resume}

Optimize this existing resume for the target role.
"""
            },
            {
                "role": "assistant",
                "content": f"""
SUMMARY:
{optimized_summary}

TECHNICAL SKILLS:
{optimized_skills}

SOFT SKILLS:
{soft_skills}

EXPERIENCE / PROJECTS:
{optimized_experience}
"""
            }
        ]
    })


# ---------------------------------------------------------
# EXAMPLE 1
# ---------------------------------------------------------

add_example(
    company="Amazon",
    role="Software Development Engineer",
    jd="""
Looking for engineers with strong programming fundamentals,
Python or Java, data structures, algorithms, SQL and REST APIs.
Cloud experience is a plus.
""",
    resume="""
SUMMARY:
B.Tech Computer Science graduate looking for a software role.

SKILLS:
Python, Java, SQL, Git

PROJECT:
Task Management Application
- Built a task management application using Python.
- Created REST APIs for adding and updating tasks.
- Used SQL to store application data.
- Worked with two classmates.
""",
    optimized_summary="""
B.Tech Computer Science graduate with hands-on experience building
Python applications and working with SQL and REST APIs. Strong
foundation in programming and problem solving, seeking a Software
Development Engineer role.
""",
    optimized_skills="""
Python, Java, SQL, REST APIs, Git
""",
    soft_skills="""
Problem Solving, Team Collaboration
""",
    optimized_experience="""
Task Management Application
- Built a task management application using Python and SQL.
- Created REST APIs for adding and updating tasks.
- Collaborated with two classmates during development.
"""
)


# ---------------------------------------------------------
# EXAMPLE 2
# ---------------------------------------------------------

add_example(
    company="Microsoft",
    role="Backend Software Engineer",
    jd="""
Develop backend services using Java and Spring Boot.
Experience with REST APIs, SQL, PostgreSQL and debugging.
""",
    resume="""
SUMMARY:
Java developer interested in backend development.

SKILLS:
Java, Spring Boot, SQL, Git

PROJECT:
Customer Management Application
- Developed a backend application using Java and Spring Boot.
- Created APIs for managing customer records.
- Fixed bugs during testing.
""",
    optimized_summary="""
Java developer with hands-on experience building backend applications
using Java and Spring Boot. Experienced in developing APIs and
debugging application issues.
""",
    optimized_skills="""
Java, Spring Boot, REST APIs, SQL, Git
""",
    soft_skills="""
Problem Solving, Debugging
""",
    optimized_experience="""
Customer Management Application
- Developed a backend application using Java and Spring Boot.
- Created REST APIs for managing customer records.
- Debugged and fixed issues during testing.
"""
)


# ---------------------------------------------------------
# EXAMPLE 3
# ---------------------------------------------------------

add_example(
    company="Google",
    role="Frontend Software Engineer",
    jd="""
Build user-facing applications using JavaScript, React,
TypeScript, HTML and CSS. Attention to detail and collaboration
are valued.
""",
    resume="""
SUMMARY:
Computer Science student interested in web development.

SKILLS:
JavaScript, React, HTML, CSS, Git

PROJECT:
Portfolio Website
- Built a responsive portfolio website using React.
- Improved the page layout after receiving feedback from classmates.
""",
    optimized_summary="""
Computer Science student with hands-on experience building responsive
web applications using React, JavaScript, HTML and CSS. Interested
in frontend development and user-focused web experiences.
""",
    optimized_skills="""
JavaScript, React, HTML, CSS, Git
""",
    soft_skills="""
Collaboration, Attention to Detail
""",
    optimized_experience="""
Portfolio Website
- Built a responsive portfolio website using React and JavaScript.
- Improved the page layout based on feedback from classmates.
"""
)


# ---------------------------------------------------------
# EXAMPLE 4
# ---------------------------------------------------------

add_example(
    company="NVIDIA",
    role="C++ Software Engineer",
    jd="""
Develop software using C++. Strong knowledge of algorithms,
data structures, Linux and debugging is preferred.
""",
    resume="""
SUMMARY:
Engineering graduate with C++ programming experience.

SKILLS:
C++, Python, Git, Linux

PROJECT:
Algorithm Practice
- Implemented sorting and searching algorithms using C++.
- Tested the programs and fixed incorrect outputs.
""",
    optimized_summary="""
Engineering graduate with hands-on C++ experience and a strong
foundation in algorithms and data structures. Experienced in
testing and debugging algorithm implementations.
""",
    optimized_skills="""
C++, Python, Data Structures, Algorithms, Git, Linux
""",
    soft_skills="""
Analytical Thinking, Debugging, Attention to Detail
""",
    optimized_experience="""
Algorithm Practice
- Implemented sorting and searching algorithms using C++.
- Tested the programs and investigated incorrect outputs.
"""
)


# ---------------------------------------------------------
# EXAMPLE 5
# ---------------------------------------------------------

add_example(
    company="Razorpay",
    role="Backend Software Engineer",
    jd="""
Build backend services using Node.js, Express, REST APIs,
PostgreSQL and Docker. Cloud experience is preferred.
""",
    resume="""
SUMMARY:
Web developer with backend experience.

SKILLS:
JavaScript, Node.js, PostgreSQL, Git

PROJECT:
Expense Tracker
- Built a Node.js application using Express.
- Created APIs for adding and retrieving expenses.
- Stored application data in PostgreSQL.
""",
    optimized_summary="""
Web developer with backend experience using JavaScript and Node.js.
Hands-on experience building REST APIs and working with PostgreSQL.
""",
    optimized_skills="""
JavaScript, Node.js, Express, REST APIs, PostgreSQL, Git
""",
    soft_skills="""
Problem Solving, Organization
""",
    optimized_experience="""
Expense Tracker
- Built a backend application using Node.js and Express.
- Created REST APIs for adding and retrieving expenses.
- Stored application data in PostgreSQL.
"""
)


# ---------------------------------------------------------
# SAVE DATASET
# ---------------------------------------------------------

with open(
    "resume_training_v3.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for example in training_examples:
        f.write(
            json.dumps(
                example,
                ensure_ascii=False
            ) + "\n"
        )

print("Dataset created!")
print("Examples:", len(training_examples))
print("File: resume_training_v3.jsonl")

Dataset created!
Examples: 5
File: resume_training_v3.jsonl


In [ ]:
from datasets import load_dataset

dataset_v3 = load_dataset(
    "json",
    data_files="resume_training_v3.jsonl"
)

print(dataset_v3["train"][0]["messages"][2]["content"])

Generating train split: 0 examples [00:00, ? examples/s]


SUMMARY:

B.Tech Computer Science graduate with hands-on experience building
Python applications and working with SQL and REST APIs. Strong
foundation in programming and problem solving, seeking a Software
Development Engineer role.


TECHNICAL SKILLS:

Python, Java, SQL, REST APIs, Git


SOFT SKILLS:

Problem Solving, Team Collaboration


EXPERIENCE / PROJECTS:

Task Management Application
- Built a task management application using Python and SQL.
- Created REST APIs for adding and updating tasks.
- Collaborated with two classmates during development.




In [ ]:
# Add more high-quality examples
# We will create variations from the 5 reviewed examples
# while keeping the rules grounded in the original resume.

import random
import json

random.seed(42)

role_profiles = {
    "Software Engineer": [
        ["Python", "SQL", "Git", "REST APIs"],
        ["Java", "Spring Boot", "SQL", "Git"],
        ["C++", "Python", "Algorithms", "Linux"],
        ["JavaScript", "React", "Node.js", "Git"],
        ["Python", "Django", "PostgreSQL", "REST APIs"],
    ],

    "Backend Software Engineer": [
        ["Python", "FastAPI", "SQL", "REST APIs"],
        ["Java", "Spring Boot", "PostgreSQL", "REST APIs"],
        ["Node.js", "Express", "MongoDB", "REST APIs"],
        ["Python", "Django", "PostgreSQL", "Git"],
        ["Java", "Spring Boot", "MySQL", "Git"],
    ],

    "Frontend Software Engineer": [
        ["JavaScript", "React", "HTML", "CSS"],
        ["TypeScript", "React", "JavaScript", "Git"],
        ["JavaScript", "Angular", "HTML", "CSS"],
        ["React", "JavaScript", "REST APIs", "Git"],
        ["Vue.js", "JavaScript", "HTML", "CSS"],
    ],

    "Data Analyst": [
        ["Python", "SQL", "Excel", "Data Analysis"],
        ["SQL", "Excel", "Power BI", "Data Visualization"],
        ["Python", "Pandas", "SQL", "Data Visualization"],
        ["SQL", "Tableau", "Excel", "Data Analysis"],
        ["Python", "SQL", "Excel", "Pandas"],
    ],

    "Machine Learning Engineer": [
        ["Python", "Machine Learning", "NumPy", "Pandas"],
        ["Python", "PyTorch", "NumPy", "Machine Learning"],
        ["Python", "TensorFlow", "Pandas", "Machine Learning"],
        ["Python", "Scikit-learn", "Pandas", "SQL"],
        ["Python", "Machine Learning", "SQL", "Data Processing"],
    ],

    "DevOps Engineer": [
        ["Linux", "Git", "Docker", "Python"],
        ["Docker", "Kubernetes", "Linux", "Git"],
        ["AWS", "Docker", "Linux", "Git"],
        ["Linux", "CI/CD", "Docker", "Git"],
        ["Azure", "Docker", "Linux", "Git"],
    ]
}

company_names = [
    "Amazon",
    "Microsoft",
    "Google",
    "NVIDIA",
    "IBM",
    "Oracle",
    "Intel",
    "Accenture",
    "Deloitte",
    "Razorpay"
]

soft_skill_combinations = [
    ["Problem Solving", "Team Collaboration"],
    ["Analytical Thinking", "Attention to Detail"],
    ["Communication", "Collaboration"],
    ["Debugging", "Problem Solving"],
    ["Adaptability", "Organization"],
    ["Critical Thinking", "Communication"],
    ["Team Collaboration", "Documentation"],
    ["Analytical Thinking", "Problem Solving"],
    ["Attention to Detail", "Organization"],
    ["Communication", "Team Collaboration"]
]


def make_training_example(company, role, skills, soft_skills):

    # Candidate only possesses these skills.
    candidate_skills = random.sample(
        skills,
        random.randint(2, len(skills))
    )

    primary_skill = candidate_skills[0]

    if len(candidate_skills) > 1:
        secondary_skill = candidate_skills[1]
    else:
        secondary_skill = primary_skill

    # Build evidence for skills through the project.
    project = f"""
    PROJECT:
    {role} Project

    - Built a project using {primary_skill}.
    - Worked with {secondary_skill} during development.
    """

    # Add evidence for soft skills.
    if "Team Collaboration" in soft_skills:
        project += """
    - Collaborated with other students during development.
    """

    if "Communication" in soft_skills:
        project += """
    - Presented the project and explained the implementation.
    """

    if "Attention to Detail" in soft_skills:
        project += """
    - Reviewed the application and fixed issues during testing.
    """

    if "Documentation" in soft_skills:
        project += """
    - Documented the setup and usage of the project.
    """

    if "Problem Solving" in soft_skills:
        project += """
    - Worked through implementation issues during development.
    """

    if "Debugging" in soft_skills:
        project += """
    - Debugged issues identified during testing.
    """

    # JD contains some additional skills that the candidate
    # does NOT necessarily have.
    jd_skills = list(skills)

    extra_jd_skills = [
        "AWS",
        "Docker",
        "Kubernetes",
        "Azure",
        "Microservices",
        "System Design",
        "TypeScript"
    ]

    for skill in random.sample(
        extra_jd_skills,
        random.randint(1, 2)
    ):
        if skill not in jd_skills:
            jd_skills.append(skill)

    jd = f"""
    We are hiring a {role}.

    Required or preferred skills:
    {", ".join(jd_skills)}

    The role involves software development, problem solving,
    testing and collaboration.
    """

    resume = f"""
    SUMMARY:
    Computer Science graduate interested in {role.lower()}.

    SKILLS:
    {", ".join(candidate_skills)}

    {project}
    """

    optimized_summary = (
        f"Computer Science graduate with hands-on experience "
        f"using {primary_skill}"
    )

    if len(candidate_skills) > 1:
        optimized_summary += (
            f" and {secondary_skill}"
        )

    optimized_summary += (
        f". Interested in {role.lower()} and building "
        f"practical software solutions."
    )

    optimized_skills = ", ".join(candidate_skills)

    optimized_project = f"""
    {role} Project
    - Built a project using {primary_skill}.
    - Worked with {secondary_skill} during development.
    """

    if "Team Collaboration" in soft_skills:
        optimized_project += (
            "- Collaborated with other students during development.\n"
        )

    if "Communication" in soft_skills:
        optimized_project += (
            "- Presented the project and explained the implementation.\n"
        )

    if "Attention to Detail" in soft_skills:
        optimized_project += (
            "- Reviewed the application and fixed issues during testing.\n"
        )

    if "Documentation" in soft_skills:
        optimized_project += (
            "- Documented the setup and usage of the project.\n"
        )

    if "Problem Solving" in soft_skills:
        optimized_project += (
            "- Worked through implementation issues during development.\n"
        )

    if "Debugging" in soft_skills:
        optimized_project += (
            "- Debugged issues identified during testing.\n"
        )

    output = f"""
    SUMMARY:
    {optimized_summary}

    TECHNICAL SKILLS:
    {optimized_skills}

    SOFT SKILLS:
    {", ".join(soft_skills)}

    PROJECTS:
    {optimized_project}
    """

    user_prompt = f"""
    COMPANY:
    {company}

    TARGET ROLE:
    {role}

    JOB DESCRIPTION:
    {jd}

    EXISTING RESUME:
    {resume}

    TASK:
    Optimize the existing resume for this specific company,
    role and job description.

    Rules:
    - Preserve the candidate's actual information.
    - Prioritize relevant skills.
    - A skill can be surfaced if it is explicitly listed or
      clearly demonstrated in the resume/project.
    - Do not add JD-only skills.
    - Soft skills must be supported by evidence.
    - Do not invent experience, achievements or metrics.
    - Improve the summary.
    - Improve project wording.
    - Keep the language natural and human-sounding.
    - Avoid generic AI phrases and keyword stuffing.
    """

    return {
        "messages": [
            {
                "role": "system",
                "content": """
                You are an expert resume optimization assistant.

                Optimize an existing resume for a company, target role
                and job description.

                Preserve truthful information.
                Surface skills demonstrated by the candidate.
                Do not invent qualifications.
                Use supported soft skills.
                Write naturally, concisely and professionally.
                Avoid generic AI-generated language.
                """
            },
            {
                "role": "user",
                "content": user_prompt
            },
            {
                "role": "assistant",
                "content": output
            }
        ]
    }


# Generate 50 examples
new_examples = []

for role, skill_sets in role_profiles.items():

    for skills in skill_sets:

        for _ in range(10):

            company = random.choice(company_names)

            soft_skills = random.choice(
                soft_skill_combinations
            )

            example = make_training_example(
                company,
                role,
                skills,
                soft_skills
            )

            new_examples.append(example)


# Save dataset
with open(
    "resume_training_v4.jsonl",
    "w",
    encoding="utf-8"
) as f:

    for example in new_examples:
        f.write(
            json.dumps(
                example,
                ensure_ascii=False
            ) + "\n"
        )


print("Dataset created successfully!")
print("Examples:", len(new_examples))
print("File: resume_training_v4.jsonl")

Dataset created successfully!
Examples: 300
File: resume_training_v4.jsonl


In [ ]:
from datasets import load_dataset

v4 = load_dataset(
    "json",
    data_files="resume_training_v4.jsonl"
)

print(v4)
print("Total:", len(v4["train"]))

print("\n========== SAMPLE ==========\n")

print(
    v4["train"][0]["messages"][1]["content"]
)

print(
    "\n========== EXPECTED OUTPUT ==========\n"
)

print(
    v4["train"][0]["messages"][2]["content"]
)

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 300
    })
})
Total: 300

========== SAMPLE ==========


    COMPANY:
    Microsoft

    TARGET ROLE:
    Software Engineer

    JOB DESCRIPTION:
    
    We are hiring a Software Engineer.

    Required or preferred skills:
    Python, SQL, Git, REST APIs, System Design

    The role involves software development, problem solving,
    testing and collaboration.
    

    EXISTING RESUME:
    
    SUMMARY:
    Computer Science graduate interested in software engineer.

    SKILLS:
    Git, Python, REST APIs, SQL

    
    PROJECT:
    Software Engineer Project

    - Built a project using Git.
    - Worked with Python during development.
    
    - Collaborated with other students during development.
    
    - Worked through implementation issues during development.
    
    

    TASK:
    Optimize the existing resume for this specific company,
    role and job description.

    Rules:
    - Preserve

In [ ]:
print("tokenizer:", "tokenizer" in globals())
print("model:", "model" in globals())
print("train_dataset:", "train_dataset" in globals())
print("train_tokenized:", "train_tokenized" in globals())
print("trainer:", "trainer" in globals())

tokenizer: True
model: True
train_dataset: True
train_tokenized: True
trainer: True


In [ ]:
!pip install -U transformers datasets peft bitsandbytes accelerate trl

In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Datasets: 5.0.1
PEFT: 0.20.0
TRL: 1.12.0
CUDA available: True
GPU: Tesla T4


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("Imports successful!")
print("CUDA:", torch.cuda.is_available())

Imports successful!
CUDA: True


In [ ]:
model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")
print("Vocabulary size:", len(tokenizer))

Tokenizer loaded successfully!
Vocabulary size: 151669


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("4-bit QLoRA configuration ready!")

4-bit QLoRA configuration ready!


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Model loaded successfully!")
print("Model device:", model.device)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Model loaded successfully!
Model device: cuda:0


In [ ]:
model = prepare_model_for_kbit_training(model)

print("Model prepared for k-bit training!")

Model prepared for k-bit training!


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("LoRA configuration ready!")

LoRA configuration ready!


In [ ]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
full_dataset = load_dataset(
    "json",
    data_files="resume_training_v4.jsonl"
)["train"]

print("Total examples:", len(full_dataset))

Total examples: 300


In [ ]:
split_1 = full_dataset.train_test_split(
    test_size=0.20,
    seed=42
)

train_dataset = split_1["train"]
temp_dataset = split_1["test"]

split_2 = temp_dataset.train_test_split(
    test_size=0.50,
    seed=42
)

validation_dataset = split_2["train"]
test_dataset = split_2["test"]

print("Training:", len(train_dataset))
print("Validation:", len(validation_dataset))
print("Test:", len(test_dataset))

Training: 240
Validation: 30
Test: 30


In [ ]:
def format_chat(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"],
            tokenize=False,
            add_generation_prompt=False,
            enable_thinking=False
        )
    }

train_text = train_dataset.map(format_chat)
validation_text = validation_dataset.map(format_chat)
test_text = test_dataset.map(format_chat)

print(train_text.column_names)
print(train_text[0]["text"][:1000])

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

['messages', 'text']
<|im_start|>system

                You are an expert resume optimization assistant.

                Optimize an existing resume for a company, target role
                and job description.

                Preserve truthful information.
                Surface skills demonstrated by the candidate.
                Do not invent qualifications.
                Use supported soft skills.
                Write naturally, concisely and professionally.
                Avoid generic AI-generated language.
                <|im_end|>
<|im_start|>user

    COMPANY:
    IBM

    TARGET ROLE:
    Software Engineer

    JOB DESCRIPTION:
    
    We are hiring a Software Engineer.

    Required or preferred skills:
    Python, Django, PostgreSQL, REST APIs, TypeScript, Kubernetes

    The role involves software development, problem solving,
    testing and collaboration.
    

    EXISTING RESUME:
    
    SUMMARY:
    Computer Science graduate interested in software engine

In [ ]:
def tokenize_dataset(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=1024,
        padding=False
    )

    tokens["labels"] = tokens["input_ids"].copy()

    return tokens


train_tokenized = train_text.map(
    tokenize_dataset,
    remove_columns=train_text.column_names
)

validation_tokenized = validation_text.map(
    tokenize_dataset,
    remove_columns=validation_text.column_names
)

test_tokenized = test_text.map(
    tokenize_dataset,
    remove_columns=test_text.column_names
)

print("Train columns:", train_tokenized.column_names)
print("Number of training examples:", len(train_tokenized))

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Train columns: ['input_ids', 'attention_mask', 'labels']
Number of training examples: 240


In [ ]:
# Check token lengths
lengths = [len(x["input_ids"]) for x in train_tokenized]

print("Maximum tokens:", max(lengths))
print("Average tokens:", sum(lengths) / len(lengths))
print("Minimum tokens:", min(lengths))

print("\nSample tokenized example:")
print(train_tokenized[0].keys())
print("Input tokens:", len(train_tokenized[0]["input_ids"]))
print("Label tokens:", len(train_tokenized[0]["labels"]))

Maximum tokens: 489
Average tokens: 451.5541666666667
Minimum tokens: 410

Sample tokenized example:
dict_keys(['input_ids', 'attention_mask', 'labels'])
Input tokens: 441
Label tokens: 441


In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("Data collator created successfully!")

Data collator created successfully!


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-resume-lora-final",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    remove_unused_columns=False,

    report_to="none"
)

print("Training arguments ready!")

Training arguments ready!


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=train_tokenized,
    eval_dataset=validation_tokenized,

    data_collator=data_collator
)

print("Trainer created successfully!")

Trainer created successfully!


In [ ]:
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
print("GPU:", torch.cuda.get_device_name(0))

memory = torch.cuda.get_device_properties(0).total_memory / (1024**3)
print(f"GPU memory: {memory:.2f} GB")

print("Allocated:",
      round(torch.cuda.memory_allocated() / (1024**3), 2), "GB")

print("Reserved:",
      round(torch.cuda.memory_reserved() / (1024**3), 2), "GB")

GPU: Tesla T4
GPU memory: 14.56 GB
Allocated: 11.91 GB
Reserved: 12.79 GB


In [ ]:
print("Starting QLoRA training...")

trainer.train()

print("Training completed!")

Starting QLoRA training...


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
50,0.051985,0.053320
100,0.041155,0.041455
120,0.039128,0.040617


/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.13/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Training completed!


In [ ]:
print("Ready to start training!")
model.print_trainable_parameters()

Ready to start training!
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


In [ ]:
adapter_path = "./qwen-resume-lora-final"

model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print("LoRA adapter saved successfully!")
print("Saved to:", adapter_path)

LoRA adapter saved successfully!
Saved to: ./qwen-resume-lora-final


In [ ]:
import os

print(os.listdir(adapter_path))

['checkpoint-100', 'README.md', 'adapter_config.json', 'checkpoint-120', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json']


In [ ]:
from peft import PeftModel

# Reload the base Qwen3-4B model in 4-bit
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load your trained LoRA adapter
test_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

test_model.eval()

print("Trained LoRA adapter loaded successfully!")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Trained LoRA adapter loaded successfully!


In [ ]:
print("Base model:", model_name)
print("Adapter:", adapter_path)
print("Model ready for testing!")

Base model: Qwen/Qwen3-4B
Adapter: ./qwen-resume-lora-final
Model ready for testing!


In [ ]:
resume = """
John Doe
Computer Science Graduate

Skills:
Python, C++, SQL, Git

Projects:
E-Commerce Backend
- Built REST APIs using Python.
- Used SQL for storing application data.
- Implemented authentication and error handling.
"""

company = "Amazon"
role = "Software Development Engineer"

job_description = """
We are looking for a Software Development Engineer.

Required skills:
- Programming in C++ or Java
- Data structures and algorithms
- Object-oriented programming
- SQL
- Git
- Problem solving
- Communication and collaboration
"""

prompt = f"""
Optimize the following existing resume for the target job.

Company: {company}
Role: {role}

Job Description:
{job_description}

Existing Resume:
{resume}

Rules:
1. Improve the resume for the target role.
2. Do not invent experience, projects, certifications, employers, metrics, or skills.
3. A skill may be included if it already exists in the resume OR is clearly demonstrated by a project or experience.
4. Do not add a skill only because it appears in the job description.
5. Keep the writing natural and human.
6. Avoid generic AI-sounding language.
7. Preserve the candidate's actual background.
8. Return an optimized resume.
"""

messages = [
    {
        "role": "system",
        "content": "You are an expert resume optimizer. Improve resumes accurately without fabricating information."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(test_model.device)

print("Input prepared successfully!")

Input prepared successfully!


In [ ]:
with torch.no_grad():
    outputs = test_model.generate(
        **inputs,
        max_new_tokens=700,
        do_sample=True,
        temperature=0.7,
        top_p=0.9
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

result = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(result)

    John Doe
    Computer Science Graduate

    Skills:
    C++, SQL, Git, Python

    Projects:
    E-Commerce Backend
    - Built REST APIs using Python.
    - Used SQL for storing application data.
    - Implemented authentication and error handling.

    Data Structures and Algorithms:
    - Worked with data structures during project development.
    - Collaborated with other students during development.
    - Presented the project and explained the implementation.

    Problem Solving:
    - Debugged issues during development.
    - Worked through implementation challenges during development.
    - Documented the setup and usage of the project.

    Communication and Collaboration:
    - Worked with other students during development.
    - Presented the project and explained the implementation.
    - Documented the setup and usage of the project.

    Git:
    - Worked with Git during development.
    - Collaborated with other students during development.
    - Worked through impl

In [ ]:
test_cases = [
    {
        "company": "Microsoft",
        "role": "Backend Software Engineer",
        "resume": """
Jane Smith
Computer Science Graduate

Skills:
Python, Java, SQL, Git

Project:
Inventory Management System
- Developed backend APIs using Python.
- Designed SQL tables for inventory data.
- Added user authentication.
""",
        "jd": """
Looking for a Backend Software Engineer with experience in Python,
Java, REST APIs, databases, Git, distributed systems, and problem solving.
"""
    },
    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "resume": """
Alex Kumar
Computer Science Graduate

Skills:
Python, NumPy, Pandas, SQL

Project:
Student Performance Analysis
- Cleaned datasets using Python.
- Used Pandas and NumPy for analysis.
- Built data visualizations to understand performance trends.
""",
        "jd": """
Seeking a Machine Learning Engineer with Python, machine learning,
deep learning, PyTorch, data preprocessing, and problem solving skills.
"""
    },
    {
        "company": "Razorpay",
        "role": "Software Engineer",
        "resume": """
Rahul
Computer Science Graduate

Skills:
C++, Python, SQL, Git

Project:
Online Payment Prototype
- Created a prototype payment workflow.
- Used C++ for core programming logic.
- Stored transaction information using SQL.
""",
        "jd": """
Software Engineer role requiring C++, data structures and algorithms,
SQL, Git, object-oriented programming, and collaboration.
"""
    }
]

print("Test cases:", len(test_cases))

Test cases: 3


In [ ]:
def generate_resume(case):
    prompt = f"""
Optimize the following existing resume for the target job.

Company: {case["company"]}
Role: {case["role"]}

Job Description:
{case["jd"]}

Existing Resume:
{case["resume"]}

Rules:
1. Improve the resume for the target role.
2. Never invent experience, projects, certifications, employers, metrics, or skills.
3. A skill may be included if it exists in the resume OR is clearly demonstrated
   by a project or experience.
4. Do not add a skill only because it appears in the JD.
5. Keep the writing natural and human.
6. Avoid keyword stuffing.
7. Use role-relevant soft skills only when supported by the resume.
8. Preserve the candidate's actual background.
9. Return only the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert resume optimizer. Optimize resumes accurately without fabricating information."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(test_model.device)

    with torch.no_grad():
        outputs = test_model.generate(
            **inputs,
            max_new_tokens=700,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
results = []

for i, case in enumerate(test_cases, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}: {case['company']} - {case['role']}")
    print("="*60)

    result = generate_resume(case)
    results.append(result)

    print(result)


TEST 1: Microsoft - Backend Software Engineer
    Jane Smith
    Computer Science Graduate

    Skills:
    Python, Java, SQL, Git

    Project:
    Inventory Management System
    - Developed backend APIs using Python.
    - Designed SQL tables for inventory data.
    - Added user authentication.

    - Worked with SQL during development.
    - Collaborated with other students during development.
    
    - Presented the project and explained the implementation.
    

TEST 2: NVIDIA - Machine Learning Engineer
    Alex Kumar
    Computer Science Graduate

    Skills:
    Python, Pandas, NumPy, SQL

    Project:
    Student Performance Analysis
    - Cleaned datasets using Python.
    - Used Pandas and NumPy for analysis.
    - Built data visualizations to understand performance trends.
    
    - Worked with real-world datasets during the project.
    - Documented the analysis process and implementation steps.
    
    - Collaborated with other students during development.
    - Pres

In [ ]:
def evaluate_output(case, output):
    output_lower = output.lower()
    resume_lower = case["resume"].lower()
    jd_lower = case["jd"].lower()

    # Skills explicitly present in original resume
    resume_skills = [
        skill for skill in [
            "python", "c++", "java", "sql", "git",
            "numpy", "pandas", "rest", "pytorch"
        ]
        if skill in resume_lower
    ]

    # Skills appearing in output
    preserved_skills = [
        skill for skill in resume_skills
        if skill in output_lower
    ]

    return {
        "original_skills_found": resume_skills,
        "preserved_in_output": preserved_skills,
        "preservation_rate":
            len(preserved_skills) / len(resume_skills)
            if resume_skills else 0
    }


for case, result in zip(test_cases, results):
    evaluation = evaluate_output(case, result)
    print("\n", case["company"], "-", case["role"])
    print(evaluation)


 Microsoft - Backend Software Engineer
{'original_skills_found': ['python', 'java', 'sql', 'git'], 'preserved_in_output': ['python', 'java', 'sql', 'git'], 'preservation_rate': 1.0}

 NVIDIA - Machine Learning Engineer
{'original_skills_found': ['python', 'sql', 'numpy', 'pandas'], 'preserved_in_output': ['python', 'sql', 'numpy', 'pandas'], 'preservation_rate': 1.0}

 Razorpay - Software Engineer
{'original_skills_found': ['python', 'c++', 'sql', 'git'], 'preserved_in_output': ['python', 'c++', 'sql', 'git'], 'preservation_rate': 1.0}


In [ ]:
hallucination_case = {
    "company": "Google",
    "role": "Software Engineer",
    "resume": """
Candidate Name

Skills:
Python, C++, SQL

Project:
Student Management System
- Built a console-based application using C++.
- Used SQL to store and retrieve student records.
- Implemented search and update functionality.
""",
    "jd": """
Software Engineer

Requirements:
- C++
- Python
- Java
- Go
- Kubernetes
- Docker
- AWS
- Distributed systems
- Machine learning
- Data structures and algorithms
"""
}

hallucination_output = generate_resume(hallucination_case)

print(hallucination_output)

    Candidate Name

    SKILLS:
    C++, Python, SQL

    PROJECTS:
    Student Management System
    - Built a console-based application using C++.
    - Used SQL to store and retrieve student records.
    - Implemented search and update functionality.

    - Collaborated with other students during development.
    - Documented the setup and usage of the application.

    - Worked through implementation issues during development.
    - Debugged issues identified during testing.

    - Reviewed the application and fixed issues during testing.
    - Documented the setup and usage of the application.

    - Worked through implementation issues during development.
    - Debugged issues identified during testing.

    - Reviewed the application and fixed issues during testing.
    - Documented the setup and usage of the application.

    - Worked through implementation issues during development.
    - Debugged issues identified during testing.

    - Reviewed the application and fixed issu

In [ ]:
project_case = {
    "company": "Microsoft",
    "role": "Backend Software Engineer",
    "resume": """
Candidate Name

Skills:
Python, SQL, Git

Project:
E-Commerce Backend
- Developed HTTP endpoints for user registration and product management.
- Implemented authentication for users.
- Stored application data in a relational database.
- Used Git for version control.
""",
    "jd": """
Backend Software Engineer

Requirements:
- Python
- REST APIs
- SQL
- Git
- Authentication
- Backend development
- Docker
"""
}

project_output = generate_resume(project_case)

print(project_output)

    Candidate Name

    SKILLS:
    Python, SQL, Git, REST APIs, Authentication

    PROJECTS:
    E-Commerce Backend
    - Developed HTTP endpoints for user registration and product management.
    - Implemented authentication for users.
    - Stored application data in a relational database.
    - Used Git for version control.

    - Built a project using Python.
    - Worked with SQL during development.
    - Collaborated using Git during development.

    - Worked through implementation issues during development.
    - Debugged issues identified during testing.
</think>



In [ ]:
soft_skill_case = {
    "company": "Amazon",
    "role": "Software Engineer",
    "resume": """
Candidate Name

Skills:
C++, Python, SQL

Project:
Team-Based Web Application
- Worked with three classmates to develop a web application.
- Divided development tasks among team members.
- Integrated individual components into one application.
- Presented the completed project to faculty members.
""",
    "jd": """
Software Development Engineer

Requirements:
- C++
- Data structures and algorithms
- Problem solving
- Communication
- Collaboration
- Ownership
"""
}

soft_output = generate_resume(soft_skill_case)

print(soft_output)

    Candidate Name

    SKILLS
    C++, Python, SQL, Data Structures, Algorithms

    PROJECTS
    Team-Based Web Application
    - Worked with three classmates to develop a web application.
    - Divided development tasks among team members.
    - Integrated individual components into one application.
    - Presented the completed project to faculty members.

    SOFT SKILLS
    Communication, Collaboration

    TASKS
    - Worked with other students during development.
    - Worked through implementation issues during development.
    - Documented the development process and presented the project.

    ACHIEVEMENTS
    - Built a functional web application during development.
- Worked through implementation issues during development.
- Documented the development process and presented the project.

    - Collaborated with other students during development.
- Worked with three classmates during development.
- Worked through implementation issues during development.
- Documented the deve

In [ ]:
# Load a fresh original Qwen3-4B model
original_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

original_model.eval()

print("Original Qwen3-4B loaded!")

NameError: name 'AutoModelForCausalLM' is not defined

In [ ]:
def generate_with_model(model_to_test, case):
    prompt = f"""
Optimize the following existing resume for the target job.

Company: {case["company"]}
Role: {case["role"]}

Job Description:
{case["jd"]}

Existing Resume:
{case["resume"]}

Rules:
1. Improve the resume for the target role.
2. Never invent experience, projects, certifications, employers, metrics, or skills.
3. A skill may be included if it exists in the resume OR is clearly demonstrated
   by a project or experience.
4. Do not add a skill only because it appears in the JD.
5. Keep the writing natural and human.
6. Avoid keyword stuffing.
7. Use evidence-based soft skills.
8. Return only the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert resume optimizer. Never fabricate candidate information."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model_to_test.device)

    with torch.no_grad():
        outputs = model_to_test.generate(
            **inputs,
            max_new_tokens=600,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
case = hallucination_case

print("=" * 70)
print("ORIGINAL QWEN3-4B")
print("=" * 70)

original_result = generate_with_model(
    original_model,
    case
)

print(original_result)


print("\n" + "=" * 70)
print("FINE-TUNED QWEN3-4B + LoRA")
print("=" * 70)

finetuned_result = generate_with_model(
    test_model,
    case
)

print(finetuned_result)

NameError: name 'hallucination_case' is not defined

In [ ]:
comparison_results = []

for i, case in enumerate(test_cases, 1):
    print(f"Running test {i}/{len(test_cases)}...")

    original = generate_with_model(original_model, case)
    finetuned = generate_with_model(test_model, case)

    comparison_results.append({
        "company": case["company"],
        "role": case["role"],
        "original_model": original,
        "finetuned_model": finetuned
    })

print("\nAll comparisons completed!")

In [ ]:
import json

with open("model_comparison_results.json", "w", encoding="utf-8") as f:
    json.dump(
        comparison_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved: model_comparison_results.json")

In [ ]:
for i, item in enumerate(comparison_results, 1):
    print("\n" + "=" * 80)
    print(f"TEST {i}: {item['company']} — {item['role']}")
    print("=" * 80)

    print("\n--- ORIGINAL QWEN ---")
    print(item["original_model"])

    print("\n--- FINE-TUNED QWEN ---")
    print(item["finetuned_model"])

In [ ]:
from peft import PeftModel

merged_model = test_model.merge_and_unload()

print("LoRA merged into base model!")

In [ ]:
SAVE_PATH = "./ResumeOptimizer-Qwen3-4B"

merged_model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print("Final model saved to:", SAVE_PATH)

In [ ]:
import shutil

shutil.make_archive(
    "ResumeOptimizer-Qwen3-4B",
    "zip",
    SAVE_PATH
)

print("ZIP created!")

In [ ]:
from google.colab import files

files.download("ResumeOptimizer-Qwen3-4B.zip")

In [ ]:
resume = """<PASTE USER'S EXISTING RESUME HERE>"""

jd = """<PASTE AMAZON SOFTWARE ENGINEER JD HERE>"""

company = "Amazon"
role = "Software Development Engineer"

prompt = f"""
You are ResumeOptimizer.

Optimize the existing resume for {company} - {role}.

Rules:
- Preserve all truthful information.
- Prioritize skills matching the JD.
- Improve summary.
- Improve projects.
- Improve skills.
- Humanize writing.
- Never invent experience.
- Never invent certifications.
- Never invent companies.
- Never invent metrics.
- ATS optimized.
- Return only the resume.

JOB DESCRIPTION:
{jd}

RESUME:
{resume}
"""

In [ ]:
messages = [
    {"role":"system","content":"You are ResumeOptimizer."},
    {"role":"user","content":prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(text, return_tensors="pt").to(merged_model.device)

outputs = merged_model.generate(
    **inputs,
    max_new_tokens=900,
    temperature=0.6,
    top_p=0.9
)

optimized_resume = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(optimized_resume)

In [ ]:
with open("optimized_resume.txt","w") as f:
    f.write(optimized_resume)

print("Saved optimized_resume.txt")

In [ ]:
!pip install reportlab

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

doc = SimpleDocTemplate("optimized_resume.pdf")
styles = getSampleStyleSheet()

story = []

for line in optimized_resume.split("\n"):
    if line.strip():
        story.append(
            Paragraph(line.replace(" ", "&nbsp;"), styles["BodyText"])
        )

doc.build(story)

print("PDF generated successfully!")

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph
from reportlab.lib.styles import getSampleStyleSheet

doc = SimpleDocTemplate("optimized_resume.pdf")
styles = getSampleStyleSheet()

story = []

for line in optimized_resume.split("\n"):
    story.append(Paragraph(line.replace(" ","&nbsp;"), styles["BodyText"]))

doc.build(story)

print("PDF generated!")

In [ ]:
files.download("optimized_resume.pdf")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_PATH = "./ResumeOptimizer-Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto"
)

In [ ]:
def optimize_resume(resume, jd, company, role):
    prompt = f"""
    Company: {company}
    Role: {role}

    JD:
    {jd}

    Resume:
    {resume}

    Optimize this resume.
    """

    messages = [
        {"role":"system","content":"You are ResumeOptimizer."},
        {"role":"user","content":prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=900,
        temperature=0.6
    )

    return tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

In [ ]:
optimized = optimize_resume(resume, jd, "Amazon", "Software Engineer")

In [ ]:
def check_resume_output(case, output):
    output_lower = output.lower()
    resume_lower = case["resume"].lower()
    jd_lower = case["jd"].lower()

    # Skills that are explicitly present in the original resume
    known_skills = [
        "python", "c++", "java", "sql", "git",
        "numpy", "pandas", "pytorch", "docker",
        "aws", "kubernetes"
    ]

    original_skills = [
        skill for skill in known_skills
        if skill in resume_lower
    ]

    jd_skills = [
        skill for skill in known_skills
        if skill in jd_lower
    ]

    output_skills = [
        skill for skill in known_skills
        if skill in output_lower
    ]

    # JD skills appearing in output but not supported by original resume
    suspicious_skills = [
        skill for skill in output_skills
        if skill in jd_skills and skill not in original_skills
    ]

    return {
        "original_skills": original_skills,
        "output_skills": output_skills,
        "suspicious_jd_skills": suspicious_skills
    }

In [ ]:
for i, case in enumerate(test_cases, 1):
    result = check_resume_output(
        case,
        comparison_results[i-1]["finetuned_model"]
    )

    print("\n" + "=" * 60)
    print(f"TEST {i}: {case['company']} - {case['role']}")
    print(result)

NameError: name 'test_cases' is not defined

In [ ]:
evaluation_summary = []

for i, case in enumerate(test_cases):
    output = comparison_results[i]["finetuned_model"]
    check = check_resume_output(case, output)

    evaluation_summary.append({
        "test": i + 1,
        "company": case["company"],
        "role": case["role"],
        "original_skills": check["original_skills"],
        "output_skills": check["output_skills"],
        "suspicious_skills": check["suspicious_jd_skills"],
        "hallucination_risk": (
            "HIGH" if len(check["suspicious_jd_skills"]) >= 3
            else "MEDIUM" if len(check["suspicious_jd_skills"]) > 0
            else "LOW"
        )
    })

for item in evaluation_summary:
    print(
        f"{item['test']}. "
        f"{item['company']} | {item['role']} | "
        f"Risk: {item['hallucination_risk']}"
    )

In [ ]:
with open("evaluation_summary.json", "w", encoding="utf-8") as f:
    json.dump(
        evaluation_summary,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Evaluation saved successfully!")

In [ ]:
def quality_check(case, output):
    output_lower = output.lower()
    resume_lower = case["resume"].lower()
    jd_lower = case["jd"].lower()

    # Common AI-sounding phrases
    ai_phrases = [
        "leveraged",
        "spearheaded",
        "proven track record",
        "results-driven",
        "highly motivated",
        "dynamic professional",
        "passionate professional",
        "cutting-edge",
        "robust solution",
        "seamlessly"
    ]

    found_ai_phrases = [
        phrase for phrase in ai_phrases
        if phrase in output_lower
    ]

    # Original resume facts that should remain visible
    important_terms = [
        "python", "c++", "java", "sql", "git",
        "numpy", "pandas", "pytorch",
        "authentication", "backend", "api"
    ]

    preserved = [
        term for term in important_terms
        if term in resume_lower and term in output_lower
    ]

    original_terms = [
        term for term in important_terms
        if term in resume_lower
    ]

    preservation_score = (
        len(preserved) / len(original_terms) * 100
        if original_terms else 100
    )

    # JD relevance
    jd_terms = [
        term for term in important_terms
        if term in jd_lower
    ]

    relevant_terms = [
        term for term in jd_terms
        if term in output_lower
    ]

    relevance_score = (
        len(relevant_terms) / len(jd_terms) * 100
        if jd_terms else 0
    )

    return {
        "preservation_score": round(preservation_score, 2),
        "jd_relevance_score": round(relevance_score, 2),
        "ai_phrases_found": found_ai_phrases,
        "preserved_terms": preserved,
        "jd_terms_used": relevant_terms
    }

In [ ]:
quality_results = []

for i, case in enumerate(test_cases):
    output = comparison_results[i]["finetuned_model"]

    result = quality_check(case, output)

    quality_results.append({
        "test": i + 1,
        "company": case["company"],
        "role": case["role"],
        **result
    })

    print("\n" + "=" * 70)
    print(f"TEST {i + 1}: {case['company']} — {case['role']}")
    print("=" * 70)

    print("Preservation:", result["preservation_score"], "%")
    print("JD relevance:", result["jd_relevance_score"], "%")
    print("AI phrases:", result["ai_phrases_found"])
    print("Preserved:", result["preserved_terms"])
    print("JD terms used:", result["jd_terms_used"])

In [ ]:
with open("quality_evaluation.json", "w", encoding="utf-8") as f:
    json.dump(
        quality_results,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Quality evaluation saved!")

In [ ]:
import statistics

preservation_scores = [
    x["preservation_score"]
    for x in quality_results
]

relevance_scores = [
    x["jd_relevance_score"]
    for x in quality_results
]

ai_phrase_counts = [
    len(x["ai_phrases_found"])
    for x in quality_results
]

avg_preservation = statistics.mean(preservation_scores)
avg_relevance = statistics.mean(relevance_scores)
avg_ai_phrases = statistics.mean(ai_phrase_counts)

print("=" * 60)
print("FINAL MODEL EVALUATION")
print("=" * 60)

print(f"Average skill preservation : {avg_preservation:.2f}%")
print(f"Average JD relevance       : {avg_relevance:.2f}%")
print(f"Average AI phrases         : {avg_ai_phrases:.2f}")

print("\nRecommendation:")

if avg_preservation >= 90 and avg_relevance >= 70 and avg_ai_phrases <= 1:
    print("✅ Model looks promising for further testing.")
elif avg_preservation >= 75 and avg_relevance >= 50:
    print("⚠️ Model needs dataset/model improvements.")
else:
    print("❌ Current fine-tuning is not good enough yet.")

In [ ]:
for x in quality_results:
    print(
        f"{x['test']}. "
        f"{x['company']} | {x['role']} | "
        f"Preservation: {x['preservation_score']}% | "
        f"Relevance: {x['jd_relevance_score']}% | "
        f"AI phrases: {len(x['ai_phrases_found'])}"
    )

In [ ]:
import shutil
import os

files_to_save = [
    "evaluation_summary.json",
    "model_comparison_results.json",
    "quality_evaluation.json"
]

# Create a backup folder
backup_dir = "./resume_optimizer_backup"
os.makedirs(backup_dir, exist_ok=True)

# Copy evaluation files
for file in files_to_save:
    if os.path.exists(file):
        shutil.copy(file, backup_dir)

# Copy the trained LoRA adapter
if os.path.exists(adapter_path):
    shutil.copytree(
        adapter_path,
        os.path.join(backup_dir, "qwen-resume-lora-final"),
        dirs_exist_ok=True
    )

# Zip everything
shutil.make_archive(
    "resume_optimizer_backup",
    "zip",
    backup_dir
)

print("Backup created successfully!")
print("File: resume_optimizer_backup.zip")

In [ ]:
from google.colab import files

files.download("resume_optimizer_backup.zip")

In [ ]:
print("Average skill preservation:", avg_preservation)
print("Average JD relevance:", avg_relevance)
print("Average AI phrases:", avg_ai_phrases)

print("\nTest cases evaluated:", len(test_cases))

In [ ]:
FINAL_ADAPTER = "./ResumeOptimizer-LoRA"

trainer.save_model(FINAL_ADAPTER)
tokenizer.save_pretrained(FINAL_ADAPTER)

print("LoRA adapter saved successfully!")

In [ ]:
import os

for root, dirs, files_list in os.walk(FINAL_ADAPTER):
    level = root.replace(FINAL_ADAPTER, "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")

    for file in files_list:
        print(indent + "  " + file)

In [ ]:
from peft import PeftModel

# Load the base model for inference
inference_base = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Load your final LoRA adapter
inference_model = PeftModel.from_pretrained(
    inference_base,
    FINAL_ADAPTER
)

inference_model.eval()

print("Final Resume Optimizer loaded!")

In [ ]:
print("Model ready for final inference testing.")
print("Adapter:", FINAL_ADAPTER)

In [ ]:
resume = """
PASTE YOUR EXISTING RESUME TEXT HERE
"""

company = "Amazon"
role = "Software Development Engineer"

jd = """
PASTE THE JOB DESCRIPTION HERE
"""

In [ ]:
prompt = f"""
Optimize the existing resume for the following job.

Company: {company}
Role: {role}

JOB DESCRIPTION:
{jd}

EXISTING RESUME:
{resume}

Rules:
- Preserve all truthful information from the original resume.
- Improve the summary, technical skills, soft skills, and experience/projects.
- A skill can be surfaced if it is already in the resume OR clearly demonstrated
  through a project or experience.
- Do NOT add a skill merely because it appears in the JD.
- Do NOT invent experience, projects, certifications, employers, metrics, or technologies.
- Keep soft skills evidence-based and role-specific.
- Use natural, human wording.
- Avoid keyword stuffing and generic AI language.
- Optimize for ATS relevance while keeping the resume readable.
- Return only the optimized resume.
"""

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an expert resume optimizer. Never fabricate candidate information."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(inference_model.device)

with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=1000,
        do_sample=False
    )

generated = outputs[0][inputs["input_ids"].shape[1]:]

optimized_resume = tokenizer.decode(
    generated,
    skip_special_tokens=True
)

print(optimized_resume)

In [ ]:
with open("optimized_resume.txt", "w", encoding="utf-8") as f:
    f.write(optimized_resume)

print("Optimized resume saved successfully!")

In [ ]:
import os

print("File exists:", os.path.exists("optimized_resume.txt"))
print("File size:", os.path.getsize("optimized_resume.txt"), "bytes")

In [ ]:
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

pdf_path = "optimized_resume.pdf"

doc = SimpleDocTemplate(pdf_path)
styles = getSampleStyleSheet()

story = []

for line in optimized_resume.split("\n"):
    line = line.strip()

    if line:
        story.append(Paragraph(line, styles["BodyText"]))
        story.append(Spacer(1, 6))

doc.build(story)

print("PDF generated successfully!")
print("Path:", pdf_path)

In [ ]:
import os

print("PDF exists:", os.path.exists(pdf_path))
print("PDF size:", os.path.getsize(pdf_path), "bytes")

In [ ]:
from google.colab import files

files.download("optimized_resume.pdf")

FileNotFoundError: Cannot find file: optimized_resume.pdf

In [ ]:
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import LETTER
from reportlab.lib.units import inch
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
import re

pdf_path = "optimized_resume_professional.pdf"

doc = SimpleDocTemplate(
    pdf_path,
    pagesize=LETTER,
    rightMargin=0.55 * inch,
    leftMargin=0.55 * inch,
    topMargin=0.45 * inch,
    bottomMargin=0.45 * inch
)

styles = getSampleStyleSheet()

name_style = ParagraphStyle(
    "Name",
    parent=styles["Heading1"],
    fontSize=16,
    leading=18,
    alignment=TA_CENTER,
    spaceAfter=6
)

heading_style = ParagraphStyle(
    "Section",
    parent=styles["Heading2"],
    fontSize=10.5,
    leading=12,
    spaceBefore=7,
    spaceAfter=4
)

body_style = ParagraphStyle(
    "Body",
    parent=styles["BodyText"],
    fontSize=9,
    leading=11,
    spaceAfter=3
)

bullet_style = ParagraphStyle(
    "Bullet",
    parent=body_style,
    leftIndent=12,
    firstLineIndent=-7,
    bulletIndent=0
)

story = []

lines = optimized_resume.splitlines()

for line in lines:
    line = line.strip()

    if not line:
        story.append(Spacer(1, 3))
        continue

    # Markdown headings
    if line.startswith("#"):
        heading = line.lstrip("#").strip()
        story.append(Paragraph(heading, heading_style))

    # Bullet points
    elif line.startswith(("-", "*", "•")):
        bullet = line[1:].strip()
        story.append(
            Paragraph(
                "• " + bullet,
                bullet_style
            )
        )

    # Likely section headings
    elif (
        line.upper() in [
            "SUMMARY",
            "PROFESSIONAL SUMMARY",
            "TECHNICAL SKILLS",
            "SKILLS",
            "SOFT SKILLS",
            "EXPERIENCE",
            "WORK EXPERIENCE",
            "PROJECTS",
            "EDUCATION",
            "CERTIFICATIONS"
        ]
    ):
        story.append(Paragraph(line, heading_style))

    else:
        story.append(Paragraph(line, body_style))

doc.build(story)

print("Professional PDF created!")
print("File:", pdf_path)

In [ ]:
files.download("optimized_resume_professional.pdf")

In [ ]:
def optimize_resume(resume, jd, company, role):

    prompt = f"""
Optimize the existing resume for this job.

Company: {company}
Role: {role}

JOB DESCRIPTION:
{jd}

EXISTING RESUME:
{resume}

RULES:
1. Preserve truthful information.
2. Never invent experience, projects, certifications, employers, metrics, or technologies.
3. A skill can be surfaced if it is already present in the resume OR clearly demonstrated by a project/experience.
4. Do not add a skill only because it appears in the JD.
5. Improve the professional summary.
6. Improve technical skills based on truthful evidence.
7. Improve soft skills only when supported by the resume.
8. Rewrite project/experience bullets naturally.
9. Avoid keyword stuffing.
10. Keep the writing human and concise.
11. Return only the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert resume optimizer. Never fabricate candidate information."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

In [ ]:
result = optimize_resume(
    resume=resume,
    jd=jd,
    company="Amazon",
    role="Software Development Engineer"
)

print(result)

In [ ]:
import shutil
import os

shutil.make_archive(
    "ResumeOptimizer-LoRA",
    "zip",
    FINAL_ADAPTER
)

print("Created ResumeOptimizer-LoRA.zip")

In [ ]:
files.download("ResumeOptimizer-LoRA.zip")

In [ ]:
import torch

from fastapi import FastAPI
from pydantic import BaseModel

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import PeftModel


app = FastAPI(title="Resume Optimizer API")


BASE_MODEL = "Qwen/Qwen3-4B"
ADAPTER_PATH = "./models/ResumeOptimizer-LoRA"


# 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


print("Loading tokenizer...")

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)


print("Loading LoRA adapter...")

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

model.eval()

print("Resume Optimizer model loaded!")


class ResumeRequest(BaseModel):
    resume: str
    company: str
    role: str
    job_description: str


@app.get("/")
def home():
    return {
        "status": "online",
        "service": "Resume Optimizer"
    }


@app.post("/optimize")
def optimize(request: ResumeRequest):

    prompt = f"""
Optimize the existing resume for this job.

Company: {request.company}
Role: {request.role}

JOB DESCRIPTION:
{request.job_description}

EXISTING RESUME:
{request.resume}

RULES:
1. Preserve all truthful information.
2. Never invent experience, projects, certifications, employers,
   metrics, or technologies.
3. A skill may be surfaced if it already exists in the resume
   OR is clearly demonstrated by a project or experience.
4. Do not add a skill merely because it appears in the JD.
5. Improve the professional summary.
6. Improve technical skills based on truthful evidence.
7. Improve projects and experience naturally.
8. Use evidence-based soft skills.
9. Avoid keyword stuffing.
10. Keep the writing human and concise.
11. Return only the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert resume optimizer. "
                "Never fabricate candidate information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return {
        "optimized_resume": result
    }

In [ ]:
import os

print("Current directory:")
print(os.getcwd())

print("\nFiles/folders:")
print(os.listdir("."))

In [ ]:
import os

print("ResumeOptimizer-LoRA exists:",
      os.path.exists("./ResumeOptimizer-LoRA"))

print("models folder exists:",
      os.path.exists("./models"))

In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nAvailable files/folders:")

for item in os.listdir("."):
    print(item)

In [ ]:
import os

adapter_path = "./ResumeOptimizer-LoRA"

if os.path.exists(adapter_path):
    print("✅ Adapter folder found")
    print(os.listdir(adapter_path))
else:
    print("❌ Adapter folder not found")

In [ ]:
import os

print("Files in /content:")
for item in os.listdir("/content"):
    print(item)

In [ ]:

import zipfile
import os

zip_path = "/content/ResumeOptimizer-LoRA.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/")

print("✅ Adapter extracted")

print(os.listdir("/content/ResumeOptimizer-LoRA"))

In [ ]:
import os

print(os.listdir("/content"))

In [ ]:
import zipfile

zip_path = "/content/ResumeOptimizer-LoRA.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/")

print("✅ Extracted!")
print(os.listdir("/content/ResumeOptimizer-LoRA"))

In [ ]:
import os

path = "/content/ResumeOptimizer-LoRA"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Is folder:", os.path.isdir(path))
    print("Is file:", os.path.isfile(path))

    if os.path.isdir(path):
        print("\nContents:")
        print(os.listdir(path))
    else:
        print("\nFile size:", os.path.getsize(path), "bytes")

In [ ]:
import os

print("Files currently in /content:")
for name in os.listdir("/content"):
    print(repr(name))

In [ ]:
import zipfile
import os

zip_path = "/content/ResumeOptimizer-LoRA.zip"

print("File exists:", os.path.exists(zip_path))
print("File size:", os.path.getsize(zip_path), "bytes")
print("Valid ZIP:", zipfile.is_zipfile(zip_path))

In [ ]:
with zipfile.ZipFile(zip_path, "r") as z:
    print("ZIP contents:")
    for name in z.namelist():
        print(name)

In [ ]:
company = "Amazon"

role = "Software Development Engineer"

job_description = """
We are looking for a Software Development Engineer.

Requirements:
- C++ or Python
- Data Structures and Algorithms
- Object-Oriented Programming
- SQL
- Git
- Problem solving
- Communication and collaboration
"""

existing_resume = """
YOUR EXISTING RESUME HERE

Education:
B.Tech in Computer Science

Skills:
Python, C++, SQL, Git

Projects:
E-Commerce Application
- Built an e-commerce application using Python and SQL.
- Implemented user authentication and product search.
- Used Git for version control.
"""

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded successfully!")

In [ ]:
print("inference_model:", "inference_model" in globals())
print("model:", "model" in globals())

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ No GPU available")

In [ ]:
prompt = f"""
You are an expert resume optimizer.

Optimize the EXISTING RESUME for the target job.

COMPANY:
{company}

ROLE:
{role}

JOB DESCRIPTION:
{job_description}

EXISTING RESUME:
{existing_resume}

IMPORTANT RULES:

1. Preserve all truthful information from the existing resume.
2. Do NOT invent experience, projects, certifications, employers,
   technologies, achievements, or metrics.
3. A skill may be included if:
   - it already exists in the existing resume, OR
   - it is clearly demonstrated by a project or experience.
4. Do NOT add a skill simply because it appears in the JD.
5. Improve the professional summary for the target role.
6. Optimize the technical skills section using only truthful skills.
7. Improve project and experience bullet points.
8. Use evidence-based soft skills only.
9. Do not use the same generic soft skills for every role.
10. Keep the writing natural and human.
11. Avoid keyword stuffing.
12. Keep the candidate's actual background unchanged.
13. Return ONLY the optimized resume.

Now optimize the resume.
"""

messages = [
    {
        "role": "system",
        "content": "You are a professional resume optimizer. Never fabricate candidate information."
    },
    {
        "role": "user",
        "content": prompt
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False
)

inputs = tokenizer(
    text,
    return_tensors="pt"
).to(inference_model.device)

with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=1200,
        do_sample=False
    )

generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

optimized_resume = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(optimized_resume)

NameError: name 'company' is not defined

In [ ]:
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ No GPU available — stop here")

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel
import torch
import os

model_name = "Qwen/Qwen3-4B"
ADAPTER_PATH = "/content/ResumeOptimizer-LoRA"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Qwen3-4B...")

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loading trained LoRA adapter...")

inference_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

inference_model.eval()

print("✅ Inference model loaded!")
print("Adapter path:", ADAPTER_PATH)

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"
import bitsandbytes as bnb
import torch

print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen3-4B"
ADAPTER_PATH = "/content/ResumeOptimizer-LoRA"

# Check GPU
assert torch.cuda.is_available(), "❌ GPU is not available"

print("GPU:", torch.cuda.get_device_name(0))

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit QLoRA configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Base Qwen model
print("Loading Qwen3-4B...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Your trained adapter
print("Loading ResumeOptimizer LoRA...")

inference_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

inference_model.eval()

print("\n✅ Resume Optimizer loaded successfully!")

In [ ]:
!pip uninstall -y bitsandbytes
!pip install -U "bitsandbytes>=0.46.1"

In [ ]:
import bitsandbytes as bnb
import torch

print("bitsandbytes:", bnb.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

MODEL_NAME = "Qwen/Qwen3-4B"
ADAPTER_PATH = "/content/ResumeOptimizer-LoRA"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("Loading Qwen3-4B...")

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

print("Loading trained LoRA adapter...")

inference_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

inference_model.eval()

print("✅ Resume Optimizer loaded!")

In [ ]:
import os

print("ZIP:")
print(os.path.exists("/content/ResumeOptimizer-LoRA.zip"))

print("\nResumeOptimizer-LoRA folder:")
print(os.path.exists("/content/ResumeOptimizer-LoRA"))

if os.path.exists("/content/ResumeOptimizer-LoRA"):
    for root, dirs, files in os.walk("/content/ResumeOptimizer-LoRA"):
        print("\nFolder:", root)
        for f in files:
            print("  ", f)

In [ ]:
import zipfile
import os

zip_path = "/content/ResumeOptimizer-LoRA.zip"

print("ZIP exists:", os.path.exists(zip_path))
print("Valid ZIP:", zipfile.is_zipfile(zip_path))

if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path, "r") as z:
        print("\nZIP contents:")
        for name in z.namelist():
            print(name)

In [ ]:
import zipfile
import os

zip_path = "/content/ResumeOptimizer-LoRA.zip"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/ResumeOptimizer-LoRA")

print("✅ Adapter extracted")
print(os.listdir("/content/ResumeOptimizer-LoRA"))

In [ ]:
ADAPTER_PATH = "/content/ResumeOptimizer-LoRA"

print(os.listdir(ADAPTER_PATH))

In [ ]:
from peft import PeftModel

print("Loading trained LoRA adapter...")

inference_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

inference_model.eval()

print("✅ Trained Resume Optimizer loaded successfully!")

In [ ]:
print("Model:", type(inference_model).__name__)
print("Adapter loaded successfully!")

In [ ]:
def optimize_resume(company, role, job_description, existing_resume):

    prompt = f"""
Company: {company}
Target Role: {role}

Job Description:
{job_description}

Existing Resume:
{existing_resume}

Task:
Optimize the existing resume specifically for this company and role.

Rules:
1. Preserve all real information from the existing resume.
2. Do NOT invent experience, projects, certifications, employers, technologies, metrics, or achievements.
3. Improve the professional summary to match the target role naturally.
4. Reorganize and improve technical skills based on the existing resume and demonstrated project/experience skills.
5. A skill may be included if it is explicitly present in the resume OR clearly demonstrated through a project or experience.
6. Do NOT add a skill only because it appears in the job description.
7. Improve project and experience descriptions to emphasize the most relevant existing evidence.
8. Use natural human-written language.
9. Avoid keyword stuffing and generic AI-sounding phrases.
10. Keep soft skills evidence-based and relevant to the candidate.
11. Do not change facts.
12. Return only the optimized resume.

Optimized Resume:
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert resume optimizer. Never fabricate candidate information."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            repetition_penalty=1.05
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return result.strip()

print("✅ Resume optimizer function ready")

In [ ]:
company = "Microsoft"

role = "Software Engineer"

job_description = """
We are looking for a Software Engineer to design, develop, test,
and maintain scalable software systems. The role involves working
with backend services, APIs, databases, cloud technologies, and
cross-functional teams. Strong programming and problem-solving
skills are required.
"""

existing_resume = """
SAHITHA REDDY

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects
Resume Optimizer
Built a web application that analyzes resumes and improves them
based on job descriptions.

Student Management System
Developed a backend application with CRUD operations and SQL database
integration.

Experience
Software Development Intern
Worked on application development and debugging.
"""

optimized = optimize_resume(
    company,
    role,
    job_description,
    existing_resume
)

print(optimized)

In [ ]:
print("=" * 80)
print("OPTIMIZED RESUME")
print("=" * 80)
print(optimized)
print("=" * 80)

In [ ]:
hallucination_jd = """
Google is looking for a Software Engineer.

Required skills:
Java, Go, Kubernetes, Docker, AWS, distributed systems,
microservices, and machine learning.
"""

hallucination_resume = """
SAHITHA REDDY

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects
Resume Optimizer
Built a web application for improving resumes using job descriptions.

Experience
Software Development Intern
Worked on application development and debugging.
"""

hallucination_test = optimize_resume(
    "Google",
    "Software Engineer",
    hallucination_jd,
    hallucination_resume
)

print(hallucination_test)

In [ ]:
test_cases = [
    {
        "company": "Microsoft",
        "role": "Backend Software Engineer",
        "jd": """
        We are looking for a Backend Software Engineer to build reliable
        backend services and APIs. Experience with databases, authentication,
        debugging, Git, and scalable software development is valuable.
        """,
        "resume": """
        SAHITHA REDDY

        Education
        B.Tech in Computer Science Engineering

        Technical Skills
        Python, C++, SQL, Git

        Projects
        Student Management System
        Developed a backend application with CRUD operations and SQL database
        integration.

        Resume Optimizer
        Built a web application that analyzes resumes and improves them
        based on job descriptions.

        Experience
        Software Development Intern
        Worked on application development and debugging.
        """
    },

    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "jd": """
        We are looking for a Machine Learning Engineer to develop and deploy
        machine learning solutions. Strong programming, data processing,
        model development, experimentation, and problem-solving skills are
        important.
        """,
        "resume": """
        SAHITHA REDDY

        Education
        B.Tech in Computer Science Engineering

        Technical Skills
        Python, C++, SQL, Git

        Projects
        Resume Optimizer
        Built a web application that analyzes resumes and improves them
        based on job descriptions.

        Student Management System
        Developed a backend application with CRUD operations and SQL database
        integration.

        Experience
        Software Development Intern
        Worked on application development and debugging.
        """
    },

    {
        "company": "Razorpay",
        "role": "Software Engineer",
        "jd": """
        We are looking for a Software Engineer to build reliable software,
        develop APIs, work with databases, debug applications, and collaborate
        with engineering teams.
        """,
        "resume": """
        SAHITHA REDDY

        Education
        B.Tech in Computer Science Engineering

        Technical Skills
        Python, C++, SQL, Git

        Projects
        Student Management System
        Developed a backend application with CRUD operations and SQL database
        integration.

        Resume Optimizer
        Built a web application for improving resumes using job descriptions.

        Experience
        Software Development Intern
        Worked on application development and debugging.
        """
    }
]

print("✅ Test cases created:", len(test_cases))

In [ ]:
results = []

for i, case in enumerate(test_cases, 1):

    print(f"\n{'='*80}")
    print(f"TEST {i}: {case['company']} — {case['role']}")
    print(f"{'='*80}")

    result = optimize_resume(
        case["company"],
        case["role"],
        case["jd"],
        case["resume"]
    )

    results.append(result)

    print(result)

In [ ]:
project_skill_test = optimize_resume(
    company="Microsoft",
    role="Backend Software Engineer",
    job_description="""
    We are looking for a Backend Software Engineer experienced in
    REST APIs, authentication, backend development, databases,
    Docker, Kubernetes, and cloud infrastructure.
    """,
    existing_resume="""
    SAHITHA REDDY

    Technical Skills
    Python, C++, SQL, Git

    Projects
    Secure Task Management API
    Built a backend application with HTTP endpoints.
    Implemented user authentication and authorization.
    Connected the application to a SQL database.
    Used Git for source-code management.

    Experience
    Software Development Intern
    Worked on application development and debugging.
    """
)

print(project_skill_test)

In [ ]:
soft_skill_test = optimize_resume(
    company="Amazon",
    role="Software Development Engineer",
    job_description="""
    We are looking for a Software Development Engineer who can work
    effectively with engineering teams, communicate technical ideas,
    solve problems, and contribute to reliable software systems.
    """,
    existing_resume="""
    SAHITHA REDDY

    Technical Skills
    Python, C++, SQL, Git

    Projects
    College Event Management Platform
    Worked with a three-member team to build an event management
    application. Coordinated development tasks, discussed feature
    requirements with teammates, integrated different modules,
    and helped debug issues before the final demonstration.

    Experience
    Software Development Intern
    Worked on application development and debugging.
    """
)

print(soft_skill_test)

In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_compare_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-4B",
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

base_compare_model.eval()

print("✅ Base Qwen3-4B loaded")

In [ ]:
def generate_base_resume(company, role, job_description, existing_resume):

    prompt = f"""
Company: {company}
Target Role: {role}

Job Description:
{job_description}

Existing Resume:
{existing_resume}

Optimize the existing resume for the target role.

Rules:
- Do not invent experience or skills.
- Do not add skills only because they appear in the JD.
- You may surface skills demonstrated in projects or experience.
- Preserve factual information.
- Improve relevance and natural human-written wording.
- Do not keyword stuff.

Return only the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert resume optimizer. Never fabricate candidate information."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(base_compare_model.device)

    with torch.no_grad():
        outputs = base_compare_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            repetition_penalty=1.05
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

print("✅ Base model function ready")

In [ ]:
comparison_company = "Microsoft"

comparison_role = "Backend Software Engineer"

comparison_jd = """
We are hiring a Backend Software Engineer to develop reliable backend
services and APIs. The engineer will work with databases, authentication,
debugging, Git, and scalable software systems. Experience working with
engineering teams and communicating technical ideas is valuable.
"""

comparison_resume = """
SAHITHA REDDY

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects
Secure Task Management API
Built a backend application with HTTP endpoints.
Implemented user authentication and authorization.
Connected the application to a SQL database.
Used Git for source-code management.

Experience
Software Development Intern
Worked on application development and debugging.
"""

print("✅ Comparison case ready")

In [ ]:
base_output = generate_base_resume(
    comparison_company,
    comparison_role,
    comparison_jd,
    comparison_resume
)

print("=" * 80)
print("BASE QWEN3-4B")
print("=" * 80)
print(base_output)

NameError: name 'generate_base_resume' is not defined

In [ ]:
finetuned_output = optimize_resume(
    comparison_company,
    comparison_role,
    comparison_jd,
    comparison_resume
)

print("=" * 80)
print("FINE-TUNED QWEN3-4B + LoRA")
print("=" * 80)
print(finetuned_output)

In [ ]:
print("\n" + "=" * 100)
print("BASE MODEL")
print("=" * 100)
print(base_output)

print("\n" + "=" * 100)
print("FINE-TUNED MODEL")
print("=" * 100)
print(finetuned_output)

print("\n" + "=" * 100)
print("END OF COMPARISON")
print("=" * 100)

In [ ]:
import re

def evaluate_resume(original_resume, job_description, output):

    original_lower = original_resume.lower()
    jd_lower = job_description.lower()
    output_lower = output.lower()

    # Common technical skills to check
    skills = [
        "python", "c++", "java", "go", "sql", "git",
        "docker", "kubernetes", "aws", "azure", "gcp",
        "api", "rest", "authentication", "authorization",
        "machine learning", "tensorflow", "pytorch"
    ]

    original_skills = {
        skill for skill in skills if skill in original_lower
    }

    output_skills = {
        skill for skill in skills if skill in output_lower
    }

    jd_skills = {
        skill for skill in skills if skill in jd_lower
    }

    # Skills preserved from original resume
    preserved = original_skills.intersection(output_skills)

    # New skills appearing in output
    new_skills = output_skills - original_skills

    # New skills that are also in JD
    jd_only_candidates = new_skills.intersection(jd_skills)

    return {
        "original_skills": sorted(original_skills),
        "output_skills": sorted(output_skills),
        "preserved_skills": sorted(preserved),
        "new_skills": sorted(new_skills),
        "new_jd_skills": sorted(jd_only_candidates),
        "output_length": len(output.split())
    }

print("✅ Evaluation function ready")

In [ ]:
evaluation = evaluate_resume(
    comparison_resume,
    comparison_jd,
    finetuned_output
)

for key, value in evaluation.items():
    print(f"{key}: {value}")

In [ ]:
base_evaluation = evaluate_resume(
    comparison_resume,
    comparison_jd,
    base_output
)

print("BASE MODEL")
print("-" * 50)

for key, value in base_evaluation.items():
    print(f"{key}: {value}")

In [ ]:
def hallucination_check(original_resume, output):

    suspicious_skills = [
        "java", "go", "kubernetes", "docker",
        "aws", "azure", "gcp", "tensorflow",
        "pytorch", "rust", "kotlin", "scala"
    ]

    original_lower = original_resume.lower()
    output_lower = output.lower()

    suspicious = []

    for skill in suspicious_skills:
        if skill in output_lower and skill not in original_lower:
            suspicious.append(skill)

    return suspicious


print("Fine-tuned suspicious skills:")
print(hallucination_check(comparison_resume, finetuned_output))

print("\nBase suspicious skills:")
print(hallucination_check(comparison_resume, base_output))

In [ ]:
def calculate_score(evaluation):

    score = 0

    # Preserve existing skills
    if evaluation["original_skills"]:
        preservation = (
            len(evaluation["preserved_skills"]) /
            len(evaluation["original_skills"])
        )
        score += preservation * 40

    # Reasonable length
    if 100 <= evaluation["output_length"] <= 700:
        score += 20

    # Some improvement beyond the original
    if len(evaluation["output_skills"]) >= len(
        evaluation["original_skills"]
    ):
        score += 20

    # Penalize unexplained JD skills
    if len(evaluation["new_jd_skills"]) == 0:
        score += 20

    return round(score, 2)


fine_score = calculate_score(evaluation)
base_score = calculate_score(base_evaluation)

print("Fine-tuned model score:", fine_score, "/ 100")
print("Base model score:", base_score, "/ 100")

In [ ]:
print("=" * 60)
print("MODEL COMPARISON")
print("=" * 60)

print(f"Base Qwen3-4B:              {base_score}/100")
print(f"Fine-tuned Qwen3-4B + LoRA: {fine_score}/100")

if fine_score > base_score:
    print("\n✅ Fine-tuned model performed better on this test.")
elif fine_score == base_score:
    print("\n⚠️ Both models scored the same.")
else:
    print("\n⚠️ Base model performed better on this test.")

print("=" * 60)

In [ ]:
unseen_tests = [
    {
        "company": "Amazon",
        "role": "Software Development Engineer",
        "jd": """
        Build scalable software services and APIs.
        Work with databases, debugging, testing and engineering teams.
        Strong programming and problem-solving skills are required.
        """,
        "resume": """
        [Candidate Name]

        Education
        B.Tech in Computer Science Engineering

        Skills
        Python, JavaScript, SQL, Git

        Projects
        Inventory Management System
        Created a web-based inventory application.
        Implemented product search, stock tracking and SQL database
        operations.

        Experience
        Software Intern
        Assisted with application testing and debugging.
        """
    },

    {
        "company": "NVIDIA",
        "role": "Machine Learning Engineer",
        "jd": """
        Develop machine learning applications and data processing
        pipelines. Experience with Python, machine learning,
        experimentation and model evaluation is preferred.
        """,
        "resume": """
        [Candidate Name]

        Education
        B.Tech in Computer Science Engineering

        Skills
        Python, C++, SQL

        Projects
        Customer Churn Analysis
        Used Python to clean and analyze customer data.
        Built classification models and evaluated model performance
        using standard evaluation metrics.

        Experience
        Research Intern
        Assisted with data analysis and experimentation.
        """
    },

    {
        "company": "Razorpay",
        "role": "Backend Software Engineer",
        "jd": """
        Build backend services and APIs for financial products.
        Work with databases, authentication, debugging and reliable
        software systems.
        """,
        "resume": """
        [Candidate Name]

        Education
        B.Tech in Computer Science Engineering

        Skills
        Python, SQL, Git

        Projects
        Online Payment Simulator
        Developed a backend application for simulating payment
        transactions.
        Implemented transaction validation, user authentication
        and SQL database storage.

        Experience
        Software Engineering Intern
        Worked on debugging application issues and implementing
        backend features.
        """
    }
]

print("✅ Created", len(unseen_tests), "unseen evaluation cases")

In [ ]:
unseen_results = []

for i, case in enumerate(unseen_tests, 1):

    print("\n" + "=" * 90)
    print(f"UNSEEN TEST {i} — {case['company']} / {case['role']}")
    print("=" * 90)

    base_result = generate_base_resume(
        case["company"],
        case["role"],
        case["jd"],
        case["resume"]
    )

    tuned_result = optimize_resume(
        case["company"],
        case["role"],
        case["jd"],
        case["resume"]
    )

    unseen_results.append({
        "base": base_result,
        "tuned": tuned_result
    })

    print("\n--- BASE MODEL ---")
    print(base_result)

    print("\n--- FINE-TUNED MODEL ---")
    print(tuned_result)

In [ ]:
print("=" * 90)
print("HALLUCINATION CHECK")
print("=" * 90)

for i, case in enumerate(unseen_tests, 1):

    base_hallucinations = hallucination_check(
        case["resume"],
        unseen_results[i - 1]["base"]
    )

    tuned_hallucinations = hallucination_check(
        case["resume"],
        unseen_results[i - 1]["tuned"]
    )

    print(f"\nTest {i}: {case['company']}")

    print("Base model:")
    print(base_hallucinations if base_hallucinations else "None detected")

    print("Fine-tuned model:")
    print(tuned_hallucinations if tuned_hallucinations else "None detected")

In [ ]:
base_scores = []
tuned_scores = []

for i, case in enumerate(unseen_tests):

    base_eval = evaluate_resume(
        case["resume"],
        case["jd"],
        unseen_results[i]["base"]
    )

    tuned_eval = evaluate_resume(
        case["resume"],
        case["jd"],
        unseen_results[i]["tuned"]
    )

    base_scores.append(calculate_score(base_eval))
    tuned_scores.append(calculate_score(tuned_eval))


print("=" * 60)
print("UNSEEN TEST SCORES")
print("=" * 60)

for i in range(len(unseen_tests)):
    print(
        f"Test {i+1}: "
        f"Base={base_scores[i]}/100 | "
        f"Fine-tuned={tuned_scores[i]}/100"
    )

print("-" * 60)

print(
    "Average Base:",
    round(sum(base_scores) / len(base_scores), 2)
)

print(
    "Average Fine-tuned:",
    round(sum(tuned_scores) / len(tuned_scores), 2)
)

In [ ]:
avg_base = sum(base_scores) / len(base_scores)
avg_tuned = sum(tuned_scores) / len(tuned_scores)

print("=" * 60)

if avg_tuned >= avg_base + 5:
    print("✅ Fine-tuning shows a meaningful improvement.")
elif avg_tuned > avg_base:
    print("🟡 Fine-tuning shows a small improvement.")
else:
    print("🔴 Fine-tuning is not outperforming the base model yet.")

print(f"Base model average:      {avg_base:.2f}")
print(f"Fine-tuned average:      {avg_tuned:.2f}")

print("=" * 60)

In [ ]:
import os
import shutil

ADAPTER_PATH = "/content/ResumeOptimizer-LoRA"

print("Checking adapter...")

if os.path.exists(ADAPTER_PATH):
    print("✅ Adapter found")
    print(os.listdir(ADAPTER_PATH))
else:
    print("❌ Adapter folder not found")

In [ ]:
FINAL_ZIP = "/content/ResumeOptimizer-Final.zip"

shutil.make_archive(
    "/content/ResumeOptimizer-Final",
    "zip",
    ADAPTER_PATH
)

print("✅ Final adapter ZIP created")
print(FINAL_ZIP)
print(
    "Size:",
    round(os.path.getsize(FINAL_ZIP) / (1024 * 1024), 2),
    "MB"
)

In [ ]:
import zipfile

with zipfile.ZipFile(FINAL_ZIP, "r") as z:
    files = z.namelist()

print("Files inside final ZIP:")
for f in files:
    print("✓", f)

required = [
    "adapter_config.json",
    "adapter_model.safetensors"
]

if all(any(f.endswith(x) for f in files) for x in required):
    print("\n✅ TRAINED LORA ADAPTER IS SAFE")
else:
    print("\n❌ Required adapter files are missing")

In [ ]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

BASE_MODEL = "Qwen/Qwen3-4B"
ADAPTER_PATH = "/content/ResumeOptimizer-LoRA"

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit configuration
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

# Your trained LoRA
inference_model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_PATH
)

inference_model.eval()

print("✅ Base Qwen3-4B loaded")
print("✅ Trained LoRA loaded")
print("✅ Ready for inference")

In [ ]:
def resume_optimizer(
    company,
    role,
    job_description,
    existing_resume
):

    prompt = f"""
Company: {company}

Target Role: {role}

Job Description:
{job_description}

Existing Resume:
{existing_resume}

TASK:
Create a highly ATS-optimized version of the EXISTING resume
for the target company, target role, and job description.

The goal is to maximize genuine JD relevance and ATS keyword
coverage while maintaining complete factual accuracy.

IMPORTANT OPTIMIZATION STRATEGY:

1. Analyze the JD carefully and identify:
   - Required technical skills
   - Preferred technical skills
   - Programming languages
   - Frameworks and tools
   - Cloud technologies
   - Databases
   - Methodologies
   - Domain knowledge
   - Important responsibilities
   - Relevant soft skills

2. Compare those JD requirements against the existing resume.

3. Prioritize JD requirements that are supported by the existing
   resume.

4. If a skill is explicitly present anywhere in the resume,
   including:
   - Technical Skills
   - Projects
   - Experience
   - Education
   - Certifications
   then surface that skill clearly in the optimized resume.

5. If a technology or skill is clearly demonstrated by a project
   or experience, it may be included in the Skills section even
   if it was missing from the original Skills section.

6. Rewrite existing project and experience bullet points so that
   relevant JD terminology appears naturally where it is
   factually supported.

7. Use the SAME terminology as the JD when the candidate has
   demonstrated the equivalent skill.

   Example:
   JD says "REST APIs"
   Resume says "HTTP endpoints"

   If the project clearly demonstrates REST API development,
   use "REST APIs" naturally.

8. Prioritize the most important JD keywords in:
   - Professional Summary
   - Technical Skills
   - Relevant Experience
   - Relevant Projects

9. Do NOT keyword stuff.

10. Do NOT add a technology simply because it appears in the JD.

11. NEVER fabricate:
   - Skills
   - Experience
   - Projects
   - Employers
   - Certifications
   - Metrics
   - Achievements
   - Technologies
   - Responsibilities

12. If a JD skill is NOT supported by the existing resume,
   do not falsely claim that the candidate has experience with it.

13. Do not remove important existing skills merely because they
   are not mentioned in the JD.

14. Keep the resume natural and human-written.

15. Avoid generic AI-generated phrases.

16. Use varied, role-specific soft skills that are supported by
   the candidate's actual experience.

17. Make the final resume strongly focused on the target JD,
   but never at the cost of factual accuracy.

18. Return ONLY the optimized resume.

Optimized Resume:
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert ATS resume optimizer. "
                "Your job is to maximize truthful JD keyword coverage "
                "and relevance while never fabricating candidate information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        formatted_text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1400,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id
        )

    generated_tokens = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


print("✅ JD-focused ATS Resume Optimizer ready")

In [ ]:
company = "Amazon"

role = "Software Development Engineer"

job_description = """
We are looking for a Software Development Engineer to design,
develop, test and maintain software applications.

The role involves backend development, APIs, databases,
debugging, problem solving and collaboration with engineering teams.
"""

existing_resume = """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects

Student Management System
Developed a backend application with CRUD operations and
SQL database integration.

Resume Optimizer
Built a web application that analyzes resumes and improves
them based on job descriptions.

Experience

Software Development Intern
Worked on application development and debugging.
"""

optimized_resume = resume_optimizer(
    company,
    role,
    job_description,
    existing_resume
)

print("=" * 80)
print("OPTIMIZED RESUME")
print("=" * 80)
print(optimized_resume)

In [ ]:
jd = """
We are looking for a Backend Software Engineer with experience
in Python, Java, SQL, REST APIs, Docker, Kubernetes, AWS,
microservices, authentication, Git and distributed systems.
"""

resume = """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects
Secure Task Management API
Built a backend application using Python.
Implemented HTTP endpoints, user authentication and authorization.
Connected the application to a SQL database.
Used Git for source-code management.

Experience
Software Development Intern
Worked on backend application development and debugging.
"""

output = resume_optimizer(
    "Microsoft",
    "Backend Software Engineer",
    jd,
    resume
)

print(output)

In [ ]:
def extract_jd_requirements(job_description):

    prompt = f"""
Analyze this job description.

JOB DESCRIPTION:
{job_description}

Extract the important requirements and return ONLY this structure:

TECHNICAL_SKILLS:
- skill 1
- skill 2
- skill 3

TOOLS_AND_TECHNOLOGIES:
- tool 1
- tool 2

RESPONSIBILITIES:
- responsibility 1
- responsibility 2

SOFT_SKILLS:
- skill 1
- skill 2

DOMAIN_KEYWORDS:
- keyword 1
- keyword 2

Do not invent requirements that are not present in the JD.
"""

    messages = [
        {
            "role": "system",
            "content": "You accurately extract requirements from job descriptions."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print("✅ JD requirement extractor ready")

In [ ]:
jd_requirements = extract_jd_requirements(comparison_jd)

print("=" * 80)
print("EXTRACTED JD REQUIREMENTS")
print("=" * 80)
print(jd_requirements)

NameError: name 'extract_jd_requirements' is not defined

In [ ]:
def ats_resume_optimizer(
    company,
    role,
    job_description,
    existing_resume
):

    # First understand the JD
    jd_requirements = extract_jd_requirements(job_description)

    prompt = f"""
Company: {company}

Target Role: {role}

JOB DESCRIPTION:
{job_description}

EXTRACTED JD REQUIREMENTS:
{jd_requirements}

EXISTING RESUME:
{existing_resume}

Create an ATS-optimized version of the existing resume.

OBJECTIVE:
Maximize truthful alignment between the resume and the JD.

FOLLOW THESE STEPS:

1. Identify the candidate's existing skills, experience,
   projects, education and demonstrated capabilities.

2. Map the candidate's existing evidence to the JD requirements.

3. Prioritize JD keywords that are supported by the candidate.

4. Use important JD terminology naturally throughout the resume
   wherever the candidate's existing evidence supports it.

5. Surface demonstrated skills into the Technical Skills section
   when appropriate.

6. Rewrite relevant bullets to emphasize JD-aligned responsibilities
   without changing the underlying facts.

7. Put the strongest JD-relevant information first.

8. Keep unrelated information but give it less emphasis when
   appropriate.

9. Use a concise role-specific professional summary.

10. Use evidence-based soft skills.

STRICT FACTUALITY:

Never invent or falsely claim:
- technologies
- programming languages
- frameworks
- cloud platforms
- certifications
- experience
- projects
- achievements
- metrics
- employers
- responsibilities

If a JD requirement is unsupported by the resume, DO NOT claim
that the candidate has that skill.

However, if the candidate clearly demonstrates a capability through
a project or experience, use the appropriate JD terminology even
if the original resume used different wording.

Example:

JD: REST APIs
Resume: HTTP endpoints

If the project clearly demonstrates API development, the optimized
resume may use "REST APIs".

The goal is:

HIGH ATS RELEVANCE
+
HIGH JD KEYWORD COVERAGE
+
100% FACTUAL ACCURACY
+
NATURAL HUMAN WRITING

Return ONLY the final optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a senior ATS resume optimization expert. "
                "Maximize truthful job-description alignment and "
                "keyword coverage while preserving factual accuracy."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1400,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print("✅ Strong ATS optimizer ready")

In [ ]:
final_output = ats_resume_optimizer(
    comparison_company,
    comparison_role,
    comparison_jd,
    comparison_resume
)

print("=" * 90)
print("FINAL ATS-OPTIMIZED RESUME")
print("=" * 90)
print(final_output)

In [ ]:
def keyword_coverage(jd, resume):

    jd_words = set(
        re.findall(r"\b[a-zA-Z][a-zA-Z+#.-]{2,}\b", jd.lower())
    )

    resume_words = set(
        re.findall(r"\b[a-zA-Z][a-zA-Z+#.-]{2,}\b", resume.lower())
    )

    matched = jd_words.intersection(resume_words)

    coverage = (
        len(matched) / len(jd_words) * 100
        if jd_words else 0
    )

    return round(coverage, 2), sorted(matched)


coverage, matched = keyword_coverage(
    comparison_jd,
    final_output
)

print("JD keyword overlap:", coverage, "%")
print("\nMatched terms:")
print(matched)

In [ ]:
import re

def extract_keywords(text):
    return set(
        re.findall(
            r"\b[a-zA-Z][a-zA-Z0-9+#./-]{2,}\b",
            text.lower()
        )
    )


def calculate_ats_score(job_description, resume):

    jd_words = extract_keywords(job_description)
    resume_words = extract_keywords(resume)

    if not jd_words:
        return 0

    matched = jd_words.intersection(resume_words)

    coverage = len(matched) / len(jd_words)

    score = coverage * 100

    return round(score, 2)


print("✅ ATS scorer ready")

In [ ]:
ats_score = calculate_ats_score(
    comparison_jd,
    final_output
)

print("ATS Alignment Score:", ats_score, "/ 100")

if ats_score >= 85:
    print("✅ TARGET ACHIEVED — 85+")
else:
    print("⚠️ Below 85 — resume needs stronger JD alignment")

In [ ]:
def weighted_ats_score(jd_requirements, resume):
    """
    jd_requirements should contain:
    TECHNICAL_SKILLS
    TOOLS_AND_TECHNOLOGIES
    RESPONSIBILITIES
    SOFT_SKILLS
    DOMAIN_KEYWORDS
    """

    resume_lower = resume.lower()

    categories = {
        "technical_skills": (
            jd_requirements.get("technical_skills", []),
            30
        ),
        "tools_and_technologies": (
            jd_requirements.get("tools_and_technologies", []),
            20
        ),
        "responsibilities": (
            jd_requirements.get("responsibilities", []),
            20
        ),
        "domain_keywords": (
            jd_requirements.get("domain_keywords", []),
            15
        ),
        "soft_skills": (
            jd_requirements.get("soft_skills", []),
            5
        )
    }

    category_scores = {}
    total_score = 0

    for category, (keywords, weight) in categories.items():

        if not keywords:
            category_scores[category] = 0
            continue

        matched = 0

        for keyword in keywords:
            keyword = keyword.lower().strip()

            if keyword and keyword in resume_lower:
                matched += 1

        percentage = matched / len(keywords)

        category_score = percentage * weight

        category_scores[category] = round(category_score, 2)
        total_score += category_score

    # Basic resume structure bonus
    structure_points = 0

    sections = [
        "summary",
        "skills",
        "experience",
        "projects",
        "education"
    ]

    for section in sections:
        if section in resume_lower:
            structure_points += 2

    structure_points = min(structure_points, 10)

    total_score += structure_points

    return {
        "total_score": round(min(total_score, 100), 2),
        "category_scores": category_scores,
        "structure_score": structure_points
    }


print("✅ Weighted ATS scorer ready")

In [ ]:
import json

def extract_structured_jd(job_description):

    prompt = f"""
Analyze this job description and extract its important requirements.

JOB DESCRIPTION:
{job_description}

Return ONLY valid JSON.

Use exactly this format:

{{
  "technical_skills": [],
  "tools_and_technologies": [],
  "responsibilities": [],
  "soft_skills": [],
  "domain_keywords": []
}}

Rules:
- Include important JD-specific terms.
- Do not add information not present in the JD.
- Keep each item concise.
"""

    messages = [
        {
            "role": "system",
            "content": "You extract job requirements accurately into JSON."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=500,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    # Remove accidental markdown fences
    result = result.replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(result)
    except json.JSONDecodeError:
        print("⚠️ Could not parse JSON")
        print(result)

        return {
            "technical_skills": [],
            "tools_and_technologies": [],
            "responsibilities": [],
            "soft_skills": [],
            "domain_keywords": []
        }


print("✅ Structured JD extractor ready")

In [ ]:
structured_jd = extract_structured_jd(
    comparison_jd
)

print(json.dumps(structured_jd, indent=2))

score_result = weighted_ats_score(
    structured_jd,
    final_output
)

print("\n" + "=" * 60)
print("ATS SCORE")
print("=" * 60)

print("Total:", score_result["total_score"], "/ 100")

for category, score in score_result["category_scores"].items():
    print(f"{category}: {score}")

print("structure:", score_result["structure_score"])

In [ ]:
TARGET_ATS_SCORE = 85

score = score_result["total_score"]

print("=" * 60)

if score >= TARGET_ATS_SCORE:
    print(f"✅ ATS TARGET ACHIEVED: {score}/100")
else:
    print(f"⚠️ ATS SCORE: {score}/100")
    print(f"Need approximately {TARGET_ATS_SCORE - score:.2f} more points.")

print("=" * 60)

In [ ]:
def final_resume_optimizer(
    company,
    role,
    job_description,
    existing_resume
):

    jd_requirements = extract_structured_jd(job_description)

    prompt = f"""
You are an expert ATS resume optimizer and professional technical
resume writer.

COMPANY:
{company}

TARGET ROLE:
{role}

JOB DESCRIPTION:
{job_description}

EXTRACTED JD REQUIREMENTS:
{json.dumps(jd_requirements, indent=2)}

EXISTING RESUME:
{existing_resume}

YOUR OBJECTIVE:

Create a highly ATS-friendly, one-page resume specifically aligned
to the target company, target role, and job description.

TARGET ATS ALIGNMENT:
Aim for an ATS alignment score between 85 and 97 when the candidate's
existing information provides enough evidence.

Do NOT manipulate the score or fabricate information to achieve the
target.

==================================================
ATS OPTIMIZATION
==================================================

1. Identify the most important keywords and requirements in the JD.

2. Prioritize JD requirements in this order:

   - Core technical skills
   - Programming languages
   - Frameworks
   - Tools
   - Databases
   - Cloud technologies
   - Role responsibilities
   - Domain terminology
   - Relevant soft skills

3. Match JD terminology with equivalent concepts already demonstrated
   by the candidate.

4. If a skill is explicitly present anywhere in the existing resume,
   surface it clearly.

5. If a skill is demonstrated through a project or experience,
   it may be added to the Technical Skills section.

6. Use important JD keywords naturally in relevant project and
   experience bullets when the candidate has evidence for them.

7. Prioritize high-value JD keywords rather than stuffing every word
   from the JD.

==================================================
ALIGNMENT
==================================================

Make the resume strongly aligned with the TARGET ROLE.

The:

SUMMARY
SKILLS
EXPERIENCE
PROJECTS

should collectively communicate why the candidate fits this
specific role.

Put the most JD-relevant information first.

Do not simply repeat the JD.

Rewrite existing information to emphasize relevant responsibilities
and technologies while preserving the original meaning.

==================================================
FACTUAL ACCURACY
==================================================

NEVER INVENT:

- Skills
- Programming languages
- Frameworks
- Tools
- Cloud platforms
- Projects
- Employers
- Internships
- Certifications
- Achievements
- Metrics
- Responsibilities
- Job titles
- Experience

If the JD contains a technology that is not supported by the
existing resume, do NOT falsely claim proficiency.

Do not add unsupported keywords just to increase ATS score.

==================================================
ONE-PAGE REQUIREMENT
==================================================

The final resume MUST fit on ONE PAGE.

To achieve this:

- Remove unnecessary repetition.
- Keep the summary concise.
- Use concise bullet points.
- Prioritize relevant experience and projects.
- Remove low-value wording.
- Avoid long paragraphs.
- Avoid excessive bullet points.
- Do not repeat the same skill multiple times unnecessarily.

Target approximately 400–600 words depending on the amount of
information available.

==================================================
ATS-FRIENDLY FORMAT
==================================================

Use a simple single-column resume structure.

Allowed sections:

NAME
CONTACT INFORMATION
SUMMARY
TECHNICAL SKILLS
EXPERIENCE
PROJECTS
EDUCATION
CERTIFICATIONS

Use standard section headings.

Do NOT use:

- Tables
- Multiple columns
- Text boxes
- Graphics
- Images
- Icons
- Logos
- Skill bars
- Progress bars
- Decorative elements
- Headers containing important information
- Footers containing important information
- Unusual symbols

The resume must be easy for ATS software to parse.

==================================================
VISUAL STYLE
==================================================

The resume must use ONLY BLACK TEXT.

No colored text.

No colored headings.

No colored backgrounds.

No unnecessary design elements.

Use a clean professional black-and-white layout.

==================================================
WRITING STYLE
==================================================

Write naturally and professionally.

Avoid obvious AI-generated language such as:

"results-driven professional"
"passionate individual"
"proven track record"
"dynamic professional"
"leveraged cutting-edge technology"
"highly motivated"
"detail-oriented team player"

unless the wording is genuinely appropriate and supported.

Use concise, specific and human-written bullet points.

==================================================
FINAL CHECK
==================================================

Before returning the resume, internally check:

1. Is the resume strongly aligned with the JD?
2. Are important supported JD keywords included?
3. Are unsupported skills excluded?
4. Are the candidate's original facts preserved?
5. Is the resume concise enough for one page?
6. Is the structure ATS-friendly?
7. Is it single-column?
8. Is it black-and-white?
9. Is the writing natural?
10. Is there unnecessary repetition?

Return ONLY the final resume.

Do not explain your changes.
Do not provide an ATS score inside the resume.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are a senior ATS resume optimization specialist. "
                "Maximize truthful JD alignment, ATS compatibility and "
                "resume quality while never fabricating candidate information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1400,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    return result


print("✅ Final ATS + Alignment + One-Page optimizer ready")

In [ ]:
final_resume = final_resume_optimizer(
    company="Microsoft",
    role="Backend Software Engineer",
    job_description=comparison_jd,
    existing_resume=comparison_resume
)

print("=" * 90)
print("FINAL OPTIMIZED RESUME")
print("=" * 90)
print(final_resume)

In [ ]:
structured_jd = extract_structured_jd(comparison_jd)

score_result = weighted_ats_score(
    structured_jd,
    final_resume
)

print("=" * 70)
print("ATS / ALIGNMENT REPORT")
print("=" * 70)

print(f"\nOverall ATS Alignment: {score_result['total_score']}/100")

print("\nCategory Scores:")

for category, score in score_result["category_scores"].items():
    print(f"{category}: {score}")

print(f"Structure: {score_result['structure_score']}/10")

print("\nTarget: 85–97")

if 85 <= score_result["total_score"] <= 97:
    print("✅ TARGET RANGE ACHIEVED")
elif score_result["total_score"] < 85:
    print("⚠️ BELOW TARGET — needs stronger truthful alignment")
else:
    print("⚠️ ABOVE TARGET RANGE — inspect for keyword stuffing")

In [ ]:
!pip install -q reportlab pypdf
print("✅ PDF libraries ready")

In [ ]:
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    KeepTogether
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.units import inch
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.pdfbase import pdfmetrics
import re
import os


PDF_PATH = "/content/Optimized_Resume.pdf"


# -----------------------------
# Styles
# -----------------------------

styles = getSampleStyleSheet()

name_style = ParagraphStyle(
    "Name",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=15,
    leading=17,
    alignment=TA_LEFT,
    textColor="black",
    spaceAfter=3
)

contact_style = ParagraphStyle(
    "Contact",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=8.5,
    leading=10,
    alignment=TA_LEFT,
    textColor="black",
    spaceAfter=5
)

section_style = ParagraphStyle(
    "Section",
    parent=styles["Normal"],
    fontName="Helvetica-Bold",
    fontSize=10,
    leading=12,
    alignment=TA_LEFT,
    textColor="black",
    spaceBefore=5,
    spaceAfter=2
)

body_style = ParagraphStyle(
    "Body",
    parent=styles["Normal"],
    fontName="Helvetica",
    fontSize=8.5,
    leading=10.5,
    alignment=TA_LEFT,
    textColor="black",
    spaceAfter=2
)

bullet_style = ParagraphStyle(
    "Bullet",
    parent=body_style,
    leftIndent=10,
    firstLineIndent=-7,
    spaceAfter=1.5
)


# -----------------------------
# Clean markdown
# -----------------------------

def clean_text(text):

    text = text.replace("**", "")
    text = text.replace("__", "")
    text = text.replace("###", "")
    text = text.replace("##", "")
    text = text.replace("#", "")

    return text.strip()


# -----------------------------
# Convert resume text → PDF
# -----------------------------

def build_resume_pdf(resume_text, output_path):

    doc = SimpleDocTemplate(
        output_path,
        pagesize=LETTER,
        rightMargin=0.48 * inch,
        leftMargin=0.48 * inch,
        topMargin=0.42 * inch,
        bottomMargin=0.42 * inch
    )

    story = []

    lines = resume_text.splitlines()

    first_content = True

    section_names = {
        "summary",
        "professional summary",
        "technical skills",
        "skills",
        "experience",
        "work experience",
        "projects",
        "education",
        "certifications",
        "achievements"
    }

    for raw_line in lines:

        line = clean_text(raw_line)

        if not line:
            continue

        lower = line.lower().strip()

        # Remove markdown bullets
        if line.startswith("- "):
            line = line[2:].strip()

        if line.startswith("* "):
            line = line[2:].strip()

        # First line = candidate name
        if first_content:
            story.append(
                Paragraph(
                    line,
                    name_style
                )
            )

            first_content = False
            continue

        # Section heading
        if lower in section_names:

            story.append(
                Paragraph(
                    line.upper(),
                    section_style
                )
            )

            continue

        # Bullet
        if raw_line.strip().startswith(("-", "*", "•")):

            story.append(
                Paragraph(
                    "• " + line,
                    bullet_style
                )
            )

        else:

            story.append(
                Paragraph(
                    line,
                    body_style
                )
            )

    doc.build(story)

    return output_path


build_resume_pdf(
    final_resume,
    PDF_PATH
)

print("✅ PDF created")
print(PDF_PATH)

ModuleNotFoundError: No module named 'reportlab'

In [ ]:
from pypdf import PdfReader

reader = PdfReader(PDF_PATH)

page_count = len(reader.pages)

print("=" * 60)
print("RESUME PAGE CHECK")
print("=" * 60)

print("Pages:", page_count)

if page_count == 1:
    print("✅ PERFECT — Resume fits on ONE PAGE")
else:
    print("⚠️ Resume is", page_count, "pages")

In [ ]:
reader = PdfReader(PDF_PATH)

pdf_text = ""

for page in reader.pages:
    pdf_text += page.extract_text() or ""

print("=" * 60)
print("PDF TEXT CHECK")
print("=" * 60)

print("Characters extracted:", len(pdf_text))

if len(pdf_text) > 100:
    print("✅ ATS-readable text detected")
else:
    print("❌ PDF may not be ATS-readable")

In [ ]:
print("=" * 70)
print("FINAL RESUME VALIDATION")
print("=" * 70)

checks = {
    "ATS score 85–97": 85 <= score_result["total_score"] <= 97,
    "One page": page_count == 1,
    "Text readable": len(pdf_text) > 100,
    "Single-column layout": True,
    "No graphics": True,
    "Black text": True,
    "No tables": True
}

for check, passed in checks.items():
    print(
        ("✅ " if passed else "❌ ") + check
    )

print("=" * 70)

In [ ]:
def improve_until_target(
    company,
    role,
    job_description,
    existing_resume,
    target_score=85,
    max_attempts=3
):

    current_resume = existing_resume
    history = []

    for attempt in range(1, max_attempts + 1):

        print(f"\n{'='*70}")
        print(f"ATS OPTIMIZATION ATTEMPT {attempt}")
        print(f"{'='*70}")

        # Generate optimized resume
        optimized = final_resume_optimizer(
            company=company,
            role=role,
            job_description=job_description,
            existing_resume=current_resume
        )

        # Extract JD requirements
        jd_requirements = extract_structured_jd(
            job_description
        )

        # Calculate score
        score_result = weighted_ats_score(
            jd_requirements,
            optimized
        )

        score = score_result["total_score"]

        print(f"ATS Score: {score}/100")

        history.append({
            "attempt": attempt,
            "score": score,
            "resume": optimized
        })

        # Target achieved
        if target_score <= score <= 97:

            print("✅ Target ATS range achieved!")

            return {
                "resume": optimized,
                "score": score,
                "history": history,
                "status": "TARGET_ACHIEVED"
            }

        # Above target
        if score > 97:

            print("⚠️ Score above target range.")
            print("Checking for possible keyword stuffing...")

            return {
                "resume": optimized,
                "score": score,
                "history": history,
                "status": "ABOVE_TARGET"
            }

        # Below target → create feedback for next iteration
        missing = []

        for category, keywords in jd_requirements.items():

            if not isinstance(keywords, list):
                continue

            for keyword in keywords:

                if keyword.lower() not in optimized.lower():
                    missing.append(keyword)

        if missing:

            print("Missing JD terms supported for review:")
            print(missing[:20])

        current_resume = optimized

    # If target wasn't reached
    best = max(
        history,
        key=lambda x: x["score"]
    )

    print("\n⚠️ Maximum optimization attempts reached.")
    print(f"Best score: {best['score']}/100")

    return {
        "resume": best["resume"],
        "score": best["score"],
        "history": history,
        "status": "BEST_AVAILABLE"
    }


print("✅ Automatic ATS optimization loop ready")

In [ ]:
result = improve_until_target(
    company="Microsoft",
    role="Backend Software Engineer",
    job_description=comparison_jd,
    existing_resume=comparison_resume,
    target_score=85,
    max_attempts=3
)

print("\n" + "=" * 80)
print("FINAL RESULT")
print("=" * 80)

print("Status:", result["status"])
print("ATS Score:", result["score"])

print("\n" + "=" * 80)
print("OPTIMIZED RESUME")
print("=" * 80)

print(result["resume"])

In [ ]:
print("=" * 60)
print("ATS IMPROVEMENT HISTORY")
print("=" * 60)

for item in result["history"]:
    print(
        f"Attempt {item['attempt']}: "
        f"{item['score']}/100"
    )

In [ ]:
import json
import re
import torch


def extract_rich_jd_data(job_description):

    prompt = f"""
You are an expert ATS/job-description analyzer.

Analyze this complete job description:

{job_description}

Return ONLY valid JSON using this exact structure:

{{
  "role_title": "",
  "technical_skills": [],
  "programming_languages": [],
  "frameworks": [],
  "libraries": [],
  "tools": [],
  "databases": [],
  "cloud": [],
  "devops": [],
  "apis": [],
  "architecture": [],
  "methodologies": [],
  "responsibilities": [],
  "domain_terms": [],
  "soft_skills": [],
  "education_requirements": [],
  "preferred_requirements": [],
  "important_phrases": []
}}

Rules:

1. Extract as many meaningful ATS-relevant terms as possible.
2. Preserve the actual terminology used in the JD.
3. Include synonyms only when they are genuinely equivalent.
4. Separate technical skills from responsibilities.
5. Include important multi-word phrases.
6. Do not include generic filler words.
7. Do not invent anything.
8. Return valid JSON only.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert ATS job-description analyzer."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    result = result.replace("```json", "")
    result = result.replace("```", "")
    result = result.strip()

    try:
        return json.loads(result)

    except Exception:
        print("⚠️ JSON parsing failed")
        print(result)

        return {
            "role_title": "",
            "technical_skills": [],
            "programming_languages": [],
            "frameworks": [],
            "libraries": [],
            "tools": [],
            "databases": [],
            "cloud": [],
            "devops": [],
            "apis": [],
            "architecture": [],
            "methodologies": [],
            "responsibilities": [],
            "domain_terms": [],
            "soft_skills": [],
            "education_requirements": [],
            "preferred_requirements": [],
            "important_phrases": []
        }


print("✅ Rich JD analyzer ready")

In [ ]:
def extract_candidate_evidence(resume):

    prompt = f"""
Analyze the following resume.

RESUME:
{resume}

Extract all skills and capabilities that are ACTUALLY supported
by the resume.

Return ONLY valid JSON:

{{
  "technical_skills": [],
  "programming_languages": [],
  "frameworks": [],
  "libraries": [],
  "tools": [],
  "databases": [],
  "cloud": [],
  "devops": [],
  "apis": [],
  "architecture": [],
  "methodologies": [],
  "responsibilities": [],
  "domain_terms": [],
  "soft_skills": [],
  "project_evidence": [],
  "experience_evidence": [],
  "education_evidence": []
}}

IMPORTANT:

Only extract information explicitly stated or clearly demonstrated
in the resume.

Do NOT infer unsupported technologies.

For example:

If a project says:
"Built HTTP endpoints and authentication"

you may identify:
"API development"
"authentication"

But do NOT identify:
"Kubernetes"
"AWS"
"Docker"

unless they are actually supported.

Return JSON only.
"""

    messages = [
        {
            "role": "system",
            "content": "You extract factual candidate evidence from resumes."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():
        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    result = tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

    result = result.replace("```json", "")
    result = result.replace("```", "")
    result = result.strip()

    try:
        return json.loads(result)

    except Exception:
        print("⚠️ Candidate evidence JSON failed")

        return {
            "technical_skills": [],
            "programming_languages": [],
            "frameworks": [],
            "libraries": [],
            "tools": [],
            "databases": [],
            "cloud": [],
            "devops": [],
            "apis": [],
            "architecture": [],
            "methodologies": [],
            "responsibilities": [],
            "domain_terms": [],
            "soft_skills": [],
            "project_evidence": [],
            "experience_evidence": [],
            "education_evidence": []
        }


print("✅ Candidate evidence extractor ready")

In [ ]:
def build_alignment_map(jd_data, candidate_data):

    jd_items = []

    for category, values in jd_data.items():

        if isinstance(values, list):

            for value in values:
                if isinstance(value, str):
                    jd_items.append(value.strip())

    candidate_items = []

    for category, values in candidate_data.items():

        if isinstance(values, list):

            for value in values:
                if isinstance(value, str):
                    candidate_items.append(value.strip())

    candidate_text = " | ".join(candidate_items).lower()

    supported = []
    unsupported = []

    for item in jd_items:

        if item.lower() in candidate_text:
            supported.append(item)
        else:
            unsupported.append(item)

    return {
        "supported_jd_keywords": sorted(set(supported)),
        "unsupported_jd_keywords": sorted(set(unsupported)),
        "total_jd_keywords": len(set(jd_items)),
        "supported_count": len(set(supported))
    }


print("✅ JD ↔ resume alignment mapper ready")

In [ ]:
rich_jd = extract_rich_jd_data(comparison_jd)

candidate_evidence = extract_candidate_evidence(
    comparison_resume
)

alignment = build_alignment_map(
    rich_jd,
    candidate_evidence
)

print("=" * 80)
print("JD ↔ RESUME ALIGNMENT")
print("=" * 80)

print(
    "Total JD keywords:",
    alignment["total_jd_keywords"]
)

print(
    "Supported keywords:",
    alignment["supported_count"]
)

print("\nSUPPORTED:")
for item in alignment["supported_jd_keywords"]:
    print("✅", item)

print("\nUNSUPPORTED:")
for item in alignment["unsupported_jd_keywords"]:
    print("⚪", item)

In [ ]:
def high_coverage_optimizer(
    company,
    role,
    job_description,
    existing_resume
):

    jd_data = extract_rich_jd_data(job_description)

    candidate_data = extract_candidate_evidence(
        existing_resume
    )

    alignment = build_alignment_map(
        jd_data,
        candidate_data
    )

    prompt = f"""
You are an elite ATS resume optimization specialist.

COMPANY:
{company}

TARGET ROLE:
{role}

JOB DESCRIPTION:
{job_description}

COMPLETE JD ANALYSIS:
{json.dumps(jd_data, indent=2)}

CANDIDATE EVIDENCE:
{json.dumps(candidate_data, indent=2)}

JD ↔ CANDIDATE ALIGNMENT:
{json.dumps(alignment, indent=2)}

ORIGINAL RESUME:
{existing_resume}

==================================================
PRIMARY OBJECTIVE
==================================================

Create the strongest possible ATS-friendly resume for this specific
role.

Target an ATS alignment score of approximately 85–97 when the
candidate's evidence supports it.

Maximize truthful keyword coverage.

==================================================
KEYWORD STRATEGY
==================================================

Use supported JD keywords aggressively but naturally.

A supported keyword can appear in:

1. Professional Summary
2. Technical Skills
3. Experience
4. Projects
5. Certifications
6. Education when relevant

Do not unnecessarily repeat the same keyword.

Prioritize:

- Required skills
- Core technologies
- Programming languages
- Tools
- Databases
- APIs
- Architecture
- Responsibilities
- Domain terminology

==================================================
SYNONYM ALIGNMENT
==================================================

When the candidate's evidence genuinely supports a JD concept,
prefer the terminology used in the JD.

Example:

JD:
"REST APIs"

Resume:
"HTTP endpoints"

If the project clearly demonstrates API development, write:

"REST APIs"

Example:

JD:
"database management"

Resume:
"SQL database integration"

These can be aligned naturally.

Do NOT perform false equivalence.

==================================================
MISSING KEYWORDS
==================================================

For every unsupported JD keyword:

DO NOT fabricate it.

Do NOT claim experience with it.

Do NOT add it to Technical Skills.

Instead, maximize the keywords that ARE supported.

==================================================
RESUME CONTENT
==================================================

Rewrite existing bullets to emphasize:

ACTION
+
TECHNOLOGY
+
RESPONSIBILITY
+
RESULT/OUTCOME

ONLY when the underlying information exists.

Do not invent metrics.

==================================================
SUMMARY
==================================================

Create a highly targeted 2–4 line summary.

Mention the target role and the strongest relevant candidate
skills supported by the resume.

==================================================
TECHNICAL SKILLS
==================================================

Organize skills for ATS readability.

Prioritize skills matching the JD.

Include demonstrated project/experience skills even if they were
missing from the original Skills section.

==================================================
PROJECTS
==================================================

Prioritize projects relevant to the target role.

Rewrite project descriptions using important supported JD terminology.

==================================================
EXPERIENCE
==================================================

Prioritize responsibilities relevant to the JD.

Use JD terminology where factually supported.

==================================================
SOFT SKILLS
==================================================

Only include soft skills supported by the candidate's experience.

Do not use the same generic list for every candidate.

==================================================
ONE PAGE
==================================================

Keep the final resume approximately 400–600 words.

Remove repetition and low-value content.

==================================================
ATS FORMAT
==================================================

Use:

- Single column
- Standard headings
- Plain text
- Simple bullets
- No tables
- No graphics
- No icons
- No text boxes
- No columns
- No skill bars
- No decorative elements

==================================================
VISUAL FORMAT
==================================================

Black and white only.

No colors.

==================================================
FINAL QUALITY TARGET
==================================================

The resume should maximize:

JD keyword coverage
+
JD responsibility alignment
+
Role relevance
+
Candidate evidence
+
ATS readability
+
Human writing quality

while maintaining 100% factual accuracy.

Return ONLY the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an elite ATS resume optimizer. "
                "Maximize truthful JD alignment and keyword coverage "
                "without fabricating candidate information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():

        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1600,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print("✅ High-coverage ATS optimizer ready")

In [ ]:
def generate_multiple_resumes(
    company,
    role,
    job_description,
    existing_resume,
    num_versions=3
):

    versions = []

    for i in range(num_versions):

        print(f"Generating version {i + 1}/{num_versions}...")

        resume = high_coverage_optimizer(
            company=company,
            role=role,
            job_description=job_description,
            existing_resume=existing_resume
        )

        versions.append(resume)

    print(f"✅ Generated {len(versions)} versions")

    return versions


print("✅ Multi-version optimizer ready")

In [ ]:
resume_versions = generate_multiple_resumes(
    company="Microsoft",
    role="Backend Software Engineer",
    job_description=comparison_jd,
    existing_resume=comparison_resume,
    num_versions=3
)

print("Number of versions:", len(resume_versions))

In [ ]:
jd_data = extract_rich_jd_data(comparison_jd)

scored_versions = []

for i, resume in enumerate(resume_versions):

    score_result = weighted_ats_score(
        jd_data,
        resume
    )

    scored_versions.append({
        "version": i + 1,
        "score": score_result["total_score"],
        "resume": resume
    })

    print(
        f"Version {i + 1}: "
        f"{score_result['total_score']}/100"
    )

In [ ]:
# Prefer scores inside the desired 85–97 range.
target_versions = [
    item for item in scored_versions
    if 85 <= item["score"] <= 97
]

if target_versions:

    best_version = max(
        target_versions,
        key=lambda x: x["score"]
    )

else:

    # If none reaches 85, choose the highest score available.
    best_version = max(
        scored_versions,
        key=lambda x: x["score"]
    )

print("=" * 70)
print("BEST VERSION")
print("=" * 70)

print(
    "Selected version:",
    best_version["version"]
)

print(
    "ATS alignment:",
    best_version["score"],
    "/100"
)

In [ ]:
best_resume = best_version["resume"]

print("=" * 90)
print("BEST ATS-OPTIMIZED RESUME")
print("=" * 90)

print(best_resume)

In [ ]:
def detailed_alignment(jd_data, resume):

    resume_lower = resume.lower()

    results = {}

    for category, keywords in jd_data.items():

        if not isinstance(keywords, list):
            continue

        if not keywords:
            continue

        matched = []
        missing = []

        for keyword in keywords:

            keyword = str(keyword).strip()

            if keyword.lower() in resume_lower:
                matched.append(keyword)
            else:
                missing.append(keyword)

        results[category] = {
            "matched": matched,
            "missing": missing,
            "coverage": round(
                len(matched) / len(keywords) * 100,
                2
            )
        }

    return results


alignment_report = detailed_alignment(
    jd_data,
    best_resume
)

for category, data in alignment_report.items():

    print("\n" + category.upper())
    print("-" * 50)

    print(
        "Coverage:",
        data["coverage"],
        "%"
    )

    print("Matched:", data["matched"])

    print("Missing:", data["missing"])

In [ ]:
def resume_length_check(resume):

    words = resume.split()

    return {
        "word_count": len(words),
        "estimated_pages": (
            1 if len(words) <= 600 else 2
        )
    }


length_info = resume_length_check(best_resume)

print("Word count:", length_info["word_count"])
print("Estimated pages:", length_info["estimated_pages"])

In [ ]:
def compress_resume(
    resume,
    job_description,
    target_words=550
):

    prompt = f"""
You are an expert one-page ATS resume editor.

JOB DESCRIPTION:
{job_description}

CURRENT RESUME:
{resume}

Rewrite this resume so it fits comfortably on ONE PAGE.

TARGET:
Approximately {target_words} words or fewer.

IMPORTANT:

1. Preserve all factual information.
2. Do not invent anything.
3. Preserve the highest-value JD-aligned keywords.
4. Preserve important technical skills.
5. Preserve the strongest relevant projects.
6. Preserve relevant experience.
7. Keep the professional summary concise.
8. Remove repetition.
9. Remove low-value wording.
10. Combine similar points when appropriate.
11. Use concise achievement-oriented bullets.
12. Do not remove an important skill merely to reduce length.
13. Do not add unsupported JD skills.
14. Maintain strong ATS alignment.
15. Use standard ATS-friendly section headings.
16. Keep single-column structure.
17. No tables.
18. No graphics.
19. No icons.
20. No decorative formatting.
21. Black-and-white resume only.

The result must be concise enough for a one-page resume.

Return ONLY the compressed resume.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You are an expert ATS resume editor specializing "
                "in concise one-page technical resumes."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():

        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1200,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print("✅ One-page compression function ready")

In [ ]:
length_info = resume_length_check(best_resume)

if length_info["word_count"] > 550:

    print(
        f"⚠️ Resume has {length_info['word_count']} words."
    )

    print("Compressing while preserving ATS alignment...")

    final_resume = compress_resume(
        best_resume,
        comparison_jd,
        target_words=550
    )

else:

    print(
        f"✅ Resume has {length_info['word_count']} words."
    )

    final_resume = best_resume


print("\nFinal word count:", len(final_resume.split()))

In [ ]:
final_jd_data = extract_rich_jd_data(
    comparison_jd
)

final_score_result = weighted_ats_score(
    final_jd_data,
    final_resume
)

print("=" * 70)
print("FINAL ATS CHECK")
print("=" * 70)

print(
    "ATS Alignment:",
    final_score_result["total_score"],
    "/100"
)

print(
    "Word count:",
    len(final_resume.split())
)

if final_score_result["total_score"] >= 85:
    print("✅ ATS target maintained")
else:
    print("⚠️ ATS score dropped below 85")

In [ ]:
FINAL_PDF = "/content/Final_ATS_Resume.pdf"

build_resume_pdf(
    final_resume,
    FINAL_PDF
)

print("✅ Final PDF generated")
print(FINAL_PDF)

In [ ]:
from pypdf import PdfReader

reader = PdfReader(FINAL_PDF)

pages = len(reader.pages)

print("=" * 60)
print("FINAL PDF VALIDATION")
print("=" * 60)

print("Pages:", pages)

if pages == 1:
    print("✅ ONE PAGE")
else:
    print("❌ More than one page")

In [ ]:
print("=" * 80)
print("RESUME OPTIMIZER — FINAL REPORT")
print("=" * 80)

print(f"ATS Alignment Score : {final_score_result['total_score']}/100")
print(f"Word Count          : {len(final_resume.split())}")
print(f"PDF Pages           : {pages}")
print("Layout              : Single Column")
print("Color               : Black")
print("Graphics            : None")
print("Tables              : None")
print("ATS Format          : Yes")

if (
    85 <= final_score_result["total_score"] <= 97
    and pages == 1
):
    print("\n🎯 FINAL TARGET ACHIEVED")
else:
    print("\n⚠️ Needs further optimization")

print("=" * 80)

In [ ]:
from reportlab.lib.pagesizes import LETTER
from reportlab.platypus import (
    SimpleDocTemplate,
    Paragraph,
    Spacer,
    HRFlowable
)
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.enums import TA_LEFT
from reportlab.lib.units import inch
from pypdf import PdfReader
import re
import os


def create_professional_ats_pdf(
    resume_text,
    output_path="/content/Final_ATS_Resume.pdf"
):

    # -----------------------------
    # Document
    # -----------------------------

    doc = SimpleDocTemplate(
        output_path,
        pagesize=LETTER,
        leftMargin=0.48 * inch,
        rightMargin=0.48 * inch,
        topMargin=0.38 * inch,
        bottomMargin=0.38 * inch
    )

    # -----------------------------
    # Styles
    # -----------------------------

    name_style = ParagraphStyle(
        "ResumeName",
        fontName="Helvetica-Bold",
        fontSize=15,
        leading=16,
        textColor="black",
        spaceAfter=2
    )

    contact_style = ParagraphStyle(
        "Contact",
        fontName="Helvetica",
        fontSize=8,
        leading=9,
        textColor="black",
        spaceAfter=4
    )

    section_style = ParagraphStyle(
        "Section",
        fontName="Helvetica-Bold",
        fontSize=9.5,
        leading=11,
        textColor="black",
        spaceBefore=4,
        spaceAfter=1
    )

    body_style = ParagraphStyle(
        "Body",
        fontName="Helvetica",
        fontSize=8.2,
        leading=9.7,
        textColor="black",
        spaceAfter=1.5
    )

    bullet_style = ParagraphStyle(
        "Bullet",
        fontName="Helvetica",
        fontSize=8.2,
        leading=9.6,
        leftIndent=9,
        firstLineIndent=-6,
        textColor="black",
        spaceAfter=1
    )

    job_style = ParagraphStyle(
        "Job",
        fontName="Helvetica-Bold",
        fontSize=8.4,
        leading=10,
        textColor="black",
        spaceAfter=1
    )

    # -----------------------------
    # Standard sections
    # -----------------------------

    sections = {
        "summary",
        "professional summary",
        "technical skills",
        "skills",
        "experience",
        "work experience",
        "professional experience",
        "projects",
        "education",
        "certifications",
        "achievements"
    }

    story = []

    lines = resume_text.splitlines()

    content_started = False

    for raw in lines:

        line = raw.strip()

        if not line:
            continue

        # Remove markdown formatting
        line = line.replace("**", "")
        line = line.replace("__", "")

        # Remove markdown heading symbols
        line = re.sub(r"^#+\s*", "", line)

        lower = line.lower().strip()

        # -----------------------------
        # Name
        # -----------------------------

        if not content_started:

            story.append(
                Paragraph(
                    line,
                    name_style
                )
            )

            content_started = True
            continue

        # -----------------------------
        # Section headings
        # -----------------------------

        if lower in sections:

            story.append(
                Paragraph(
                    lower.upper(),
                    section_style
                )
            )

            story.append(
                HRFlowable(
                    width="100%",
                    thickness=0.5,
                    color="black",
                    spaceBefore=0,
                    spaceAfter=2
                )
            )

            continue

        # -----------------------------
        # Bullets
        # -----------------------------

        if raw.strip().startswith((
            "-",
            "*",
            "•"
        )):

            clean = re.sub(
                r"^[\-\*•]\s*",
                "",
                line
            )

            story.append(
                Paragraph(
                    "• " + clean,
                    bullet_style
                )
            )

            continue

        # -----------------------------
        # Contact line
        # -----------------------------

        if (
            "@" in line
            or "linkedin" in lower
            or "github" in lower
            or re.search(r"\+?\d[\d\s\-()]{7,}", line)
        ):

            story.append(
                Paragraph(
                    line,
                    contact_style
                )
            )

            continue

        # -----------------------------
        # Likely job/project title
        # -----------------------------

        if (
            "|" in line
            or " — " in line
            or " - " in line
        ):

            story.append(
                Paragraph(
                    line,
                    job_style
                )
            )

            continue

        # -----------------------------
        # Normal text
        # -----------------------------

        story.append(
            Paragraph(
                line,
                body_style
            )
        )

    doc.build(story)

    return output_path


print("✅ Professional ATS PDF renderer ready")

In [ ]:
PROFESSIONAL_PDF = "/content/Final_ATS_Resume.pdf"

create_professional_ats_pdf(
    final_resume,
    PROFESSIONAL_PDF
)

print("✅ Professional ATS resume generated")
print(PROFESSIONAL_PDF)

In [ ]:
reader = PdfReader(PROFESSIONAL_PDF)

page_count = len(reader.pages)

extracted_text = ""

for page in reader.pages:
    extracted_text += page.extract_text() or ""

print("=" * 70)
print("PROFESSIONAL ATS PDF CHECK")
print("=" * 70)

print("Pages:", page_count)
print("Words:", len(extracted_text.split()))
print("Extracted characters:", len(extracted_text))

print("\nFormat:")
print("✓ Single column")
print("✓ Black text")
print("✓ Standard font")
print("✓ No tables")
print("✓ No graphics")
print("✓ ATS-readable")

if page_count == 1:
    print("\n🎯 ONE-PAGE PDF CONFIRMED")
else:
    print("\n⚠️ PDF exceeds one page")

In [ ]:
final_score = final_score_result["total_score"]

print("=" * 80)
print("🚀 RESUME OPTIMIZER — FINAL QUALITY CHECK")
print("=" * 80)

print(f"""
ATS Alignment       : {final_score}/100
Target              : 85–97
Pages               : {page_count}
Word Count          : {len(final_resume.split())}

ATS Structure       : Single Column
Font                : Standard
Color               : Black
Tables              : None
Graphics            : None
Icons               : None
Keyword Optimization: Enabled
JD Alignment        : Enabled
Fact Protection     : Enabled
One-Page Target     : Enabled
""")

if 85 <= final_score <= 97 and page_count == 1:
    print("🎯 ALL PRIMARY TARGETS ACHIEVED")
else:
    print("⚠️ One or more targets need improvement")

print("=" * 80)

In [ ]:
!pip install -q datasets

from datasets import Dataset
import json
import random
import os

print("✅ Dataset tools ready")

In [ ]:
roles = [

    # =========================
    # SOFTWARE / ENGINEERING
    # =========================

    "Software Engineer",
    "Software Developer",
    "Backend Developer",
    "Frontend Developer",
    "Full Stack Developer",
    "Web Developer",
    "Application Developer",
    "Systems Engineer",
    "Platform Engineer",
    "Integration Engineer",
    "API Developer",
    "Mobile App Developer",
    "Android Developer",
    "iOS Developer",
    "Embedded Software Engineer",
    "Firmware Engineer",
    "Game Developer",

    # =========================
    # DATA / AI
    # =========================

    "Data Analyst",
    "Data Scientist",
    "Data Engineer",
    "Business Intelligence Analyst",
    "BI Developer",
    "Machine Learning Engineer",
    "AI Engineer",
    "Deep Learning Engineer",
    "NLP Engineer",
    "Computer Vision Engineer",
    "AI Research Engineer",
    "Research Analyst",
    "Quantitative Analyst",

    # =========================
    # CLOUD / DEVOPS / IT
    # =========================

    "Cloud Engineer",
    "Cloud Architect",
    "DevOps Engineer",
    "Site Reliability Engineer",
    "Infrastructure Engineer",
    "Network Engineer",
    "Network Administrator",
    "System Administrator",
    "IT Support Specialist",
    "IT Analyst",
    "IT Consultant",
    "Technical Support Engineer",
    "Database Administrator",
    "Database Engineer",
    "Solutions Architect",

    # =========================
    # CYBERSECURITY
    # =========================

    "Cybersecurity Analyst",
    "Security Engineer",
    "Information Security Analyst",
    "SOC Analyst",
    "Security Consultant",
    "Cloud Security Engineer",
    "Application Security Engineer",
    "Cybersecurity Engineer",
    "Risk and Compliance Analyst",

    # =========================
    # TESTING / QA
    # =========================

    "QA Engineer",
    "Quality Assurance Analyst",
    "Software Test Engineer",
    "Automation Test Engineer",
    "QA Automation Engineer",
    "Performance Test Engineer",
    "Quality Analyst",

    # =========================
    # BUSINESS / ANALYTICS
    # =========================

    "Business Analyst",
    "Business Systems Analyst",
    "Business Intelligence Analyst",
    "Process Analyst",
    "Operations Analyst",
    "Strategy Analyst",
    "Management Analyst",
    "Product Analyst",
    "Financial Analyst",
    "Risk Analyst",
    "Investment Analyst",
    "Research Analyst",
    "Data Analyst",

    # =========================
    # PRODUCT / PROJECT
    # =========================

    "Product Manager",
    "Associate Product Manager",
    "Product Owner",
    "Project Manager",
    "Program Manager",
    "Technical Program Manager",
    "Project Coordinator",
    "PMO Analyst",
    "Scrum Master",
    "Agile Project Coordinator",

    # =========================
    # FINANCE / ACCOUNTING
    # =========================

    "Accountant",
    "Financial Analyst",
    "Financial Planning Analyst",
    "FP&A Analyst",
    "Investment Banking Analyst",
    "Credit Analyst",
    "Risk Analyst",
    "Audit Analyst",
    "Internal Auditor",
    "Tax Analyst",
    "Treasury Analyst",
    "Accounts Payable Specialist",
    "Accounts Receivable Specialist",
    "Finance Associate",

    # =========================
    # SALES
    # =========================

    "Sales Executive",
    "Sales Representative",
    "Business Development Executive",
    "Business Development Representative",
    "Business Development Manager",
    "Account Executive",
    "Account Manager",
    "Sales Development Representative",
    "Inside Sales Representative",
    "Sales Operations Analyst",
    "Pre-Sales Consultant",
    "Sales Engineer",

    # =========================
    # MARKETING
    # =========================

    "Marketing Analyst",
    "Digital Marketing Specialist",
    "Digital Marketing Executive",
    "Marketing Coordinator",
    "Marketing Manager",
    "Content Marketing Specialist",
    "SEO Specialist",
    "SEM Specialist",
    "Social Media Manager",
    "Social Media Specialist",
    "Brand Manager",
    "Growth Marketing Analyst",
    "Product Marketing Specialist",
    "Market Research Analyst",
    "Content Strategist",

    # =========================
    # HR / PEOPLE
    # =========================

    "Human Resources Associate",
    "HR Analyst",
    "HR Generalist",
    "HR Coordinator",
    "Talent Acquisition Specialist",
    "Recruiter",
    "Technical Recruiter",
    "Recruitment Coordinator",
    "Talent Management Specialist",
    "Learning and Development Specialist",
    "People Operations Specialist",
    "HR Operations Analyst",
    "Compensation Analyst",

    # =========================
    # OPERATIONS / SUPPLY CHAIN
    # =========================

    "Operations Analyst",
    "Operations Executive",
    "Operations Manager",
    "Supply Chain Analyst",
    "Supply Chain Coordinator",
    "Logistics Coordinator",
    "Logistics Analyst",
    "Procurement Analyst",
    "Procurement Specialist",
    "Inventory Analyst",
    "Demand Planning Analyst",
    "Supply Chain Planner",
    "Business Operations Analyst",

    # =========================
    # CONSULTING
    # =========================

    "Business Consultant",
    "Management Consultant",
    "Technology Consultant",
    "IT Consultant",
    "Strategy Consultant",
    "Risk Consultant",
    "Financial Consultant",
    "Business Transformation Consultant",
    "Digital Transformation Consultant",

    # =========================
    # DESIGN / CREATIVE
    # =========================

    "UI Designer",
    "UX Designer",
    "UX Researcher",
    "Product Designer",
    "UI/UX Designer",
    "Graphic Designer",
    "Visual Designer",
    "Interaction Designer",
    "Creative Designer",
    "Content Designer",

    # =========================
    # CONTENT / COMMUNICATION
    # =========================

    "Content Writer",
    "Technical Writer",
    "Copywriter",
    "Content Strategist",
    "Communications Specialist",
    "Corporate Communications Specialist",
    "Public Relations Specialist",
    "Content Editor",
    "Copy Editor",

    # =========================
    # CUSTOMER SUCCESS
    # =========================

    "Customer Success Associate",
    "Customer Success Manager",
    "Customer Support Specialist",
    "Customer Experience Analyst",
    "Client Success Specialist",
    "Client Relationship Manager",
    "Customer Operations Specialist",

    # =========================
    # LEGAL / COMPLIANCE
    # =========================

    "Legal Analyst",
    "Legal Associate",
    "Compliance Analyst",
    "Compliance Specialist",
    "Regulatory Analyst",
    "Contract Analyst",
    "Contract Specialist",
    "Privacy Analyst",

    # =========================
    # HEALTHCARE / LIFE SCIENCE
    # =========================

    "Healthcare Analyst",
    "Healthcare Data Analyst",
    "Clinical Research Associate",
    "Clinical Data Analyst",
    "Medical Research Assistant",
    "Pharmaceutical Analyst",
    "Regulatory Affairs Associate",

    # =========================
    # EDUCATION
    # =========================

    "Teaching Assistant",
    "Teacher",
    "Academic Coordinator",
    "Instructional Designer",
    "Education Program Coordinator",
    "Training Specialist",
    "Learning Specialist",

    # =========================
    # ADMIN / GENERAL
    # =========================

    "Administrative Assistant",
    "Executive Assistant",
    "Office Administrator",
    "Administrative Coordinator",
    "Operations Coordinator",
    "Program Coordinator",
    "Business Coordinator",
    "Executive Coordinator"
]

print("Total roles:", len(roles))

In [ ]:
skill_groups = {

    "languages": [
        "Python",
        "Java",
        "C++",
        "JavaScript",
        "TypeScript",
        "Go",
        "C#",
        "SQL"
    ],

    "frontend": [
        "React",
        "Angular",
        "Vue",
        "HTML",
        "CSS",
        "Next.js"
    ],

    "backend": [
        "Node.js",
        "Django",
        "Flask",
        "Spring Boot",
        "REST APIs",
        "GraphQL",
        "FastAPI"
    ],

    "databases": [
        "MySQL",
        "PostgreSQL",
        "MongoDB",
        "Redis",
        "SQL Server"
    ],

    "cloud": [
        "AWS",
        "Azure",
        "Google Cloud",
        "Docker",
        "Kubernetes"
    ],

    "ml": [
        "PyTorch",
        "TensorFlow",
        "scikit-learn",
        "Pandas",
        "NumPy",
        "Machine Learning",
        "Deep Learning"
    ],

    "tools": [
        "Git",
        "GitHub",
        "Jenkins",
        "Linux",
        "CI/CD"
    ]
}

for group, skills in skill_groups.items():
    print(group, ":", len(skills))

In [ ]:
project_templates = [
    "E-commerce Platform",
    "Task Management System",
    "Student Management System",
    "Inventory Management System",
    "Online Payment Simulator",
    "Resume Optimization System",
    "Healthcare Management Platform",
    "Banking Application",
    "Customer Analytics Dashboard",
    "Recommendation System",
    "Fraud Detection System",
    "Real-Time Chat Application",
    "Weather Application",
    "Job Portal",
    "Food Delivery Platform",
    "Expense Tracker",
    "Social Media Application",
    "Learning Management System",
    "Image Classification System",
    "Customer Churn Prediction"
]

print("Project types:", len(project_templates))

In [ ]:
def generate_candidate_profile(index):

    role = random.choice(roles)

    selected_languages = random.sample(
        skill_groups["languages"],
        k=random.randint(1, 3)
    )

    selected_tools = random.sample(
        skill_groups["tools"],
        k=random.randint(1, 3)
    )

    selected_database = random.choice(
        skill_groups["databases"]
    )

    project = random.choice(
        project_templates
    )

    project_skills = selected_languages + [
        selected_database
    ]

    return {
        "name": "[Candidate Name]",
        "role": role,
        "languages": selected_languages,
        "tools": selected_tools,
        "database": selected_database,
        "project": project,
        "project_skills": project_skills
    }


profiles = [
    generate_candidate_profile(i)
    for i in range(1000)
]

print("✅ Generated", len(profiles), "candidate profiles")

In [ ]:
responsibility_templates = [
    "develop scalable applications",
    "build and maintain backend services",
    "design REST APIs",
    "develop responsive user interfaces",
    "analyze datasets and generate insights",
    "build machine learning models",
    "automate deployment pipelines",
    "maintain cloud infrastructure",
    "write automated tests",
    "debug and improve software systems",
    "design database solutions",
    "collaborate with engineering teams"
]


def generate_jd(profile):

    role = profile["role"]

    required = profile["languages"]

    extra = random.sample(
        responsibility_templates,
        k=random.randint(3, 5)
    )

    return f"""
We are hiring a {role}.

The candidate will work with
{", ".join(required)}
and contribute to software development.

Responsibilities include:
{"; ".join(extra)}.

Strong problem-solving, communication and collaboration skills
are valued.
"""


job_descriptions = [
    generate_jd(profile)
    for profile in profiles
]

print("✅ Generated", len(job_descriptions), "job descriptions")

In [ ]:
def generate_resume(profile):

    skills = (
        profile["languages"]
        + profile["tools"]
        + [profile["database"]]
    )

    return f"""
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
{", ".join(skills)}

Projects

{profile["project"]}
Developed a software application using
{", ".join(profile["project_skills"])}.
Implemented application functionality and database integration.
Tested and debugged the application during development.

Experience

Software Development Intern
Worked on application development, debugging and testing.
"""


resumes = [
    generate_resume(profile)
    for profile in profiles
]

print("✅ Generated", len(resumes), "candidate resumes")

In [ ]:
for i in range(5):

    print("=" * 70)
    print("EXAMPLE", i + 1)
    print("=" * 70)

    print("ROLE:")
    print(profiles[i]["role"])

    print("\nJOB DESCRIPTION:")
    print(job_descriptions[i])

    print("\nRESUME:")
    print(resumes[i])

In [ ]:
role_categories = {

    "technology": [
        "Software Engineer",
        "Software Developer",
        "Backend Developer",
        "Frontend Developer",
        "Full Stack Developer",
        "Web Developer",
        "Application Developer",
        "Systems Engineer",
        "Platform Engineer",
        "Integration Engineer",
        "API Developer",
        "Mobile App Developer",
        "Android Developer",
        "iOS Developer",
        "Embedded Software Engineer",
        "Firmware Engineer",
        "Game Developer"
    ],

    "data_ai": [
        "Data Analyst",
        "Data Scientist",
        "Data Engineer",
        "Business Intelligence Analyst",
        "BI Developer",
        "Machine Learning Engineer",
        "AI Engineer",
        "Deep Learning Engineer",
        "NLP Engineer",
        "Computer Vision Engineer",
        "AI Research Engineer",
        "Research Analyst",
        "Quantitative Analyst"
    ],

    "it_cloud": [
        "Cloud Engineer",
        "Cloud Architect",
        "DevOps Engineer",
        "Site Reliability Engineer",
        "Infrastructure Engineer",
        "Network Engineer",
        "Network Administrator",
        "System Administrator",
        "IT Support Specialist",
        "IT Analyst",
        "IT Consultant",
        "Technical Support Engineer",
        "Database Administrator",
        "Database Engineer",
        "Solutions Architect"
    ],

    "cybersecurity": [
        "Cybersecurity Analyst",
        "Security Engineer",
        "Information Security Analyst",
        "SOC Analyst",
        "Security Consultant",
        "Cloud Security Engineer",
        "Application Security Engineer",
        "Cybersecurity Engineer",
        "Risk and Compliance Analyst"
    ],

    "business": [
        "Business Analyst",
        "Business Systems Analyst",
        "Process Analyst",
        "Operations Analyst",
        "Strategy Analyst",
        "Management Analyst",
        "Product Analyst",
        "Financial Analyst",
        "Risk Analyst",
        "Investment Analyst"
    ],

    "product_project": [
        "Product Manager",
        "Associate Product Manager",
        "Product Owner",
        "Project Manager",
        "Program Manager",
        "Technical Program Manager",
        "Project Coordinator",
        "PMO Analyst",
        "Scrum Master"
    ],

    "finance": [
        "Accountant",
        "Financial Analyst",
        "FP&A Analyst",
        "Investment Banking Analyst",
        "Credit Analyst",
        "Audit Analyst",
        "Internal Auditor",
        "Tax Analyst",
        "Treasury Analyst",
        "Finance Associate"
    ],

    "sales": [
        "Sales Executive",
        "Sales Representative",
        "Business Development Executive",
        "Business Development Representative",
        "Business Development Manager",
        "Account Executive",
        "Account Manager",
        "Sales Development Representative",
        "Sales Operations Analyst",
        "Pre-Sales Consultant",
        "Sales Engineer"
    ],

    "marketing": [
        "Marketing Analyst",
        "Digital Marketing Specialist",
        "Marketing Coordinator",
        "Marketing Manager",
        "Content Marketing Specialist",
        "SEO Specialist",
        "SEM Specialist",
        "Social Media Manager",
        "Brand Manager",
        "Growth Marketing Analyst",
        "Product Marketing Specialist",
        "Market Research Analyst"
    ],

    "hr": [
        "Human Resources Associate",
        "HR Analyst",
        "HR Generalist",
        "HR Coordinator",
        "Talent Acquisition Specialist",
        "Recruiter",
        "Technical Recruiter",
        "Talent Management Specialist",
        "Learning and Development Specialist",
        "People Operations Specialist"
    ],

    "operations": [
        "Operations Analyst",
        "Operations Executive",
        "Operations Manager",
        "Supply Chain Analyst",
        "Supply Chain Coordinator",
        "Logistics Coordinator",
        "Logistics Analyst",
        "Procurement Analyst",
        "Inventory Analyst",
        "Demand Planning Analyst",
        "Supply Chain Planner"
    ],

    "consulting": [
        "Business Consultant",
        "Management Consultant",
        "Technology Consultant",
        "IT Consultant",
        "Strategy Consultant",
        "Risk Consultant",
        "Financial Consultant",
        "Business Transformation Consultant",
        "Digital Transformation Consultant"
    ],

    "design": [
        "UI Designer",
        "UX Designer",
        "UX Researcher",
        "Product Designer",
        "UI/UX Designer",
        "Graphic Designer",
        "Visual Designer",
        "Interaction Designer",
        "Creative Designer",
        "Content Designer"
    ],

    "content": [
        "Content Writer",
        "Technical Writer",
        "Copywriter",
        "Content Strategist",
        "Communications Specialist",
        "Public Relations Specialist",
        "Content Editor",
        "Copy Editor"
    ],

    "customer": [
        "Customer Success Associate",
        "Customer Success Manager",
        "Customer Support Specialist",
        "Customer Experience Analyst",
        "Client Success Specialist",
        "Client Relationship Manager",
        "Customer Operations Specialist"
    ],

    "legal_compliance": [
        "Legal Analyst",
        "Legal Associate",
        "Compliance Analyst",
        "Compliance Specialist",
        "Regulatory Analyst",
        "Contract Analyst",
        "Contract Specialist",
        "Privacy Analyst"
    ],

    "healthcare": [
        "Healthcare Analyst",
        "Healthcare Data Analyst",
        "Clinical Research Associate",
        "Clinical Data Analyst",
        "Medical Research Assistant",
        "Pharmaceutical Analyst",
        "Regulatory Affairs Associate"
    ],

    "education": [
        "Teaching Assistant",
        "Teacher",
        "Academic Coordinator",
        "Instructional Designer",
        "Education Program Coordinator",
        "Training Specialist",
        "Learning Specialist"
    ],

    "administration": [
        "Administrative Assistant",
        "Executive Assistant",
        "Office Administrator",
        "Administrative Coordinator",
        "Operations Coordinator",
        "Program Coordinator",
        "Business Coordinator",
        "Executive Coordinator"
    ]
}

print("Role categories:", len(role_categories))

In [ ]:
role_skills = {

    # TECH
    "technology": [
        "Python", "Java", "C++", "JavaScript", "TypeScript",
        "React", "Angular", "Node.js", "Django", "Spring Boot",
        "REST APIs", "GraphQL", "Git", "Linux", "PostgreSQL"
    ],

    # DATA / AI
    "data_ai": [
        "Python", "SQL", "Pandas", "NumPy", "Power BI",
        "Tableau", "Machine Learning", "Deep Learning",
        "Statistics", "Data Visualization", "TensorFlow",
        "PyTorch", "scikit-learn"
    ],

    # BUSINESS ANALYSIS
    "business": [
        "Business Analysis", "Requirements Gathering",
        "Requirements Elicitation", "Stakeholder Management",
        "Process Mapping", "Business Process Analysis",
        "Gap Analysis", "Root Cause Analysis",
        "UAT", "User Stories", "Acceptance Criteria",
        "SQL", "Excel", "Power BI", "Tableau",
        "JIRA", "Confluence", "Agile", "Scrum"
    ],

    # PRODUCT / PROJECT
    "product_project": [
        "Product Strategy", "Product Roadmap", "Product Lifecycle",
        "Requirements Gathering", "User Stories", "Agile",
        "Scrum", "JIRA", "A/B Testing", "Product Analytics",
        "Stakeholder Management", "Project Planning",
        "Risk Management", "Cross-functional Collaboration"
    ],

    # FINANCE
    "finance": [
        "Financial Analysis", "Financial Modeling",
        "Forecasting", "Budgeting", "Financial Reporting",
        "Excel", "Power BI", "SQL", "Variance Analysis",
        "Accounting", "Accounts Payable", "Accounts Receivable",
        "Risk Analysis", "Financial Statements"
    ],

    # SALES
    "sales": [
        "Lead Generation", "Prospecting", "Sales Pipeline",
        "CRM", "Salesforce", "HubSpot", "Account Management",
        "Business Development", "Negotiation", "Client Relationship",
        "Sales Forecasting", "Revenue Growth", "B2B Sales"
    ],

    # MARKETING
    "marketing": [
        "Digital Marketing", "SEO", "SEM", "Google Analytics",
        "Content Marketing", "Social Media Marketing",
        "Market Research", "Campaign Management",
        "Email Marketing", "Lead Generation", "Brand Strategy",
        "Marketing Analytics", "Customer Segmentation"
    ],

    # HR
    "hr": [
        "Talent Acquisition", "Recruitment", "Employee Relations",
        "HR Operations", "Onboarding", "Performance Management",
        "HRIS", "Workday", "People Analytics",
        "Payroll", "Learning and Development",
        "Compensation", "Benefits Administration"
    ],

    # OPERATIONS
    "operations": [
        "Operations Management", "Process Improvement",
        "Supply Chain Management", "Inventory Management",
        "Logistics", "Procurement", "Demand Planning",
        "Forecasting", "Vendor Management", "Excel",
        "Power BI", "KPI Reporting", "Root Cause Analysis"
    ],

    # CONSULTING
    "consulting": [
        "Business Analysis", "Strategy", "Market Research",
        "Data Analysis", "Stakeholder Management",
        "Problem Solving", "Process Improvement",
        "Financial Analysis", "Presentation",
        "Client Management", "PowerPoint", "Excel"
    ],

    # DESIGN
    "design": [
        "Figma", "Adobe XD", "Photoshop", "Illustrator",
        "Wireframing", "Prototyping", "User Research",
        "Usability Testing", "Information Architecture",
        "Interaction Design", "Design Systems", "UI Design",
        "UX Design"
    ],

    # CONTENT
    "content": [
        "Content Writing", "Copywriting", "SEO",
        "Content Strategy", "Editing", "Proofreading",
        "Content Management", "WordPress", "Research",
        "Storytelling", "Technical Writing", "Social Media"
    ],

    # CUSTOMER SUCCESS
    "customer": [
        "Customer Success", "Customer Relationship Management",
        "Client Management", "Customer Retention",
        "Onboarding", "Customer Support", "CRM",
        "Salesforce", "Zendesk", "Problem Solving",
        "Communication", "Account Management"
    ],

    # CYBERSECURITY
    "cybersecurity": [
        "Network Security", "SIEM", "SOC",
        "Incident Response", "Threat Detection",
        "Vulnerability Assessment", "Penetration Testing",
        "Risk Assessment", "IAM", "Firewalls",
        "Linux", "Security Monitoring"
    ],

    # CLOUD / DEVOPS
    "it_cloud": [
        "AWS", "Azure", "Google Cloud", "Docker",
        "Kubernetes", "Terraform", "Jenkins",
        "CI/CD", "Linux", "Ansible",
        "Cloud Infrastructure", "Monitoring"
    ],

    # HEALTHCARE
    "healthcare": [
        "Clinical Research", "Clinical Data",
        "Healthcare Analytics", "Medical Documentation",
        "Regulatory Compliance", "Patient Data",
        "Healthcare Operations", "Research Methods",
        "Data Analysis", "Reporting"
    ],

    # EDUCATION
    "education": [
        "Curriculum Development", "Lesson Planning",
        "Instructional Design", "Learning Management Systems",
        "Student Engagement", "Assessment",
        "Training", "Educational Technology",
        "Classroom Management", "Content Development"
    ],

    # ADMIN
    "administration": [
        "Calendar Management", "Documentation",
        "Microsoft Office", "Excel", "Data Entry",
        "Scheduling", "Travel Coordination",
        "Office Administration", "Records Management",
        "Communication", "Event Coordination"
    ]
}

print("✅ Role-specific skill database created")

In [ ]:
def get_role_category(role):

    role_lower = role.lower()

    for category, category_roles in role_categories.items():

        for category_role in category_roles:

            if category_role.lower() == role_lower:
                return category

    # Fallback keyword matching

    if any(x in role_lower for x in [
        "engineer", "developer", "programmer"
    ]):
        return "technology"

    if any(x in role_lower for x in [
        "data", "machine learning", "ai"
    ]):
        return "data_ai"

    if "business analyst" in role_lower:
        return "business"

    if any(x in role_lower for x in [
        "marketing", "seo", "brand"
    ]):
        return "marketing"

    if any(x in role_lower for x in [
        "finance", "accountant", "financial"
    ]):
        return "finance"

    if any(x in role_lower for x in [
        "hr", "human resource", "recruit"
    ]):
        return "hr"

    if any(x in role_lower for x in [
        "sales", "business development"
    ]):
        return "sales"

    return "business"


# Test
for role in [
    "Software Engineer",
    "Business Analyst",
    "Marketing Analyst",
    "Financial Analyst",
    "HR Analyst"
]:
    print(role, "→", get_role_category(role))

In [ ]:
def generate_role_aware_profile():

    role = random.choice(roles)

    category = get_role_category(role)

    available_skills = role_skills.get(
        category,
        []
    )

    number_of_skills = min(
        random.randint(4, 8),
        len(available_skills)
    )

    selected_skills = random.sample(
        available_skills,
        number_of_skills
    )

    project = random.choice(
        project_templates
    )

    return {
        "role": role,
        "category": category,
        "skills": selected_skills,
        "project": project
    }


role_aware_profiles = [
    generate_role_aware_profile()
    for _ in range(3000)
]

print(
    "✅ Generated",
    len(role_aware_profiles),
    "role-aware profiles"
)

In [ ]:
from collections import Counter

distribution = Counter(
    profile["category"]
    for profile in role_aware_profiles
)

print("=" * 60)
print("TRAINING DATA DISTRIBUTION")
print("=" * 60)

for category, count in distribution.items():
    print(f"{category:20} : {count}")

In [ ]:
for i in range(10):

    profile = role_aware_profiles[i]

    print("=" * 70)
    print(f"Role     : {profile['role']}")
    print(f"Category : {profile['category']}")
    print(f"Skills   : {', '.join(profile['skills'])}")
    print(f"Project  : {profile['project']}")

In [ ]:
def create_realistic_profile(profile):

    role = profile["role"]
    category = profile["category"]
    skills = profile["skills"]
    project = profile["project"]

    # Pick 2–4 skills for the main project
    project_skills = random.sample(
        skills,
        min(len(skills), random.randint(2, 4))
    )

    # Pick 2–3 skills for experience
    experience_skills = random.sample(
        skills,
        min(len(skills), random.randint(2, 3))
    )

    resume = f"""
[Candidate Name]

EDUCATION
Bachelor's Degree

TECHNICAL SKILLS
{", ".join(skills)}

EXPERIENCE
{role} Intern
Worked on tasks related to {", ".join(experience_skills)}.
Supported team activities, documentation, testing, analysis,
and day-to-day project execution.

PROJECTS
{project}
Developed a project using {", ".join(project_skills)}.
Worked on implementation, testing, troubleshooting and
documentation.

CERTIFICATIONS
Relevant coursework and academic training.
"""

    return resume.strip()


realistic_resumes = [
    create_realistic_profile(profile)
    for profile in role_aware_profiles
]

print("✅ Created", len(realistic_resumes), "realistic resumes")

In [ ]:
def create_realistic_jd(profile):

    role = profile["role"]
    category = profile["category"]
    skills = profile["skills"]

    selected_skills = random.sample(
        skills,
        min(len(skills), random.randint(4, 7))
    )

    responsibilities = random.sample(
        responsibility_templates,
        min(len(responsibility_templates), 4)
    )

    jd = f"""
JOB TITLE: {role}

ABOUT THE ROLE

We are looking for a {role} to join our team and contribute
to business and project objectives.

REQUIRED SKILLS

{", ".join(selected_skills)}

RESPONSIBILITIES

- {responsibilities[0]}
- {responsibilities[1]}
- {responsibilities[2]}
- {responsibilities[3]}

The candidate should demonstrate strong problem-solving,
communication, collaboration and analytical abilities.
"""

    return jd.strip()


realistic_jds = [
    create_realistic_jd(profile)
    for profile in role_aware_profiles
]

print("✅ Created", len(realistic_jds), "realistic JDs")

In [ ]:
def generate_training_target(
    role,
    jd,
    resume
):

    prompt = f"""
You are creating a high-quality training example for an ATS
resume optimization model.

TARGET ROLE:
{role}

JOB DESCRIPTION:
{jd}

ORIGINAL RESUME:
{resume}

Create the ideal optimized resume.

Requirements:

- Strong JD alignment.
- Maximize relevant ATS keyword coverage.
- Use JD terminology when supported by the resume.
- Surface skills demonstrated in projects and experience.
- Prioritize important JD requirements.
- Rewrite weak bullets using concise professional language.
- Keep the candidate's actual facts unchanged.
- Do not fabricate skills, tools, experience, projects,
  achievements, metrics or certifications.
- Do not add unsupported JD keywords.
- Keep the resume concise and suitable for one page.
- Use standard ATS-friendly headings.
- Use a single-column structure.
- No tables.
- No graphics.
- No icons.
- No decorative formatting.
- Natural human-written language.

Return ONLY the optimized resume.
"""

    messages = [
        {
            "role": "system",
            "content": (
                "You create accurate ATS-optimized resumes "
                "without fabricating candidate information."
            )
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(inference_model.device)

    with torch.no_grad():

        outputs = inference_model.generate(
            **inputs,
            max_new_tokens=1000,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


print("✅ Training-target generator ready")

In [ ]:
training_targets = []

for i in range(20):

    print(f"Generating {i + 1}/20...")

    target = generate_training_target(
        role_aware_profiles[i]["role"],
        realistic_jds[i],
        realistic_resumes[i]
    )

    training_targets.append(target)

print("✅ Generated", len(training_targets), "training targets")

In [ ]:
for i in range(5):

    print("=" * 90)
    print("EXAMPLE", i + 1)
    print("=" * 90)

    print("ROLE:")
    print(role_aware_profiles[i]["role"])

    print("\nORIGINAL RESUME:")
    print(realistic_resumes[i])

    print("\nOPTIMIZED TRAINING TARGET:")
    print(training_targets[i])

In [ ]:
role_responsibilities = {

    "technology": [
        "Develop and maintain software applications",
        "Build and integrate REST APIs",
        "Debug and troubleshoot application issues",
        "Write clean and maintainable code",
        "Work with databases and backend services",
        "Use Git for version control",
        "Collaborate with engineering teams"
    ],

    "data_ai": [
        "Analyze structured and unstructured datasets",
        "Clean and preprocess data",
        "Build analytical models and reports",
        "Create dashboards and data visualizations",
        "Identify trends and actionable insights",
        "Evaluate model or analysis results",
        "Communicate findings to stakeholders"
    ],

    "business": [
        "Gather and document business requirements",
        "Analyze business processes and identify gaps",
        "Create user stories and acceptance criteria",
        "Work with stakeholders to clarify requirements",
        "Support user acceptance testing",
        "Perform root cause and impact analysis",
        "Document process improvements and recommendations"
    ],

    "product_project": [
        "Gather and prioritize product or project requirements",
        "Create user stories and acceptance criteria",
        "Track project milestones and deliverables",
        "Coordinate with cross-functional teams",
        "Identify and manage project risks",
        "Support Agile ceremonies and planning",
        "Communicate progress to stakeholders"
    ],

    "finance": [
        "Analyze financial data and business performance",
        "Prepare financial reports and analysis",
        "Support budgeting and forecasting activities",
        "Perform variance analysis",
        "Maintain financial models and spreadsheets",
        "Review financial statements",
        "Present financial insights to stakeholders"
    ],

    "sales": [
        "Generate and qualify business leads",
        "Manage customer and prospect interactions",
        "Maintain sales pipeline information",
        "Support business development activities",
        "Conduct product or service presentations",
        "Build and maintain client relationships",
        "Track sales performance and opportunities"
    ],

    "marketing": [
        "Support marketing campaigns and initiatives",
        "Conduct market and competitor research",
        "Analyze campaign performance",
        "Create and manage marketing content",
        "Support digital marketing activities",
        "Track customer and audience engagement",
        "Prepare marketing performance reports"
    ],

    "hr": [
        "Support recruitment and talent acquisition activities",
        "Coordinate employee onboarding processes",
        "Maintain employee records and documentation",
        "Support HR operations and employee queries",
        "Coordinate interviews and hiring activities",
        "Prepare HR reports and documentation",
        "Support employee engagement initiatives"
    ],

    "operations": [
        "Monitor operational processes and KPIs",
        "Identify process inefficiencies",
        "Support process improvement initiatives",
        "Prepare operational reports",
        "Coordinate with internal teams and vendors",
        "Analyze operational data",
        "Track workflows and deliverables"
    ],

    "consulting": [
        "Analyze client business problems",
        "Conduct market and business research",
        "Develop data-driven recommendations",
        "Analyze business processes",
        "Prepare client presentations and reports",
        "Collaborate with client stakeholders",
        "Support implementation of improvement initiatives"
    ],

    "design": [
        "Create user interfaces and design concepts",
        "Develop wireframes and interactive prototypes",
        "Conduct user research",
        "Perform usability testing",
        "Collaborate with product and engineering teams",
        "Maintain consistent design systems",
        "Incorporate user feedback into designs"
    ],

    "content": [
        "Research and create high-quality content",
        "Write and edit articles or marketing content",
        "Optimize content for target audiences",
        "Proofread and maintain content quality",
        "Conduct keyword and topic research",
        "Manage content publishing workflows",
        "Collaborate with marketing and editorial teams"
    ],

    "customer": [
        "Support customer onboarding and adoption",
        "Respond to customer questions and issues",
        "Maintain customer relationships",
        "Track customer satisfaction and engagement",
        "Identify opportunities to improve customer experience",
        "Document customer feedback",
        "Coordinate with internal teams to resolve issues"
    ],

    "cybersecurity": [
        "Monitor security events and alerts",
        "Investigate potential security incidents",
        "Perform vulnerability and risk assessments",
        "Support security monitoring activities",
        "Document security findings",
        "Assist with incident response",
        "Follow security policies and procedures"
    ],

    "it_cloud": [
        "Maintain cloud and IT infrastructure",
        "Monitor systems and infrastructure performance",
        "Support deployment and configuration activities",
        "Troubleshoot infrastructure issues",
        "Maintain CI/CD workflows",
        "Automate infrastructure and operational tasks",
        "Support system reliability and availability"
    ],

    "healthcare": [
        "Analyze healthcare or clinical data",
        "Maintain accurate research documentation",
        "Support healthcare reporting activities",
        "Conduct research and data analysis",
        "Follow regulatory and documentation requirements",
        "Prepare analytical reports",
        "Collaborate with healthcare and research teams"
    ],

    "education": [
        "Develop educational materials and content",
        "Support lesson or curriculum planning",
        "Assess learner progress",
        "Coordinate educational activities",
        "Use educational technology and learning platforms",
        "Support learner engagement",
        "Prepare educational reports and documentation"
    ],

    "administration": [
        "Coordinate administrative activities",
        "Maintain records and documentation",
        "Manage schedules and meetings",
        "Prepare reports and correspondence",
        "Coordinate internal communications",
        "Organize office or program activities",
        "Support day-to-day administrative operations"
    ]
}

In [ ]:
import random

def create_realistic_jd(profile):

    role = profile["role"]
    category = get_role_category(role)

    skills = profile["skills"]

    responsibilities = random.sample(
        role_responsibilities[category],
        min(5, len(role_responsibilities[category]))
    )

    jd = f"""
Job Title: {role}

We are looking for a {role} to join our team.

Responsibilities:
{chr(10).join("- " + x for x in responsibilities)}

Required Skills:
{", ".join(skills)}

The ideal candidate should have strong problem-solving,
communication and collaboration skills.
"""

    return jd.strip()

In [ ]:
def create_realistic_profile(profile):

    role = profile["role"]
    category = get_role_category(role)

    skills = profile["skills"]

    # Split skills into demonstrated and listed skills
    demonstrated_count = max(2, int(len(skills) * 0.6))

    demonstrated_skills = random.sample(
        skills,
        min(demonstrated_count, len(skills))
    )

    listed_only_skills = [
        s for s in skills
        if s not in demonstrated_skills
    ]

    responsibilities = random.sample(
        role_responsibilities[category],
        min(3, len(role_responsibilities[category]))
    )

    project_name = f"{role} Practical Project"

    project = f"""
{project_name}
- Built a practical project related to {role}.
- Applied {", ".join(demonstrated_skills[:3])}.
- Worked on problem solving, testing and documentation.
"""

    experience = f"""
{role} Intern
- Supported {responsibilities[0].lower()}.
- {responsibilities[1]}.
- {responsibilities[2]}.
"""

    resume = f"""
[Candidate Name]

Education
B.Tech / Bachelor's Degree

Technical Skills
{", ".join(skills)}

Projects
{project.strip()}

Experience
{experience.strip()}
"""

    return {
        **profile,
        "demonstrated_skills": demonstrated_skills,
        "listed_only_skills": listed_only_skills,
        "project": project.strip(),
        "experience": experience.strip(),
        "resume": resume.strip()
    }

In [ ]:
def add_unsupported_jd_skills(jd, candidate_skills, all_role_skills):

    unsupported_pool = [
        skill for skill in all_role_skills
        if skill not in candidate_skills
    ]

    if not unsupported_pool:
        return jd

    unsupported = random.sample(
        unsupported_pool,
        min(2, len(unsupported_pool))
    )

    jd += f"""

Preferred Skills:
{", ".join(unsupported)}
"""

    return jd

In [ ]:
def generate_training_pair():

    profile = random.choice(role_aware_profiles)

    profile = create_realistic_profile(profile)

    jd = create_realistic_jd(profile)

    category = get_role_category(profile["role"])

    jd = add_unsupported_jd_skills(
        jd,
        profile["skills"],
        role_skills[category]
    )

    return {
        "role": profile["role"],
        "category": category,
        "jd": jd,
        "resume": profile["resume"],
        "demonstrated_skills": profile["demonstrated_skills"],
        "listed_only_skills": profile["listed_only_skills"]
    }

sample = generate_training_pair()

print("ROLE:")
print(sample["role"])

print("\nJD:")
print(sample["jd"])

print("\nRESUME:")
print(sample["resume"])

print("\nDEMONSTRATED SKILLS:")
print(sample["demonstrated_skills"])

In [ ]:
test_pairs = [
    generate_training_pair()
    for _ in range(100)
]

print("Generated:", len(test_pairs))

for i in range(3):
    print("\n" + "="*80)
    print("SAMPLE", i + 1)
    print("="*80)
    print(test_pairs[i]["role"])
    print(test_pairs[i]["jd"])
    print("\nRESUME:")
    print(test_pairs[i]["resume"])

In [ ]:
import re

def normalize_text(text):
    return re.sub(r"[^a-z0-9+#.]", " ", text.lower())


def skill_in_text(skill, text):
    text_norm = normalize_text(text)
    skill_norm = normalize_text(skill)

    return skill_norm in text_norm


def validate_training_pair(pair):

    jd = pair["jd"]
    resume = pair["resume"]

    issues = []

    # --------------------------------------------------
    # 1. Basic completeness
    # --------------------------------------------------
    if not jd.strip():
        issues.append("empty_jd")

    if not resume.strip():
        issues.append("empty_resume")

    # --------------------------------------------------
    # 2. Role must exist
    # --------------------------------------------------
    if pair["role"].lower() not in jd.lower():
        issues.append("role_missing_from_jd")

    # --------------------------------------------------
    # 3. Resume should contain candidate name placeholder
    # --------------------------------------------------
    if "[Candidate Name]" not in resume:
        issues.append("candidate_name_missing")

    # --------------------------------------------------
    # 4. Resume should contain basic sections
    # --------------------------------------------------
    required_sections = [
        "Education",
        "Technical Skills",
        "Projects",
        "Experience"
    ]

    for section in required_sections:
        if section.lower() not in resume.lower():
            issues.append(f"missing_{section.lower().replace(' ', '_')}")

    # --------------------------------------------------
    # 5. Check demonstrated skills actually appear
    # --------------------------------------------------
    for skill in pair["demonstrated_skills"]:

        if not skill_in_text(skill, resume):
            issues.append(
                f"demonstrated_skill_missing:{skill}"
            )

    # --------------------------------------------------
    # 6. JD should not be ridiculously short
    # --------------------------------------------------
    if len(jd.split()) < 40:
        issues.append("jd_too_short")

    # --------------------------------------------------
    # 7. Resume should have reasonable length
    # --------------------------------------------------
    resume_words = len(resume.split())

    if resume_words < 50:
        issues.append("resume_too_short")

    if resume_words > 500:
        issues.append("resume_too_long")

    # --------------------------------------------------
    # 8. Detect obvious generic software contamination
    # --------------------------------------------------
    non_tech_roles = [
        "business analyst",
        "financial analyst",
        "accountant",
        "marketing analyst",
        "hr associate",
        "recruiter",
        "operations analyst",
        "content writer",
        "graphic designer"
    ]

    tech_phrases = [
        "scalable backend",
        "microservices architecture",
        "rest api",
        "software application",
        "source code"
    ]

    role_lower = pair["role"].lower()

    if any(r in role_lower for r in non_tech_roles):

        contamination_count = sum(
            phrase in resume.lower()
            for phrase in tech_phrases
        )

        if contamination_count >= 3:
            issues.append("possible_role_contamination")

    # --------------------------------------------------
    # Final decision
    # --------------------------------------------------
    valid = len(issues) == 0

    return {
        "valid": valid,
        "issues": issues,
        "word_count": resume_words
    }

In [ ]:
validation_results = []

for pair in test_pairs:

    result = validate_training_pair(pair)

    validation_results.append({
        "role": pair["role"],
        "category": pair["category"],
        "valid": result["valid"],
        "issues": result["issues"],
        "word_count": result["word_count"]
    })

valid_count = sum(
    x["valid"] for x in validation_results
)

invalid_count = len(validation_results) - valid_count

print("Total samples:", len(validation_results))
print("Valid:", valid_count)
print("Invalid:", invalid_count)
print(
    "Quality rate:",
    round(valid_count / len(validation_results) * 100, 2),
    "%"
)

In [ ]:
from collections import Counter

all_issues = []

for result in validation_results:
    all_issues.extend(result["issues"])

issue_counts = Counter(all_issues)

print("ISSUE FREQUENCY")
print("=" * 50)

for issue, count in issue_counts.most_common():
    print(f"{issue}: {count}")

In [ ]:
clean_pairs = []

for pair, result in zip(test_pairs, validation_results):

    if result["valid"]:
        clean_pairs.append(pair)

print("Original samples:", len(test_pairs))
print("Clean samples:", len(clean_pairs))

In [ ]:
for i, pair in enumerate(clean_pairs[:5]):

    print("\n" + "=" * 90)
    print(f"CLEAN SAMPLE {i + 1}")
    print("=" * 90)

    print("\nROLE:")
    print(pair["role"])

    print("\nJD:")
    print(pair["jd"])

    print("\nRESUME:")
    print(pair["resume"])

    print("\nDEMONSTRATED:")
    print(pair["demonstrated_skills"])

    print("\nLISTED ONLY:")
    print(pair["listed_only_skills"])

In [ ]:
def extract_claimed_skills(resume, candidate_data):

    all_candidate_skills = set()

    for key, values in candidate_data.items():

        if isinstance(values, list):
            for value in values:
                if isinstance(value, str):
                    all_candidate_skills.add(value.strip())

    return list(all_candidate_skills)

In [ ]:
def detect_hallucinated_skills(
    original_resume,
    optimized_resume,
    jd_data
):

    candidate_data = extract_candidate_evidence(
        original_resume
    )

    candidate_skills = extract_claimed_skills(
        original_resume,
        candidate_data
    )

    unsupported = []

    jd_skill_categories = [
        "technical_skills",
        "programming_languages",
        "frameworks",
        "libraries",
        "tools",
        "databases",
        "cloud",
        "devops",
        "apis",
        "architecture",
        "methodologies"
    ]

    for category in jd_skill_categories:

        for skill in jd_data.get(category, []):

            if not skill:
                continue

            supported = any(
                skill_in_text(skill, original_resume)
                for _ in [0]
            )

            appears_in_optimized = skill_in_text(
                skill,
                optimized_resume
            )

            if appears_in_optimized and not supported:
                unsupported.append(skill)

    return list(set(unsupported))

In [ ]:
test_jd_data = extract_rich_jd_data(
    comparison_jd
)

optimized_test = high_coverage_optimizer(
    comparison_jd,
    comparison_resume
)

hallucinated = detect_hallucinated_skills(
    comparison_resume,
    optimized_test,
    test_jd_data
)

print("Unsupported skills detected:")
print(hallucinated)

In [ ]:
def classify_skill_support(
    skill,
    original_resume,
    jd
):

    in_resume = skill_in_text(
        skill,
        original_resume
    )

    in_jd = skill_in_text(
        skill,
        jd
    )

    if in_resume:
        return "SUPPORTED"

    if in_jd:
        return "JD_ONLY"

    return "NOT_FOUND"

In [ ]:
def validate_optimized_resume(
    original_resume,
    jd,
    optimized_resume
):

    jd_data = extract_rich_jd_data(jd)

    hallucinated = detect_hallucinated_skills(
        original_resume,
        optimized_resume,
        jd_data
    )

    issues = []

    if hallucinated:
        issues.append({
            "type": "HALLUCINATED_SKILLS",
            "skills": hallucinated
        })

    # Basic fabrication patterns
    fabrication_patterns = [
        r"\b\d+%\b",
        r"\b\d+\+?\s*(years|months)\b",
        r"\bincreased .* by\b",
        r"\breduced .* by\b",
        r"\bsaved .* hours\b",
        r"\bmanaged \$[\d,.]+",
    ]

    for pattern in fabrication_patterns:

        if re.search(
            pattern,
            optimized_resume,
            re.IGNORECASE
        ):
            if not re.search(
                pattern,
                original_resume,
                re.IGNORECASE
            ):
                issues.append({
                    "type": "POSSIBLE_FABRICATED_METRIC",
                    "pattern": pattern
                })

    return {
        "valid": len(issues) == 0,
        "issues": issues,
        "hallucinated_skills": hallucinated
    }

In [ ]:
validation = validate_optimized_resume(
    comparison_resume,
    comparison_jd,
    optimized_test
)

print("VALID:", validation["valid"])
print("\nISSUES:")

for issue in validation["issues"]:
    print(issue)

In [ ]:
def generate_valid_resumes(
    company,
    role,
    job_description,
    existing_resume,
    num_versions=3
):

    candidates = []

    jd_data = extract_rich_jd_data(job_description)

    for i in range(num_versions):

        print(f"\nGenerating version {i+1}/{num_versions}...")

        resume = high_coverage_optimizer(
            company,
            role,
            job_description,
            existing_resume
        )

        # ATS score
        score_result = weighted_ats_score(
            jd_data,
            resume
        )

        # Hallucination validation
        validation = validate_optimized_resume(
            existing_resume,
            job_description,
            resume
        )

        candidates.append({
            "version": i + 1,
            "resume": resume,
            "ats_score": score_result["total_score"],
            "valid": validation["valid"],
            "issues": validation["issues"],
            "hallucinated_skills": validation["hallucinated_skills"]
        })

        print("ATS Score:", score_result["total_score"])
        print("Valid:", validation["valid"])

        if validation["hallucinated_skills"]:
            print(
                "Hallucinated:",
                validation["hallucinated_skills"]
            )

    return candidates


print("Final candidate generator ready")

In [ ]:
def select_best_resume(candidates):

    # First prefer resumes with no hallucinations
    valid_candidates = [
        c for c in candidates
        if c["valid"]
    ]

    if valid_candidates:

        # Prefer 85–97 range
        target_candidates = [
            c for c in valid_candidates
            if 85 <= c["ats_score"] <= 97
        ]

        if target_candidates:

            best = max(
                target_candidates,
                key=lambda x: x["ats_score"]
            )

        else:

            best = max(
                valid_candidates,
                key=lambda x: x["ats_score"]
            )

    else:

        # If all fail safety checks,
        # choose highest score but flag it
        best = max(
            candidates,
            key=lambda x: x["ats_score"]
        )

    return best


print("Best resume selector ready")

In [ ]:
candidates = generate_valid_resumes(
    company="Microsoft",
    role="Backend Software Engineer",
    job_description=comparison_jd,
    existing_resume=comparison_resume,
    num_versions=3
)

best = select_best_resume(candidates)

print("\n" + "=" * 90)
print("FINAL SELECTED RESUME")
print("=" * 90)

print("Version:", best["version"])
print("ATS Score:", best["ats_score"])
print("Safety Valid:", best["valid"])
print("Issues:", best["issues"])

print("\nRESUME:\n")
print(best["resume"])

In [ ]:
print("=" * 70)
print("ALL GENERATED VERSIONS")
print("=" * 70)

for candidate in candidates:

    print(
        f"Version {candidate['version']} | "
        f"ATS: {candidate['ats_score']}/100 | "
        f"Valid: {candidate['valid']}"
    )

    if candidate["hallucinated_skills"]:
        print(
            "  Unsupported skills:",
            candidate["hallucinated_skills"]
        )

In [ ]:
import json
import random

def create_training_example(pair, optimized_resume):

    system_prompt = """
You are an expert resume optimization assistant.

Optimize an existing resume for a specific job description.

Rules:
1. Preserve the candidate's factual background.
2. Never invent skills, experience, projects, certifications, employers,
   education, metrics, or achievements.
3. A JD skill may be included only when supported by the candidate's
   existing resume or clearly demonstrated experience/project evidence.
4. Use the job description's terminology when it truthfully matches
   candidate evidence.
5. Prioritize relevant skills, responsibilities and keywords.
6. Keep the resume ATS-friendly and concise.
7. Produce a professional one-page resume.
8. Use natural, human wording.
"""

    user_prompt = f"""
Company:
{pair.get("company", "Target Company")}

Target Role:
{pair["role"]}

Job Description:
{pair["jd"]}

Existing Resume:
{pair["resume"]}

Optimize the existing resume while following all rules.
"""

    return {
        "messages": [
            {
                "role": "system",
                "content": system_prompt.strip()
            },
            {
                "role": "user",
                "content": user_prompt.strip()
            },
            {
                "role": "assistant",
                "content": optimized_resume.strip()
            }
        ]
    }

In [ ]:
final_generated_examples = []

for pair, candidate in zip(test_pairs, candidates):

    # This part is only for the current test batch.
    if candidate["valid"]:

        example = create_training_example(
            pair,
            candidate["resume"]
        )

        final_generated_examples.append(example)

print(
    "Validated training examples:",
    len(final_generated_examples)
)

In [ ]:
def generate_training_dataset(
    profiles,
    examples_per_batch=20
):

    training_examples = []

    for start in range(
        0,
        len(profiles),
        examples_per_batch
    ):

        batch = profiles[
            start:start + examples_per_batch
        ]

        print(
            f"\nProcessing "
            f"{start + 1}-{start + len(batch)} "
            f"of {len(profiles)}"
        )

        for profile in batch:

            pair = create_realistic_profile(profile)

            jd = create_realistic_jd(pair)

            category = get_role_category(
                pair["role"]
            )

            jd = add_unsupported_jd_skills(
                jd,
                pair["skills"],
                role_skills[category]
            )

            try:

                optimized = high_coverage_optimizer(
                    "Target Company",
                    pair["role"],
                    jd,
                    pair["resume"]
                )

                validation = validate_optimized_resume(
                    pair["resume"],
                    jd,
                    optimized
                )

                if validation["valid"]:

                    training_examples.append(
                        create_training_example(
                            {
                                "role": pair["role"],
                                "category": category,
                                "jd": jd,
                                "resume": pair["resume"]
                            },
                            optimized
                        )
                    )

            except Exception as e:

                print("Skipped example:", e)

    return training_examples

In [ ]:
training_examples = generate_training_dataset(
    role_aware_profiles[:500],
    examples_per_batch=20
)

print("\nFINAL GENERATED:", len(training_examples))

In [ ]:
def remove_duplicate_examples(examples):

    seen = set()
    unique = []

    for example in examples:

        text = json.dumps(
            example,
            sort_keys=True
        )

        key = hash(text)

        if key not in seen:

            seen.add(key)
            unique.append(example)

    return unique


training_examples = remove_duplicate_examples(
    training_examples
)

print(
    "Unique examples:",
    len(training_examples)
)

In [ ]:
random.seed(42)

random.shuffle(training_examples)

print("Dataset shuffled.")

In [ ]:
total = len(training_examples)

train_end = int(total * 0.80)
val_end = int(total * 0.90)

train_data = training_examples[:train_end]
validation_data = training_examples[
    train_end:val_end
]
test_data = training_examples[val_end:]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

In [ ]:
def save_jsonl(data, path):

    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        for item in data:

            f.write(
                json.dumps(
                    item,
                    ensure_ascii=False
                ) + "\n"
            )


save_jsonl(
    train_data,
    "/content/resume_train_new.jsonl"
)

save_jsonl(
    validation_data,
    "/content/resume_validation_new.jsonl"
)

save_jsonl(
    test_data,
    "/content/resume_test_new.jsonl"
)

print("Saved new datasets.")

In [ ]:
print(
    json.dumps(
        training_examples[0],
        indent=2,
        ensure_ascii=False
    )
)

In [ ]:
import json
import os

original_path = "/content/resume_training_v4.jsonl"

original_examples = []

with open(
    original_path,
    "r",
    encoding="utf-8"
) as f:

    for line in f:
        line = line.strip()

        if line:
            original_examples.append(
                json.loads(line)
            )

print("Original examples:", len(original_examples))

In [ ]:
combined_examples = (
    original_examples +
    training_examples
)

print(
    "Original:",
    len(original_examples)
)

print(
    "New:",
    len(training_examples)
)

print(
    "Combined:",
    len(combined_examples)
)

In [ ]:
combined_examples = remove_duplicate_examples(
    combined_examples
)

print(
    "After duplicate removal:",
    len(combined_examples)
)

In [ ]:
random.seed(42)

random.shuffle(combined_examples)

print("Combined dataset shuffled.")

In [ ]:
total = len(combined_examples)

train_end = int(total * 0.80)
val_end = int(total * 0.90)

final_train = combined_examples[:train_end]

final_validation = combined_examples[
    train_end:val_end
]

final_test = combined_examples[
    val_end:
]

print("Total:", total)
print("Train:", len(final_train))
print("Validation:", len(final_validation))
print("Test:", len(final_test))

In [ ]:
save_jsonl(
    final_train,
    "/content/resume_train_final.jsonl"
)

save_jsonl(
    final_validation,
    "/content/resume_validation_final.jsonl"
)

save_jsonl(
    final_test,
    "/content/resume_test_final.jsonl"
)

print("Final dataset saved successfully.")

In [ ]:
def verify_jsonl(path):

    count = 0

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for line_number, line in enumerate(f, 1):

            try:
                item = json.loads(line)

                assert "messages" in item
                assert len(item["messages"]) == 3

                roles = [
                    x["role"]
                    for x in item["messages"]
                ]

                assert roles == [
                    "system",
                    "user",
                    "assistant"
                ]

                count += 1

            except Exception as e:

                print(
                    f"Invalid line {line_number}:",
                    e
                )

    return count


print(
    "Train verified:",
    verify_jsonl(
        "/content/resume_train_final.jsonl"
    )
)

print(
    "Validation verified:",
    verify_jsonl(
        "/content/resume_validation_final.jsonl"
    )
)

print(
    "Test verified:",
    verify_jsonl(
        "/content/resume_test_final.jsonl"
    )
)

In [ ]:
from collections import Counter

role_counter = Counter()

for example in combined_examples:

    user_text = example["messages"][1]["content"]

    match = re.search(
        r"Target Role:\s*(.+)",
        user_text
    )

    if match:
        role_counter[
            match.group(1).strip()
        ] += 1

print("ROLE DISTRIBUTION")
print("=" * 50)

for role, count in role_counter.most_common():
    print(f"{role}: {count}")

In [ ]:
category_counter = Counter()

for example in combined_examples:

    user_text = example["messages"][1]["content"]

    for role, category in role_categories.items():

        if f"Target Role:\n{role}" in user_text:
            category_counter[category] += 1
            break

print("CATEGORY DISTRIBUTION")
print("=" * 50)

for category, count in category_counter.most_common():
    print(f"{category}: {count}")

In [ ]:
print("=" * 70)
print("FINAL RESUME OPTIMIZER DATASET")
print("=" * 70)

print("Total examples      :", len(combined_examples))
print("Training examples   :", len(final_train))
print("Validation examples :", len(final_validation))
print("Test examples       :", len(final_test))

print("\nRoles covered:", len(role_counter))
print("Categories covered:", len(category_counter))

In [ ]:
import re

def normalize_keyword(text):
    text = str(text).lower().strip()

    # Normalize common variations
    text = text.replace("&", " and ")
    text = re.sub(r"[/_-]", " ", text)
    text = re.sub(r"[^a-z0-9+#. ]", " ", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def keyword_present(keyword, resume):
    return normalize_keyword(keyword) in normalize_keyword(resume)

In [ ]:
def get_jd_keyword_categories(jd_data):

    categories = {
        "technical_skills": jd_data.get(
            "technical_skills", []
        ),

        "programming_languages": jd_data.get(
            "programming_languages", []
        ),

        "frameworks": jd_data.get(
            "frameworks", []
        ),

        "libraries": jd_data.get(
            "libraries", []
        ),

        "tools": jd_data.get(
            "tools", []
        ),

        "databases": jd_data.get(
            "databases", []
        ),

        "cloud": jd_data.get(
            "cloud", []
        ),

        "devops": jd_data.get(
            "devops", []
        ),

        "apis": jd_data.get(
            "apis", []
        ),

        "architecture": jd_data.get(
            "architecture", []
        ),

        "methodologies": jd_data.get(
            "methodologies", []
        ),

        "responsibilities": jd_data.get(
            "responsibilities", []
        ),

        "domain_terms": jd_data.get(
            "domain_terms", []
        ),

        "soft_skills": jd_data.get(
            "soft_skills", []
        ),

        "important_phrases": jd_data.get(
            "important_phrases", []
        )
    }

    return categories

In [ ]:
ATS_WEIGHTS = {

    "technical_skills": 15,

    "programming_languages": 8,

    "frameworks": 7,

    "libraries": 5,

    "tools": 7,

    "databases": 5,

    "cloud": 5,

    "devops": 5,

    "apis": 4,

    "architecture": 4,

    "methodologies": 4,

    "responsibilities": 15,

    "domain_terms": 6,

    "soft_skills": 3,

    "important_phrases": 2
}

print("Total weight:", sum(ATS_WEIGHTS.values()))

In [ ]:
def calculate_category_score(keywords, resume):

    if not keywords:
        return 100.0, [], []

    matched = []
    missing = []

    for keyword in keywords:

        if keyword_present(
            keyword,
            resume
        ):
            matched.append(keyword)
        else:
            missing.append(keyword)

    score = (
        len(matched) /
        len(keywords)
    ) * 100

    return score, matched, missing

In [ ]:
def advanced_ats_score(jd_data, resume):

    categories = get_jd_keyword_categories(
        jd_data
    )

    results = {}

    total_score = 0

    for category, keywords in categories.items():

        category_score, matched, missing = (
            calculate_category_score(
                keywords,
                resume
            )
        )

        weight = ATS_WEIGHTS[category]

        weighted_score = (
            category_score *
            weight /
            100
        )

        total_score += weighted_score

        results[category] = {
            "score": round(category_score, 2),
            "weight": weight,
            "weighted_score": round(
                weighted_score,
                2
            ),
            "matched": matched,
            "missing": missing
        }

    # ---------------------------------------------
    # Structure score
    # ---------------------------------------------

    structure_checks = {
        "education": "education",
        "skills": "skills",
        "projects": "projects",
        "experience": "experience"
    }

    structure_points = 0

    for section in structure_checks.values():

        if section in resume.lower():
            structure_points += 2.5

    # Structure is capped at 10
    structure_points = min(
        structure_points,
        10
    )

    # IMPORTANT:
    # Reduce keyword score proportionally so
    # structure contributes to the final 100.
    keyword_score = total_score * 0.90

    final_score = (
        keyword_score +
        structure_points
    )

    return {
        "total_score": round(
            min(final_score, 100),
            2
        ),
        "keyword_score": round(
            keyword_score,
            2
        ),
        "structure_score": round(
            structure_points,
            2
        ),
        "categories": results
    }

In [ ]:
jd_data = extract_rich_jd_data(
    comparison_jd
)

score = advanced_ats_score(
    jd_data,
    comparison_resume
)

print(
    "ATS ALIGNMENT SCORE:",
    score["total_score"]
)

print(
    "\nKeyword score:",
    score["keyword_score"]
)

print(
    "Structure score:",
    score["structure_score"]
)

In [ ]:
for category, result in score["categories"].items():

    if result["matched"] or result["missing"]:

        print("\n" + "=" * 60)
        print(category.upper())

        print(
            "Score:",
            result["score"]
        )

        print(
            "Matched:",
            result["matched"]
        )

        print(
            "Missing:",
            result["missing"]
        )

In [ ]:
def print_ats_report(jd_data, resume):

    result = advanced_ats_score(
        jd_data,
        resume
    )

    print("=" * 70)
    print("ATS ALIGNMENT REPORT")
    print("=" * 70)

    print(
        f"\nOverall Score: "
        f"{result['total_score']}/100"
    )

    for category, data in result[
        "categories"
    ].items():

        if not data["matched"] and not data["missing"]:
            continue

        print(
            f"\n{category}: "
            f"{data['score']:.1f}%"
        )

        if data["matched"]:
            print(
                "  ✓ Matched:",
                ", ".join(data["matched"])
            )

        if data["missing"]:
            print(
                "  ✗ Missing:",
                ", ".join(data["missing"])
            )

    return result

In [ ]:
optimized_score = print_ats_report(
    jd_data,
    optimized_test
)

In [ ]:
def generate_with_model(
    model,
    company,
    role,
    jd,
    resume,
    max_new_tokens=1200
):

    prompt = f"""
You are an expert resume optimization assistant.

Company:
{company}

Target Role:
{role}

Job Description:
{jd}

Existing Resume:
{resume}

Optimize the existing resume.

Rules:
- Do not invent skills, experience, projects, certifications,
  employers, education or metrics.
- Use JD terminology only when supported by the existing resume.
- Preserve truthful candidate evidence.
- Prioritize relevant keywords and responsibilities.
- Keep the resume concise and ATS-friendly.
- Produce a professional one-page resume.
"""

    messages = [
        {
            "role": "system",
            "content": "You are an expert resume optimizer."
        },
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = outputs[
        0,
        inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()

In [ ]:
benchmark_jd = """
We are hiring a Backend Software Engineer to develop reliable
backend services and APIs.

Responsibilities:
- Develop backend services
- Build REST APIs
- Work with SQL databases
- Debug application issues
- Collaborate with engineering teams
- Use Git and software development best practices

Required Skills:
Python, SQL, Git, REST APIs
"""

benchmark_resume = """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects
Task Management API
- Built a backend application with HTTP endpoints.
- Implemented authentication and authorization.
- Connected the application to a SQL database.
- Used Git for source-code management.

Experience
Software Development Intern
- Worked on application development and debugging.
"""

In [ ]:
print("Generating BASE MODEL resume...")

base_resume = generate_with_model(
    base_eval_model,
    "Target Company",
    "Backend Software Engineer",
    benchmark_jd,
    benchmark_resume
)

print("\nBASE MODEL OUTPUT\n")
print(base_resume)

print("\n" + "=" * 80)

print("\nGenerating LoRA resume...")

lora_resume = generate_with_model(
    inference_model,
    "Target Company",
    "Backend Software Engineer",
    benchmark_jd,
    benchmark_resume
)

print("\nLORA OUTPUT\n")
print(lora_resume)

In [ ]:
benchmark_jd_data = extract_rich_jd_data(
    benchmark_jd
)

base_score = advanced_ats_score(
    benchmark_jd_data,
    base_resume
)

lora_score = advanced_ats_score(
    benchmark_jd_data,
    lora_resume
)

print("=" * 70)
print("MODEL COMPARISON")
print("=" * 70)

print(
    f"Base Model : {base_score['total_score']}/100"
)

print(
    f"LoRA Model : {lora_score['total_score']}/100"
)

print(
    f"Improvement: "
    f"{lora_score['total_score'] - base_score['total_score']:+.2f}"
)

In [ ]:
base_safety = validate_optimized_resume(
    benchmark_resume,
    benchmark_jd,
    base_resume
)

lora_safety = validate_optimized_resume(
    benchmark_resume,
    benchmark_jd,
    lora_resume
)

print("=" * 70)
print("SAFETY COMPARISON")
print("=" * 70)

print("\nBASE MODEL")
print("Valid:", base_safety["valid"])
print(
    "Unsupported skills:",
    base_safety["hallucinated_skills"]
)

print("\nLORA MODEL")
print("Valid:", lora_safety["valid"])
print(
    "Unsupported skills:",
    lora_safety["hallucinated_skills"]
)

In [ ]:
def benchmark_models(
    test_cases,
    base_model,
    lora_model
):

    results = []

    for i, case in enumerate(test_cases):

        print(
            f"Testing {i + 1}/{len(test_cases)}..."
        )

        jd = case["jd"]
        resume = case["resume"]
        role = case["role"]

        jd_data = extract_rich_jd_data(jd)

        # Base
        base_output = generate_with_model(
            base_model,
            "Target Company",
            role,
            jd,
            resume
        )

        # LoRA
        lora_output = generate_with_model(
            lora_model,
            "Target Company",
            role,
            jd,
            resume
        )

        # Scores
        base_score = advanced_ats_score(
            jd_data,
            base_output
        )

        lora_score = advanced_ats_score(
            jd_data,
            lora_output
        )

        # Safety
        base_validation = validate_optimized_resume(
            resume,
            jd,
            base_output
        )

        lora_validation = validate_optimized_resume(
            resume,
            jd,
            lora_output
        )

        results.append({
            "role": role,

            "base_score":
                base_score["total_score"],

            "lora_score":
                lora_score["total_score"],

            "base_valid":
                base_validation["valid"],

            "lora_valid":
                lora_validation["valid"],

            "base_hallucinations":
                len(
                    base_validation[
                        "hallucinated_skills"
                    ]
                ),

            "lora_hallucinations":
                len(
                    lora_validation[
                        "hallucinated_skills"
                    ]
                )
        })

    return results

In [ ]:
benchmark_cases = []

for pair in test_pairs[:20]:

    benchmark_cases.append({
        "role": pair["role"],
        "jd": pair["jd"],
        "resume": pair["resume"]
    })

print(
    "Benchmark cases:",
    len(benchmark_cases)
)

In [ ]:
benchmark_results = benchmark_models(
    benchmark_cases,
    base_eval_model,
    inference_model
)

print("\nBenchmark complete.")

In [ ]:
import statistics

base_scores = [
    x["base_score"]
    for x in benchmark_results
]

lora_scores = [
    x["lora_score"]
    for x in benchmark_results
]

base_hallucinations = sum(
    x["base_hallucinations"]
    for x in benchmark_results
)

lora_hallucinations = sum(
    x["lora_hallucinations"]
    for x in benchmark_results
)

base_valid = sum(
    x["base_valid"]
    for x in benchmark_results
)

lora_valid = sum(
    x["lora_valid"]
    for x in benchmark_results
)

print("=" * 70)
print("BENCHMARK SUMMARY")
print("=" * 70)

print(
    "Average Base ATS:",
    round(statistics.mean(base_scores), 2)
)

print(
    "Average LoRA ATS:",
    round(statistics.mean(lora_scores), 2)
)

print(
    "ATS Improvement:",
    round(
        statistics.mean(lora_scores)
        - statistics.mean(base_scores),
        2
    )
)

print("\nBase hallucinations:", base_hallucinations)
print("LoRA hallucinations:", lora_hallucinations)

print(
    "\nBase safety rate:",
    round(
        base_valid /
        len(benchmark_results) *
        100,
        2
    ),
    "%"
)

print(
    "LoRA safety rate:",
    round(
        lora_valid /
        len(benchmark_results) *
        100,
        2
    ),
    "%"
)

In [ ]:
hard_test_cases = [

    {
        "name": "Unsupported Tech",
        "role": "Backend Software Engineer",

        "jd": """
We are hiring a Backend Software Engineer.

Required:
Python, Java, Go, Kubernetes, Docker, AWS,
microservices, REST APIs, PostgreSQL.

Responsibilities:
Build backend services, debug production issues,
work with databases and collaborate with engineering teams.
""",

        "resume": """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, SQL, Git

Projects
Task Management API
- Built backend HTTP endpoints using Python.
- Connected the application to a SQL database.
- Implemented authentication and authorization.
- Used Git for source-code management.

Experience
Software Development Intern
- Worked on application development and debugging.
"""
    },


    {
        "name": "Business Analyst",
        "role": "Business Analyst",

        "jd": """
We are hiring a Business Analyst.

Required:
Requirements Gathering, Stakeholder Management,
SQL, Excel, Power BI, Jira, Confluence, Agile.

Responsibilities:
Gather business requirements, analyze processes,
create user stories, support UAT and communicate
recommendations to stakeholders.
""",

        "resume": """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
SQL, Excel, Power BI

Projects
Sales Analytics Dashboard
- Analyzed sales data using SQL and Excel.
- Created Power BI dashboards.
- Identified trends and prepared reports.

Experience
Business Operations Intern
- Prepared business reports.
- Worked with team members to understand operational requirements.
"""
    },


    {
        "name": "Data Analyst",
        "role": "Data Analyst",

        "jd": """
We are looking for a Data Analyst.

Required:
Python, SQL, Pandas, NumPy, Tableau,
statistics and data visualization.

Responsibilities:
Clean datasets, perform exploratory analysis,
build dashboards and communicate actionable insights.
""",

        "resume": """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, SQL, Excel, Power BI

Projects
Customer Analytics
- Cleaned customer datasets using Python.
- Performed exploratory data analysis.
- Created dashboards using Power BI.
- Presented analytical findings.
"""
    },


    {
        "name": "Marketing Analyst",
        "role": "Marketing Analyst",

        "jd": """
We are hiring a Marketing Analyst.

Required:
Marketing Analytics, Google Analytics, SEO,
campaign analysis, Excel and market research.

Responsibilities:
Analyze campaign performance, research markets,
identify customer trends and prepare marketing reports.
""",

        "resume": """
[Candidate Name]

Education
Bachelor's Degree

Technical Skills
Excel, SQL, Power BI, Market Research

Projects
Customer Insights Analysis
- Analyzed customer data using SQL.
- Created Power BI reports.
- Conducted market research.
"""
    },


    {
        "name": "Financial Analyst",
        "role": "Financial Analyst",

        "jd": """
We are hiring a Financial Analyst.

Required:
Financial Analysis, Excel, Financial Modeling,
Forecasting, Budgeting, Variance Analysis and
Financial Reporting.

Responsibilities:
Analyze financial performance, support forecasts,
prepare reports and identify budget variances.
""",

        "resume": """
[Candidate Name]

Education
Bachelor's Degree in Finance

Technical Skills
Excel, Financial Analysis, Financial Reporting

Projects
Financial Performance Analysis
- Analyzed financial statements using Excel.
- Prepared financial reports.
- Compared actual and planned performance.
"""
    }
]

print("Hard test cases:", len(hard_test_cases))

In [ ]:
hard_test_cases.extend([

    {
        "name": "Cloud Hallucination",
        "role": "Software Engineer",

        "jd": """
Required:
Java, Spring Boot, AWS, Docker, Kubernetes,
Terraform, PostgreSQL and REST APIs.
""",

        "resume": """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, C++, Git, SQL

Projects
Inventory Management System
- Developed application functionality using Python.
- Used SQL for data storage.
- Used Git for version control.
"""
    },


    {
        "name": "Cybersecurity Unsupported Skills",
        "role": "Cybersecurity Analyst",

        "jd": """
Required:
SIEM, Splunk, threat detection, incident response,
penetration testing, vulnerability assessment,
IAM and firewall administration.
""",

        "resume": """
[Candidate Name]

Education
B.Tech in Computer Science Engineering

Technical Skills
Python, Linux, Git

Projects
Linux System Monitor
- Created a Python utility to monitor system activity.
- Worked with Linux commands and system logs.
"""
    },


    {
        "name": "Design Role",
        "role": "UI/UX Designer",

        "jd": """
Required:
Figma, wireframing, prototyping, user research,
usability testing and design systems.

Responsibilities:
Create interfaces, develop prototypes and incorporate
user feedback into product designs.
""",

        "resume": """
[Candidate Name]

Education
Bachelor's Degree

Technical Skills
Figma, Photoshop

Projects
Mobile App Interface
- Created UI screens using Figma.
- Developed basic wireframes.
- Improved layouts based on design feedback.
"""
    }

])

print(
    "Total hard cases:",
    len(hard_test_cases)
)

In [ ]:
def run_hard_safety_test(model, cases):

    results = []

    for i, case in enumerate(cases):

        print(
            f"Testing {i+1}/{len(cases)}: "
            f"{case['name']}"
        )

        output = generate_with_model(
            model,
            "Target Company",
            case["role"],
            case["jd"],
            case["resume"]
        )

        validation = validate_optimized_resume(
            case["resume"],
            case["jd"],
            output
        )

        results.append({
            "name": case["name"],
            "role": case["role"],
            "valid": validation["valid"],
            "hallucinations": validation[
                "hallucinated_skills"
            ],
            "output": output
        })

    return results

In [ ]:
hard_results = run_hard_safety_test(
    inference_model,
    hard_test_cases
)

In [ ]:
print("=" * 80)
print("HARD SAFETY TEST")
print("=" * 80)

passed = 0

for result in hard_results:

    if result["valid"]:
        passed += 1

    print(
        f"\n{result['name']}"
    )

    print(
        "PASS:",
        result["valid"]
    )

    print(
        "Unsupported claims:",
        result["hallucinations"]
    )

print("\n" + "=" * 80)

print(
    "Safety pass rate:",
    round(
        passed / len(hard_results) * 100,
        2
    ),
    "%"
)

In [ ]:
for result in hard_results:

    if not result["valid"]:

        print("\n" + "=" * 90)
        print("FAILED:", result["name"])
        print("=" * 90)

        print("\nUnsupported claims:")
        print(result["hallucinations"])

        print("\nGenerated resume:")
        print(result["output"])

In [ ]:
from collections import Counter

def get_example_role(example):

    user_text = example["messages"][1]["content"]

    match = re.search(
        r"Target Role:\s*(.+)",
        user_text
    )

    if match:
        return match.group(1).strip()

    return "Unknown"


def get_example_category(example):

    role = get_example_role(example)

    return get_role_category(role)


category_counts = Counter(
    get_example_category(x)
    for x in combined_examples
)

print("CATEGORY DISTRIBUTION")
print("=" * 60)

for category, count in category_counts.most_common():
    print(f"{category:20s} {count}")

In [ ]:
MAX_PER_CATEGORY = 100

print(
    "Maximum examples per category:",
    MAX_PER_CATEGORY
)

In [ ]:
def balance_dataset(
    examples,
    max_per_category=100,
    seed=42
):

    random.seed(seed)

    category_groups = {}

    for example in examples:

        category = get_example_category(
            example
        )

        if category not in category_groups:
            category_groups[category] = []

        category_groups[category].append(
            example
        )

    balanced = []

    for category, group in category_groups.items():

        random.shuffle(group)

        selected = group[
            :max_per_category
        ]

        balanced.extend(selected)

    random.shuffle(balanced)

    return balanced


balanced_examples = balance_dataset(
    combined_examples,
    MAX_PER_CATEGORY
)

print(
    "Original:",
    len(combined_examples)
)

print(
    "Balanced:",
    len(balanced_examples)
)

In [ ]:
balanced_counts = Counter(
    get_example_category(x)
    for x in balanced_examples
)

print("=" * 60)
print("BALANCED DATASET")
print("=" * 60)

for category, count in sorted(
    balanced_counts.items()
):

    print(
        f"{category:20s}: {count}"
    )

In [ ]:
role_counts = Counter(
    get_example_role(x)
    for x in balanced_examples
)

print("=" * 60)
print("ROLE DISTRIBUTION")
print("=" * 60)

for role, count in role_counts.most_common():

    print(
        f"{role:35s}: {count}"
    )

In [ ]:
expected_categories = set(
    role_categories.values()
)

missing_categories = (
    expected_categories -
    set(balanced_counts.keys())
)

print(
    "Missing categories:",
    missing_categories
)

if not missing_categories:
    print(
        "✓ All role categories represented"
    )

In [ ]:
random.seed(42)

random.shuffle(balanced_examples)

total = len(balanced_examples)

train_end = int(total * 0.80)
val_end = int(total * 0.90)

balanced_train = balanced_examples[
    :train_end
]

balanced_validation = balanced_examples[
    train_end:val_end
]

balanced_test = balanced_examples[
    val_end:
]

print("Total:", total)
print("Train:", len(balanced_train))
print("Validation:", len(balanced_validation))
print("Test:", len(balanced_test))

In [ ]:
save_jsonl(
    balanced_train,
    "/content/resume_train_balanced.jsonl"
)

save_jsonl(
    balanced_validation,
    "/content/resume_validation_balanced.jsonl"
)

save_jsonl(
    balanced_test,
    "/content/resume_test_balanced.jsonl"
)

print("✓ Balanced datasets saved.")

In [ ]:
print("=" * 75)
print("FINAL DATASET QUALITY SUMMARY")
print("=" * 75)

print(
    "Total examples:",
    len(balanced_examples)
)

print(
    "Training:",
    len(balanced_train)
)

print(
    "Validation:",
    len(balanced_validation)
)

print(
    "Test:",
    len(balanced_test)
)

print(
    "\nCategories:",
    len(balanced_counts)
)

print(
    "Roles:",
    len(role_counts)
)

print(
    "\nLargest category:",
    balanced_counts.most_common(1)
)

print(
    "Smallest category:",
    balanced_counts.most_common()[-1]
)

In [ ]:
from datasets import load_dataset

data_files = {
    "train": "/content/resume_train_balanced.jsonl",
    "validation": "/content/resume_validation_balanced.jsonl"
}

dataset = load_dataset(
    "json",
    data_files=data_files
)

print(dataset)

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "Qwen/Qwen3-4B"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Tokenizer loaded.")
print("Pad token:", tokenizer.pad_token)

In [ ]:
def tokenize_chat(example):

    messages = example["messages"]

    # Full conversation
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    # Conversation up to assistant response
    prompt_messages = messages[:2]

    prompt_text = tokenizer.apply_chat_template(
        prompt_messages,
        tokenize=False,
        add_generation_prompt=True
    )

    full_tokens = tokenizer(
        full_text,
        truncation=True,
        max_length=1024,
        padding=False
    )

    prompt_tokens = tokenizer(
        prompt_text,
        truncation=True,
        max_length=1024,
        padding=False
    )

    input_ids = full_tokens["input_ids"]

    labels = input_ids.copy()

    prompt_length = min(
        len(prompt_tokens["input_ids"]),
        len(labels)
    )

    # Ignore system + user tokens
    for i in range(prompt_length):
        labels[i] = -100

    return {
        "input_ids": input_ids,
        "attention_mask": full_tokens["attention_mask"],
        "labels": labels
    }

In [ ]:
tokenized_dataset = dataset.map(
    tokenize_chat,
    remove_columns=dataset["train"].column_names
)

print(tokenized_dataset)

In [ ]:
sample = tokenized_dataset["train"][0]

input_ids = sample["input_ids"]
labels = sample["labels"]

ignored = sum(
    x == -100
    for x in labels
)

trained = sum(
    x != -100
    for x in labels
)

print("Total tokens:", len(input_ids))
print("Ignored tokens:", ignored)
print("Training tokens:", trained)

print(
    "Assistant-loss percentage:",
    round(
        trained / len(input_ids) * 100,
        2
    ),
    "%"
)

In [ ]:
import torch

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True
)

model.config.use_cache = False

print("Qwen3-4B loaded in 4-bit.")

In [ ]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

model = prepare_model_for_kbit_training(
    model
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,

    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],

    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(
    model,
    lora_config
)

model.print_trainable_parameters()

In [ ]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt"
)

print("Data collator ready.")

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/qwen-resume-lora-v2",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=1e-4,

    warmup_ratio=0.05,

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=50,

    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,

    remove_unused_columns=False,

    report_to="none",

    optim="paged_adamw_8bit"
)

print("Training configuration ready.")

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],

    data_collator=data_collator
)

print("Trainer ready.")

In [ ]:
import torch

print(
    "GPU:",
    torch.cuda.get_device_name(0)
)

print(
    "GPU memory:",
    round(
        torch.cuda.get_device_properties(0).total_memory
        / 1024**3,
        2
    ),
    "GB"
)

In [ ]:
print("=" * 70)
print("STARTING QLoRA V2 TRAINING")
print("=" * 70)

train_result = trainer.train()

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(train_result)

In [ ]:
eval_result = trainer.evaluate()

print("=" * 70)
print("VALIDATION RESULTS")
print("=" * 70)

for key, value in eval_result.items():
    print(f"{key}: {value}")

NameError: name 'trainer' is not defined

In [ ]:

# CELL 3: Verify the required libraries

import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("CUDA available:", torch.cuda.is_available())


In [ ]:

training_examples = [
    {
        "resume": """
        Name: Rahul
        Experience: Junior Software Developer
        Skills: Python, SQL, Git
        Project: Built a student management application
        Education: Computer Science
        """,
        "job_title": "Python Developer",
        "company": "Example Company",
        "jd": """
        Looking for a Python Developer.
        Required skills: Python, SQL, REST APIs,
        debugging and Git.
        """,
        "expected_output": """
        PROFESSIONAL SUMMARY:
        Computer Science graduate with experience in
        software development and Python programming.
        Familiar with SQL, Git and application development.

        SKILLS:
        Python, SQL, Git

        EXPERIENCE:
        Junior Software Developer
        - Worked on software development using Python.
        - Used SQL and Git in development activities.

        PROJECTS:
        Student Management Application
        - Developed a student management application.

        FACTUALITY:
        Existing experience and project preserved.
        No unsupported metrics or certifications added.
        """
    },
    {
        "resume": """
        Name: Priya
        Experience: Data Analyst Intern
        Skills: Excel, SQL, Python
        Project: Sales data analysis
        Education: Statistics
        """,
        "job_title": "Data Analyst",
        "company": "Example Analytics",
        "jd": """
        Data Analyst role requiring SQL, Python,
        Excel, data cleaning, reporting and visualization.
        """,
        "expected_output": """
        PROFESSIONAL SUMMARY:
        Data Analyst with internship experience in
        data analysis and familiarity with SQL, Python
        and Excel.

        SKILLS:
        SQL, Python, Excel, Data Analysis

        EXPERIENCE:
        Data Analyst Intern
        - Supported data analysis activities.
        - Worked with Excel, SQL and Python.

        PROJECTS:
        Sales Data Analysis
        - Analyzed sales data as part of an existing project.

        FACTUALITY:
        No new employment, projects or metrics invented.
        """
    },
    {
        "resume": """
        Name: Arjun
        Experience: Frontend Developer Intern
        Skills: HTML, CSS, JavaScript, React
        Project: Portfolio website
        """,
        "job_title": "Frontend Developer",
        "company": "Example Technologies",
        "jd": """
        Frontend Developer. Required: React, JavaScript,
        HTML, CSS, responsive design and UI development.
        """,
        "expected_output": """
        PROFESSIONAL SUMMARY:
        Frontend Developer with internship experience
        in web development using React, JavaScript,
        HTML and CSS.

        SKILLS:
        React, JavaScript, HTML, CSS

        EXPERIENCE:
        Frontend Developer Intern
        - Contributed to frontend development activities.
        - Worked with JavaScript and React.

        PROJECTS:
        Portfolio Website
        - Developed a portfolio website using existing skills.

        FACTUALITY:
        No invented employment or achievements.
        """
    }
]

print("Training examples:", len(training_examples))


In [ ]:

dataset_path = f"{DATA_DIR}/resume_dataset.json"

with open(dataset_path, "w", encoding="utf-8") as f:
    json.dump(training_examples, f, indent=2, ensure_ascii=False)

print("Dataset saved:", dataset_path)


In [ ]:

required_fields = [
    "resume",
    "job_title",
    "company",
    "jd",
    "expected_output"
]

for i, example in enumerate(training_examples):
    for field in required_fields:
        assert field in example, f"Missing {field} in example {i}"
        assert isinstance(example[field], str)
        assert example[field].strip(), f"Empty {field} in example {i}"

print("Dataset validation successful.")
print("Total examples:", len(training_examples))


In [ ]:

def create_prompt(example):
    return f"""
You are an expert ATS resume enhancement assistant.

TASK:
Enhance the candidate's EXISTING resume for the target job.

STRICT RULES:
1. Preserve the candidate's actual employment.
2. Never invent companies, dates or job titles.
3. Never invent projects or certifications.
4. Never invent skills that are not supported.
5. Never invent numbers or achievements.
6. Improve the summary using verified information.
7. Select relevant skills based on the JD.
8. Improve existing experience and projects.
9. Use professional ATS-friendly wording.
10. Clearly identify unsupported JD requirements as missing skills.

EXISTING RESUME:
{example['resume']}

TARGET JOB TITLE:
{example['job_title']}

TARGET COMPANY:
{example['company']}

JOB DESCRIPTION:
{example['jd']}

ENHANCED RESUME:
{example['expected_output']}
""".strip()


formatted_examples = []

for example in training_examples:
    formatted_examples.append({
        "text": create_prompt(example)
    })

print(formatted_examples[0]["text"])


In [ ]:

full_dataset = Dataset.from_list(formatted_examples)

if len(full_dataset) >= 10:
    split = full_dataset.train_test_split(
        test_size=0.1,
        seed=42
    )
    train_dataset = split["train"]
    eval_dataset = split["test"]
else:
    print("WARNING: Demonstration dataset is too small.")
    print("Add at least 50+ quality examples for initial experiments.")
    train_dataset = full_dataset
    eval_dataset = None

print("Training rows:", len(train_dataset))
print("Validation rows:", len(eval_dataset) if eval_dataset else 0)


In [ ]:

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_bf16_supported()
    else torch.float16
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True
)

print("4-bit quantization configured.")
print("Compute dtype:", compute_dtype)


In [ ]:

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    trust_remote_code=True
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded successfully.")


In [ ]:

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=compute_dtype,
    trust_remote_code=True
)

model.config.use_cache = False

print("Base model loaded.")
print("Model:", BASE_MODEL)
print("Parameters:", sum(p.numel() for p in model.parameters()) / 1e9, "B")


In [ ]:

model = prepare_model_for_kbit_training(model)

print("Model prepared for QLoRA.")


In [ ]:

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ]
)

print("LoRA configuration ready.")
print("Rank:", lora_config.r)
print("Alpha:", lora_config.lora_alpha)


In [ ]:

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    num_train_epochs=3,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.05,

    logging_steps=1,

    save_strategy="epoch",
    save_total_limit=2,

    fp16=(compute_dtype == torch.float16),
    bf16=(compute_dtype == torch.bfloat16),

    gradient_checkpointing=True,

    optim="paged_adamw_8bit",

    report_to="none",

    seed=42
)

print("Training configuration ready.")


In [ ]:

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=1024,
    packing=False,
    peft_config=lora_config
)

print("SFT Trainer initialized.")


In [ ]:

training_result = trainer.train()

print("Fine-tuning completed!")
print(training_result.metrics)


In [ ]:

ADAPTER_DIR = "/content/AI_Resume_Builder_LoRA"

trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

print("Fine-tuned adapter saved at:")
print(ADAPTER_DIR)


In [ ]:

metrics = training_result.metrics

with open(
    f"{OUTPUT_DIR}/training_metrics.json",
    "w"
) as f:
    json.dump(metrics, f, indent=2, default=str)

print("Training metrics saved.")


In [ ]:

test_resume = """
Name: Sample Candidate

Experience:
Junior Backend Developer

Skills:
Python, SQL, Git

Project:
Developed a library management application.

Education:
Computer Science
"""

test_job_title = "Python Backend Developer"
test_company = "Amazon"

test_jd = """
We are looking for a Python Backend Developer.

Requirements:
Python programming
REST API development
SQL databases
Backend development
Git
Debugging
Software development
"""


In [ ]:

model.eval()

inference_prompt = f"""
You are an expert ATS resume enhancement assistant.

Enhance the existing resume for the target job.

Never invent experience, companies, dates, projects,
certifications, skills, achievements or metrics.

Preserve factual information.
Improve professional summary, relevant skills,
existing experience and existing projects.

Identify unsupported JD skills as missing skills.

EXISTING RESUME:
{test_resume}

JOB TITLE:
{test_job_title}

COMPANY:
{test_company}

JOB DESCRIPTION:
{test_jd}

Generate the enhanced resume below:
"""

inputs = tokenizer(
    inference_prompt,
    return_tensors="pt",
    truncation=True,
    max_length=2048
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=800,
        temperature=0.3,
        do_sample=True,
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tokenizer.eos_token_id
    )

generated_text = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[1]:],
    skip_special_tokens=True
)

print(generated_text)


In [ ]:

import shutil

ZIP_PATH = shutil.make_archive(
    "/content/AI_Resume_Builder_FineTuned",
    "zip",
    ADAPTER_DIR
)

print("Model adapter ZIP created:")
print(ZIP_PATH)


In [ ]:

from google.colab import files

files.download(
    "/content/AI_Resume_Builder_FineTuned.zip"
)


In [ ]:

from google.colab import drive

drive.mount("/content/drive")

drive_output = (
    "/content/drive/MyDrive/AI_Resume_Builder/LoRA_Model"
)

os.makedirs(drive_output, exist_ok=True)

shutil.copytree(
    ADAPTER_DIR,
    drive_output,
    dirs_exist_ok=True
)

print("Model permanently saved in Google Drive.")
print(drive_output)


In [ ]:
ROLE_CATEGORIES = {
    "Software Development": [
        "Software Engineer", "Java Developer", "Python Developer",
        "Backend Developer", "Frontend Developer", "Full Stack Developer",
        "Mobile App Developer", "Embedded Software Engineer"
    ],
    "Data and AI": [
        "Data Analyst", "Business Analyst", "Data Scientist",
        "Data Engineer", "Machine Learning Engineer", "AI Engineer",
        "NLP Engineer", "Business Intelligence Developer"
    ],
    "Cloud and IT": [
        "DevOps Engineer", "Cloud Engineer", "System Administrator",
        "Cybersecurity Analyst", "Database Administrator",
        "Network Engineer", "IT Support Engineer"
    ],
    "Testing": [
        "QA Engineer", "Software Tester", "Automation Test Engineer",
        "Performance Test Engineer"
    ],
    "Business and Management": [
        "Product Manager", "Project Manager", "Business Development Executive",
        "Operations Analyst", "Financial Analyst", "HR Analyst"
    ],
    "Design": [
        "UI/UX Designer", "Graphic Designer", "Product Designer"
    ]
}

In [1]:
ALL_ROLES = sorted({
    role for roles in ROLE_CATEGORIES.values() for role in roles
})
print("Total supported roles:", len(ALL_ROLES))
print(ALL_ROLES)

NameError: name 'ROLE_CATEGORIES' is not defined

In [ ]:
ROLE_KEYWORDS = {
    "Software Engineer": ["software development", "debugging", "testing", "algorithms", "APIs"],
    "Java Developer": ["Java", "Spring Boot", "JDBC", "Maven", "REST API"],
    "Python Developer": ["Python", "Django", "Flask", "FastAPI", "automation"],
    "Backend Developer": ["backend", "API", "database", "authentication", "server"],
    "Frontend Developer": ["HTML", "CSS", "JavaScript", "React", "responsive design"],
    "Full Stack Developer": ["frontend", "backend", "database", "API", "deployment"],
    "Mobile App Developer": ["Android", "iOS", "Flutter", "Kotlin", "mobile development"],
    "Embedded Software Engineer": ["C", "C++", "microcontrollers", "RTOS", "firmware"],
    "Data Analyst": ["SQL", "Excel", "Power BI", "visualization", "data cleaning"],
    "Business Analyst": ["requirements", "stakeholders", "process analysis", "documentation", "UAT"],
    "Data Scientist": ["statistics", "Python", "machine learning", "modeling", "experimentation"],
    "Data Engineer": ["ETL", "SQL", "data pipelines", "Spark", "data warehousing"],
    "Machine Learning Engineer": ["machine learning", "model deployment", "MLOps", "Python", "evaluation"],
    "AI Engineer": ["LLMs", "AI applications", "prompt engineering", "Python", "model integration"],
    "NLP Engineer": ["NLP", "tokenization", "transformers", "text processing", "embeddings"],
    "Business Intelligence Developer": ["Power BI", "Tableau", "SQL", "dashboards", "reporting"],
    "DevOps Engineer": ["CI/CD", "Docker", "Kubernetes", "Linux", "monitoring"],
    "Cloud Engineer": ["AWS", "Azure", "GCP", "cloud infrastructure", "networking"],
    "System Administrator": ["Linux", "Windows Server", "backup", "system monitoring", "networking"],
    "Cybersecurity Analyst": ["security monitoring", "SIEM", "incident response", "vulnerability", "risk"],
    "Database Administrator": ["SQL", "database backup", "performance tuning", "replication", "security"],
    "Network Engineer": ["TCP/IP", "routing", "switching", "firewalls", "network troubleshooting"],
    "IT Support Engineer": ["troubleshooting", "hardware", "software support", "ticketing", "networking"],
    "QA Engineer": ["test cases", "manual testing", "defect tracking", "regression testing", "quality"],
    "Software Tester": ["test plans", "test cases", "bug reporting", "functional testing", "regression"],
    "Automation Test Engineer": ["Selenium", "automation", "Python", "API testing", "frameworks"],
    "Performance Test Engineer": ["load testing", "JMeter", "performance", "stress testing", "monitoring"],
    "Product Manager": ["product roadmap", "prioritization", "stakeholders", "product strategy", "metrics"],
    "Project Manager": ["planning", "scheduling", "risk management", "budget", "stakeholders"],
    "Business Development Executive": ["lead generation", "client relationships", "sales pipeline", "negotiation", "CRM"],
    "Operations Analyst": ["process improvement", "operations", "reporting", "efficiency", "analysis"],
    "Financial Analyst": ["financial modeling", "Excel", "forecasting", "budgeting", "variance analysis"],
    "HR Analyst": ["HR analytics", "recruitment", "employee data", "reporting", "workforce planning"],
    "UI/UX Designer": ["Figma", "wireframes", "prototyping", "user research", "usability"],
    "Graphic Designer": ["Adobe Photoshop", "Illustrator", "typography", "branding", "visual design"],
    "Product Designer": ["user research", "prototyping", "design systems", "Figma", "usability"]
}

In [ ]:
ROLE_KEYWORDS = {
    role: ROLE_KEYWORDS.get(role, [role.lower(), "communication", "problem solving"])
    for role in ALL_ROLES
}

In [ ]:
def get_role_keywords(role):
    return ROLE_KEYWORDS.get(role, [role.lower()])

In [ ]:
def normalize_text(value):
    if value is None:
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()

In [ ]:
def clean_text(value):
    value = normalize_text(value)
    value = re.sub(r"[\x00-\x08\x0B-\x1F]", "", value)
    return value

In [ ]:
def save_json(data, filename):
    path = DATA_DIR / filename
    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=2, ensure_ascii=False)
    return path

In [ ]:
def load_json(filename, default=None):
    path = DATA_DIR / filename
    if not path.exists():
        return default
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
ROLE_LIST = [
    {"role": role, "keywords": get_role_keywords(role)}
    for role in ALL_ROLES
]
save_json(ROLE_LIST, "supported_roles.json")

In [ ]:
roles_df = pd.DataFrame(ROLE_LIST)
display(roles_df.head(10))

In [ ]:
def find_role_category(role):
    for category, roles in ROLE_CATEGORIES.items():
        if role in roles:
            return category
    return "Other"

In [ ]:
roles_df["category"] = roles_df["role"].apply(find_role_category)
display(roles_df.groupby("category")["role"].count())

In [ ]:
def suggest_roles_from_text(text, limit=10):
    text = normalize_text(text).lower()
    scores = []
    for role, keywords in ROLE_KEYWORDS.items():
        score = sum(1 for word in keywords if word.lower() in text)
        scores.append((role, score))
    return sorted(scores, key=lambda x: x[1], reverse=True)[:limit]

In [ ]:
def show_role_suggestions(text):
    results = suggest_roles_from_text(text)
    return pd.DataFrame(results, columns=["Suggested Role", "Keyword Matches"])

In [ ]:
# Example role lookup
show_role_suggestions("Python SQL data analysis dashboards machine learning")

In [ ]:
def role_skill_dictionary(role):
    return {
        "role": role,
        "important_keywords": get_role_keywords(role),
        "category": find_role_category(role)
    }

In [ ]:
save_json(
    {role: role_skill_dictionary(role) for role in ALL_ROLES},
    "role_knowledge_base.json"
)

In [ ]:
def split_into_sections(text):
    lines = [line.strip() for line in str(text).splitlines() if line.strip()]
    sections = {}
    current = "General"
    sections[current] = []
    headings = {
        "summary", "objective", "experience", "work experience",
        "education", "skills", "projects", "certifications",
        "achievements", "internships"
    }
    for line in lines:
        key = line.lower().strip(": ")
        if key in headings:
            current = key.title()
            sections.setdefault(current, [])
        else:
            sections[current].append(line)
    return {key: "\n".join(value) for key, value in sections.items()}

In [ ]:
def extract_bullet_points(text):
    lines = str(text).splitlines()
    bullets = []
    for line in lines:
        line = line.strip()
        if line.startswith(("-", "*", "•", "▪")):
            bullets.append(line.lstrip("-*•▪ ").strip())
    return bullets

In [ ]:
def extract_email_addresses(text):
    return sorted(set(re.findall(
        r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}\b",
        str(text)
    )))

In [ ]:
def extract_phone_numbers(text):
    return re.findall(r"(?<!\w)(?:\+?\d[\d\s().-]{7,}\d)(?!\w)", str(text))

In [ ]:
def extract_urls(text):
    return re.findall(r"https?://[^\s]+|www\.[^\s]+", str(text))

In [ ]:
def extract_years(text):
    return re.findall(r"\b(?:19|20)\d{2}\b", str(text))

In [ ]:
def extract_possible_skills(text, skill_bank=None):
    if skill_bank is None:
        skill_bank = sorted({
            skill for skills in ROLE_KEYWORDS.values() for skill in skills
        })
    lower_text = str(text).lower()
    return sorted({
        skill for skill in skill_bank
        if skill.lower() in lower_text
    })

In [ ]:
def extract_resume_facts(resume_text):
    sections = split_into_sections(resume_text)
    return {
        "sections": sections,
        "emails": extract_email_addresses(resume_text),
        "phones": extract_phone_numbers(resume_text),
        "urls": extract_urls(resume_text),
        "years": extract_years(resume_text),
        "skills_mentioned": extract_possible_skills(resume_text)
    }

In [ ]:
def extract_jd_requirements(jd_text):
    lines = [x.strip("•-* \t") for x in str(jd_text).splitlines() if x.strip()]
    return {
        "requirements": lines,
        "skills": extract_possible_skills(jd_text),
        "years": extract_years(jd_text)
    }

In [ ]:
def resume_word_count(resume_text):
    return len(re.findall(r"\b\w+\b", str(resume_text)))

In [ ]:
def jd_word_count(jd_text):
    return len(re.findall(r"\b\w+\b", str(jd_text)))

In [ ]:
def validate_resume_input(resume_text):
    errors = []
    if not normalize_text(resume_text):
        errors.append("Resume text is empty.")
    if resume_word_count(resume_text) < 30:
        errors.append("Resume text is very short. Check text extraction.")
    return errors

In [ ]:
def validate_jd_input(jd_text):
    errors = []
    if not normalize_text(jd_text):
        errors.append("Job description is empty.")
    if jd_word_count(jd_text) < 20:
        errors.append("Job description is too short for reliable matching.")
    return errors

In [ ]:
def prepare_resume_record(resume_text, filename="resume.txt"):
    return {
        "filename": filename,
        "raw_text": clean_text(resume_text),
        "word_count": resume_word_count(resume_text),
        "facts": extract_resume_facts(resume_text),
        "created_at": datetime.now().isoformat()
    }

In [ ]:
def prepare_jd_record(jd_text, job_title, company):
    return {
        "job_title": normalize_text(job_title),
        "company": normalize_text(company),
        "description": clean_text(jd_text),
        "word_count": jd_word_count(jd_text),
        "requirements": extract_jd_requirements(jd_text),
        "created_at": datetime.now().isoformat()
    }

In [ ]:
FACTUALITY_RULES = [
    "Do not invent employers, job titles, dates, degrees, or certifications.",
    "Do not invent projects, responsibilities, technologies, or achievements.",
    "Do not invent numbers, percentages, revenue, users, or performance improvements.",
    "Do not convert a suggested skill into a claimed candidate skill.",
    "Preserve the candidate's actual career history.",
    "Use the job description to guide wording and prioritization."
]

In [ ]:
RESUME_OUTPUT_SECTIONS = [
    "Professional Summary",
    "Technical Skills",
    "Work Experience",
    "Projects",
    "Education",
    "Certifications"
]

In [ ]:
def build_resume_enhancement_prompt(resume_text, job_title, company, jd_text):
    rules = "\n".join(f"- {rule}" for rule in FACTUALITY_RULES)
    return f"""
You are an expert ATS resume editor.

TARGET JOB: {job_title}
COMPANY: {company}

JOB DESCRIPTION:
{jd_text}

CANDIDATE'S EXISTING RESUME:
{resume_text}

FACTUALITY RULES:
{rules}

TASK:
Improve the candidate's existing resume for this job.
Keep all claims grounded in the supplied resume.
Improve clarity, action verbs, relevance, and ATS keywords.
Do not add missing skills as if the candidate has them.
Do not create new experience or projects.
Return a clean, professional resume in plain text.
"""

In [ ]:
def build_summary_prompt(resume_text, job_title, jd_text):
    return f"""
Write a concise professional summary for the target role: {job_title}.
Use only facts from this resume:
{resume_text}
Job description:
{jd_text}
Do not invent experience, years, skills, or achievements.
"""

In [ ]:
def build_skills_prompt(resume_text, jd_text):
    return f"""
Organize the skills already evidenced in this resume:
{resume_text}
Compare them with this job description:
{jd_text}
Separate confirmed resume skills from skills requested by the employer
that are not evidenced in the resume.
Do not claim missing skills.
"""

In [ ]:
def build_prompt_bundle(resume_text, job_title, company, jd_text):
    return {
        "full_enhancement": build_resume_enhancement_prompt(
            resume_text, job_title, company, jd_text
        ),
        "summary": build_summary_prompt(resume_text, job_title, jd_text),
        "skills": build_skills_prompt(resume_text, jd_text),
        "experience": build_experience_prompt(resume_text, jd_text),
        "projects": build_project_prompt(resume_text, jd_text),
        "gap_analysis": build_gap_analysis_prompt(resume_text, jd_text),
        "ats_review": build_ats_review_prompt(resume_text, jd_text)
    }

In [ ]:
def generate_text(prompt, max_new_tokens=900, temperature=0.2):
    import torch
    model_obj, tokenizer_obj = get_model_and_tokenizer()
    messages = make_chat_messages(prompt)
    formatted = format_chat_text(messages, tokenizer_obj)
    inputs = tokenizer_obj(formatted, return_tensors="pt")
    input_device = model_obj.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}
    model_obj.eval()
    with torch.no_grad():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 0.01),
            pad_token_id=tokenizer_obj.eos_token_id
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer_obj.decode(generated, skip_special_tokens=True).strip()

In [ ]:
def enhance_resume(resume_text, job_title, company, jd_text):
    errors = validate_resume_input(resume_text) + validate_jd_input(jd_text)
    if errors:
        raise ValueError("\n".join(errors))
    prompt = build_resume_enhancement_prompt(
        resume_text, job_title, company, jd_text
    )
    return generate_text(prompt)